# Fast3R 复现：从多视图图片到 3D 点云

这是一个面向学习和求职展示的、可重复运行的 Fast3R 实验记录。Fast3R 把多张同一场景的图片一次性送入 Transformer，预测每个视角的 global/local 3D 点图与置信度；相机位姿随后通过 PnP 从点图估计。

本 Notebook 分成两层：

1. **功能复现（主线）**：使用仓库提供的 Hugging Face checkpoint，完成视频抽帧、图片预处理、一次多视图推理、PnP 相机估计、置信度可视化和 PLY 点云导出。
2. **数据集评测**：公开 HF 权重可通过官方指标实现评测，DTU 22 场景已运行；本 Notebook 末尾进一步记录论文对应、median 指标差距、视角数与 local/global 点图消融。完整论文还包含其他数据集和独立训练实验。

> 建议环境：Python 3.11、CUDA GPU（建议显存至少 12 GB）和仓库根目录。CPU 可以完成环境检查和结果可视化，但不适合运行 ViT-L/512 的完整推理。

先运行下面的代码单元，确认 Notebook 使用的是 `fast3r` Conda 环境，并检查 Python、PyTorch、CUDA、GPU、Fast3R 和关键依赖。

## 0. 运行前准备

在终端执行一次下面的官方安装流程，然后在 Jupyter 中选择 `fast3r` kernel。已有环境可以直接跳过：

```bash
conda create -n fast3r python=3.11 cmake=3.14.0 -y
conda activate fast3r
# 按本机 CUDA 版本安装 PyTorch；再安装项目依赖
pip install -r requirements.txt
pip install -e .
python -m ipykernel install --user --name fast3r --display-name "fast3r"
```

仓库 README 特别提醒：不要安装 DUSt3R 的 cuROPE 模块，否则可能破坏 Fast3R 的预测。模型权重也可以提前下载到 `checkpoints/Fast3R_ViT_Large_512/`，这样 Notebook 不依赖运行时网络。

In [6]:
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path


def package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "未安装"


print("=== Python 环境 ===")
print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("Conda 环境:", os.environ.get("CONDA_DEFAULT_ENV", "未检测到"))
print("系统:", platform.platform())

print("\n=== PyTorch / CUDA ===")
try:
    import torch

    print("PyTorch:", torch.__version__)
    print("Torch CUDA:", torch.version.cuda)
    print("CUDA 可用:", torch.cuda.is_available())
    print("GPU 数量:", torch.cuda.device_count())
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            properties = torch.cuda.get_device_properties(index)
            memory_gb = properties.total_memory / 1024**3
            print(f"GPU {index}: {properties.name} ({memory_gb:.1f} GB)")
except ImportError as error:
    print("PyTorch 导入失败:", error)

print("\n=== Fast3R / 关键依赖 ===")
for package_name in [
    "fast3r",
    "numpy",
    "open3d",
    "jupyterlab",
    "ipykernel",
    "gradio",
    "lightning",
    "huggingface-hub",
]:
    print(f"{package_name}: {package_version(package_name)}")

print("\n=== 项目路径 ===")
project_root = Path.cwd()
print("当前目录:", project_root)
for relative_path in [
    "fast3r",
    "checkpoints/Fast3R_ViT_Large_512",
    "demo_examples",
    "requirements.txt",
]:
    path = project_root / relative_path
    print(f"{relative_path}: {'存在' if path.exists() else '缺失'}")

print("\n=== Fast3R 导入检查 ===")
try:
    import fast3r
    from fast3r.dust3r.inference_multiview import inference
    from fast3r.models.fast3r import Fast3R

    print("Fast3R 导入: 成功")
    print("Fast3R 路径:", fast3r.__file__)
except Exception as error:
    print("Fast3R 导入失败:", repr(error))

=== Python 环境 ===
Python: 3.11.16
解释器: /home/yyz/miniconda3/envs/fast3r/bin/python
Conda 环境: fast3r
系统: Linux-6.8.0-138-generic-x86_64-with-glibc2.35

=== PyTorch / CUDA ===
PyTorch: 2.7.1+cu128
Torch CUDA: 12.8
CUDA 可用: True
GPU 数量: 1
GPU 0: NVIDIA GeForce RTX 5070 Ti Laptop GPU (11.5 GB)

=== Fast3R / 关键依赖 ===
fast3r: 1.0
numpy: 1.26.4
open3d: 0.19.0
jupyterlab: 4.6.3
ipykernel: 7.3.0
gradio: 5.23.0
lightning: 2.6.5
huggingface-hub: 1.30.0

=== 项目路径 ===
当前目录: /home/yyz/fast3r
fast3r: 存在
checkpoints/Fast3R_ViT_Large_512: 存在
demo_examples: 存在
requirements.txt: 存在

=== Fast3R 导入检查 ===
Fast3R 导入: 成功
Fast3R 路径: /home/yyz/fast3r/fast3r/__init__.py


## 在 Notebook 中启动 Gradio Demo

运行下面的代码单元后，模型和 Gradio 服务会在后台启动。请在浏览器中打开输出的本地地址完成上传和可视化。

> 说明：官方 Demo 还会启动 Viser 3D 服务并申请临时 share URL；如果网络受限，Gradio 页面可能能打开但 3D iframe 不可用。主线推理和本 Notebook 的 Plotly/PLY 预览不依赖这个临时链接。

In [7]:
import os
import subprocess
import sys
import time
from pathlib import Path

project_root = Path.cwd()
checkpoint_dir = project_root / "checkpoints" / "Fast3R_ViT_Large_512"
demo_script = project_root / "fast3r" / "viz" / "demo.py"

demo_process = globals().get("demo_process")
if demo_process is not None and demo_process.poll() is None:
    print(f"Demo 已在运行，PID: {demo_process.pid}")
    print("浏览器地址: http://127.0.0.1:7860")
else:
    import torch

    can_start = True
    if torch.cuda.is_available():
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_gb = free_bytes / 1024**3
        print(f"GPU 可用显存: {free_gb:.2f} / {total_bytes / 1024**3:.2f} GB")
        if free_gb < 4.0:
            can_start = False
            print("暂不启动 Demo：可用显存不足 4 GB。")
            print("请先停止其他 GPU 训练任务，再重新运行本单元。")

    if can_start:
        if not checkpoint_dir.exists():
            raise FileNotFoundError(f"找不到模型目录: {checkpoint_dir}")
        if not demo_script.exists():
            raise FileNotFoundError(f"找不到 Demo 脚本: {demo_script}")

        environment = os.environ.copy()
        environment["GRADIO_TEMP_DIR"] = str(project_root / "gradio_tmp_dir")
        demo_process = subprocess.Popen(
            [
                sys.executable,
                str(demo_script),
                "--checkpoint_dir",
                str(checkpoint_dir),
                "--examples_dir",
                str(project_root / "demo_examples"),
                "--output_dir",
                str(project_root / "demo_outputs"),
            ],
            cwd=project_root,
            env=environment,
            start_new_session=True,
        )
        time.sleep(2)
        if demo_process.poll() is not None:
            raise RuntimeError(f"Demo 启动失败，退出码: {demo_process.returncode}")

        print(f"Demo 已在后台启动，PID: {demo_process.pid}")
        print("请在浏览器打开: http://127.0.0.1:7860")
        print("如果端口被占用，请先运行停止单元，再重新启动。")

GPU 可用显存: 10.42 / 11.47 GB
Demo 已在后台启动，PID: 85761
请在浏览器打开: http://127.0.0.1:7860
如果端口被占用，请先运行停止单元，再重新启动。


## 停止 Notebook 启动的 Demo

在浏览器中完成观察后，运行下面的单元释放模型占用的显存。

In [8]:
demo_process = globals().get("demo_process")
if demo_process is not None and demo_process.poll() is None:
    demo_process.terminate()
    try:
        demo_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        demo_process.kill()
        demo_process.wait()
    print(f"Demo 已停止，PID: {demo_process.pid}")
    demo_process = None
else:
    print("当前没有由本 Notebook 启动的 Demo 进程。")

Demo 已停止，PID: 85761


## 直接运行多视图推理

下面的单元使用本地 Fast3R checkpoint 和仓库中的示例图片完成一次推理。先运行配置和图片加载单元，再运行模型推理单元。

In [9]:
from pathlib import Path

import torch

from fast3r.dust3r.inference_multiview import inference
from fast3r.dust3r.utils.image import load_images
from fast3r.utils.checkpoint_utils import load_model
from fast3r.viz.video_utils import extract_frames_from_video

project_root = Path.cwd()
checkpoint_dir = project_root / "checkpoints" / "Fast3R_ViT_Large_512"
output_dir = project_root / "demo_outputs" / "notebook_inference"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not checkpoint_dir.exists():
    raise FileNotFoundError(f"找不到模型目录: {checkpoint_dir}")

candidate_dirs = [
    project_root / "demo_outputs" / "notebook_family_frames",
    project_root / "demo_outputs" / "example_scenes",
]
image_extensions = {".jpg", ".jpeg", ".png", ".heic", ".heif"}
filelist = []
input_dir = None
for candidate_dir in candidate_dirs:
    candidate_files = sorted(
        path
        for path in candidate_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    if len(candidate_files) >= 2:
        input_dir = candidate_dir
        filelist = candidate_files
        break

if len(filelist) < 2:
    # 让 Notebook 从仓库自带的视频自动生成输入，而不是要求手动准备图片。
    video_path = project_root / "demo_examples" / "family" / "Family.mp4"
    input_dir = project_root / "demo_outputs" / "notebook_family_frames"
    input_dir.mkdir(parents=True, exist_ok=True)
    print("没有找到现成图片，正在从示例视频按约 1 FPS 抽帧：", video_path)
    extract_frames_from_video(str(video_path), str(input_dir))
    filelist = sorted(
        path for path in input_dir.iterdir()
        if path.is_file() and path.suffix.lower() in image_extensions
    )

if len(filelist) < 2:
    raise RuntimeError("至少需要 2 张输入图片才能进行多视图推理。")

if device.type != "cuda":
    print("警告：当前没有 CUDA GPU；准备步骤可以继续，但 ViT-L/512 推理建议切换到 GPU 内核。")
print("设备:", device)
print("模型目录:", checkpoint_dir)
print("输入目录:", input_dir)
print("示例图片数量:", len(filelist))
print("前 5 张图片:")
for path in filelist[:5]:
    print(" -", path.relative_to(project_root))

/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/__init__.py:11: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/__init__.py:11: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/lightning_fabric/__init__.py:29: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:34: UnderReviewWarning: The feature generate_power_seq is currently marked under review. The compatibility with other Lightning projects

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
设备: cuda
模型目录: /home/yyz/fast3r/checkpoints/Fast3R_ViT_Large_512
输入目录: /home/yyz/fast3r/demo_outputs/notebook_family_frames
示例图片数量: 29
前 5 张图片:
 - demo_outputs/notebook_family_frames/frame_000000.jpg
 - demo_outputs/notebook_family_frames/frame_000001.jpg
 - demo_outputs/notebook_family_frames/frame_000002.jpg
 - demo_outputs/notebook_family_frames/frame_000003.jpg
 - demo_outputs/notebook_family_frames/frame_000004.jpg


In [10]:
max_views = 12
selected_filelist = [str(path) for path in filelist[:max_views]]
images = load_images(selected_filelist, size=512, verbose=True)

print(f"已加载 {len(images)} 个视角")
print("首个视角字段:", sorted(images[0].keys()))
print("首个视角图像形状:", tuple(images[0]["img"].shape))

>> Loading a list of 12 images
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000000.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000001.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000002.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000003.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000004.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000005.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000006.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/notebook_family_frames/frame_000007.jpg with resolution 854x480 --> 512x288
 - adding /home/yyz/fast3r/demo_outputs/n

In [11]:
ALLOW_CPU_INFERENCE = False
if device.type != "cuda" and not ALLOW_CPU_INFERENCE:
    raise RuntimeError("Fast3R ViT-L/512 主线推理建议使用 CUDA GPU；如需 CPU 试跑，请先把 ALLOW_CPU_INFERENCE 改为 True 并把 max_views 调小。")

print("正在加载 Fast3R 模型...")
model, lit_module = load_model(
    str(checkpoint_dir),
    device=device,
    is_lightning_checkpoint=False,
)
model.eval()
lit_module.eval()

# inference_multiview 内部把 "16-mixed" / "32" 作为精度开关。
# profiling=True 会调用 CUDA synchronize，因此 CPU 时必须关闭 profiling。
inference_precision = "16-mixed" if device.type == "cuda" else "32"
profiling_enabled = device.type == "cuda"

with torch.inference_mode():
    inference_result = inference(
        images,
        model,
        device,
        dtype=inference_precision,
        verbose=True,
        profiling=profiling_enabled,
    )
output_dict, profiling_info = (
    inference_result if profiling_enabled else (inference_result, None)
)

print("推理完成")
print("推理精度:", inference_precision)
print("模型前向耗时:", profiling_info.get("total_time", "未提供") if profiling_info else "CPU 未启用 profiling")
print("输出字段:", sorted(output_dict.keys()))
print("视角数量:", len(output_dict["views"]))
print("预测数量:", len(output_dict["preds"]))

正在加载 Fast3R 模型...
Loading weights from local directory
>> Inference with model on 12 images
encode_images time: 0.5055067539215088
pos emb time: 0.0033676624298095703
decoder time: 0.3022279739379883
head prepare input time: 0.0012867450714111328
head forward time: 0.38594746589660645
total Fast3R forward time: 1.1985869407653809
推理完成
模型前向耗时: 1.1985869407653809
输出字段: ['loss', 'preds', 'views']
视角数量: 12
预测数量: 12


In [12]:
print("=== 预测结构 ===")
for index, prediction in enumerate(output_dict["preds"][:3]):
    print(f"视角 {index}:")
    print("  字段:", sorted(prediction.keys()))
    for key in ["pts3d_in_other_view", "conf", "pts3d_local", "conf_local"]:
        value = prediction.get(key)
        if value is not None:
            print(f"  {key}: shape={tuple(value.shape)}, dtype={value.dtype}")

first_points = output_dict["preds"][0]["pts3d_in_other_view"]
first_confidence = output_dict["preds"][0]["conf"]
print("首个视角点图数值范围:", float(first_points.min()), float(first_points.max()))
print("首个视角置信度范围:", float(first_confidence.min()), float(first_confidence.max()))
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"模型参数量: {parameter_count / 1e6:.1f}M")

=== 预测结构 ===
视角 0:
  字段: ['conf', 'conf_local', 'pts3d_in_other_view', 'pts3d_local']
  pts3d_in_other_view: shape=(1, 288, 512, 3), dtype=torch.float32
  conf: shape=(1, 288, 512), dtype=torch.float32
  pts3d_local: shape=(1, 288, 512, 3), dtype=torch.float32
  conf_local: shape=(1, 288, 512), dtype=torch.float32
视角 1:
  字段: ['conf', 'conf_local', 'pts3d_in_other_view', 'pts3d_local']
  pts3d_in_other_view: shape=(1, 288, 512, 3), dtype=torch.float32
  conf: shape=(1, 288, 512), dtype=torch.float32
  pts3d_local: shape=(1, 288, 512, 3), dtype=torch.float32
  conf_local: shape=(1, 288, 512), dtype=torch.float32
视角 2:
  字段: ['conf', 'conf_local', 'pts3d_in_other_view', 'pts3d_local']
  pts3d_in_other_view: shape=(1, 288, 512, 3), dtype=torch.float32
  conf: shape=(1, 288, 512), dtype=torch.float32
  pts3d_local: shape=(1, 288, 512, 3), dtype=torch.float32
  conf_local: shape=(1, 288, 512), dtype=torch.float32
首个视角点图数值范围: -0.6345456838607788 1.9926409721374512
首个视角置信度范围: 1.0 37.087173461

In [13]:
poses_c2w_batch, estimated_focals = lit_module.estimate_camera_poses(
    output_dict["preds"],
    niter_PnP=100,
    focal_length_estimation_method="first_view_from_global_head",
)

camera_poses = poses_c2w_batch[0]
focals = estimated_focals[0]
print("相机数量:", len(camera_poses))
print("第一个相机位姿形状:", tuple(camera_poses[0].shape))
print("估计焦距数量:", len(focals))
print("第一个相机位姿:\n", camera_poses[0])
print("估计焦距:", [float(focal) for focal in focals])

相机数量: 12
第一个相机位姿形状: (4, 4)
估计焦距数量: 12
第一个相机位姿:
 [[ 9.9999356e-01 -2.7541840e-03  2.2939586e-03  8.4927487e-05]
 [ 2.7572943e-03  9.9999529e-01 -1.3537960e-03  8.4198039e-04]
 [-2.2902193e-03  1.3601125e-03  9.9999654e-01  1.0126389e-04]
 [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  1.0000000e+00]]
估计焦距: [311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125, 311.0323486328125]


In [14]:
import numpy as np

output_dir.mkdir(parents=True, exist_ok=True)
pointmaps = np.stack(
    [prediction["pts3d_in_other_view"].detach().cpu().numpy().squeeze(0)
     for prediction in output_dict["preds"]]
)
confidence_maps = np.stack(
    [prediction["conf"].detach().cpu().numpy().squeeze(0)
     for prediction in output_dict["preds"]]
)
poses_array = np.stack(
    [pose.detach().cpu().numpy() if hasattr(pose, "detach") else np.asarray(pose)
     for pose in camera_poses]
)
focals_array = np.asarray([float(focal) for focal in focals], dtype=np.float32)

result_path = output_dir / "reconstruction_results.npz"
np.savez_compressed(
    result_path,
    pointmaps=pointmaps,
    confidence_maps=confidence_maps,
    poses_c2w=poses_array,
    estimated_focals=focals_array,
    image_paths=np.asarray(selected_filelist),
)
print("复现结果已保存:", result_path)
print("点图数组:", pointmaps.shape)
print("置信度数组:", confidence_maps.shape)

复现结果已保存: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_results.npz
点图数组: (12, 288, 512, 3)
置信度数组: (12, 288, 512)


## 模块级验证

这一部分使用 forward hooks 观察一次真实推理中各组件的输出，不改变模型计算。每个组件都检查输出是否存在、形状是否合理、数值是否有限。

In [15]:
from collections.abc import Mapping


def tensor_summary(value):
    if torch.is_tensor(value):
        detached = value.detach()
        return {
            "type": "tensor",
            "shape": tuple(detached.shape),
            "dtype": str(detached.dtype),
            "device": str(detached.device),
            "finite": bool(torch.isfinite(detached).all().item()),
            "min": float(detached.min().item()) if detached.numel() else None,
            "max": float(detached.max().item()) if detached.numel() else None,
        }
    if isinstance(value, Mapping):
        return {str(key): tensor_summary(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [tensor_summary(item) for item in value[:3]]
    return {"type": type(value).__name__}


module_outputs = {}


def capture_output(module_name):
    def hook(_module, _inputs, output):
        module_outputs[module_name] = tensor_summary(output)
    return hook


hook_handles = [
    model.encoder.patch_embed.register_forward_hook(capture_output("patch_embed")),
    model.encoder.register_forward_hook(capture_output("encoder")),
    model.decoder.register_forward_hook(capture_output("decoder")),
    model.downstream_head.register_forward_hook(capture_output("global_head")),
]
if model.downstream_head_local is not None:
    hook_handles.append(
        model.downstream_head_local.register_forward_hook(capture_output("local_head"))
    )

hooked_result = inference(
    images,
    model,
    device,
    dtype=inference_precision,
    verbose=False,
    profiling=profiling_enabled,
)
hooked_output, hooked_profile = (
    hooked_result if profiling_enabled else (hooked_result, None)
)

for handle in hook_handles:
    handle.remove()

required_modules = {"patch_embed", "encoder", "decoder", "global_head"}
missing_modules = required_modules - module_outputs.keys()
if missing_modules:
    raise RuntimeError(f"没有捕获到模块输出: {sorted(missing_modules)}")

for module_name, summary in module_outputs.items():
    print(f"[{module_name}]\n{summary}")

hooked_predictions = hooked_output["preds"]
assert len(hooked_predictions) == len(images)
assert all("pts3d_in_other_view" in prediction for prediction in hooked_predictions)
assert all(torch.isfinite(prediction["pts3d_in_other_view"]).all() for prediction in hooked_predictions)
assert all(torch.isfinite(prediction["conf"]).all() for prediction in hooked_predictions)
print("模块级验证通过")

encode_images time: 0.2693517208099365
pos emb time: 0.0005466938018798828
decoder time: 0.29361510276794434
head prepare input time: 0.0002491474151611328
head forward time: 0.2619936466217041
total Fast3R forward time: 0.8259179592132568
[patch_embed]
[{'type': 'tensor', 'shape': (12, 576, 1024), 'dtype': 'torch.float16', 'device': 'cuda:0', 'finite': True, 'min': -13.984375, 'max': 13.3203125}, {'type': 'tensor', 'shape': (12, 576, 2), 'dtype': 'torch.int64', 'device': 'cuda:0', 'finite': True, 'min': 0.0, 'max': 31.0}]
[encoder]
[{'type': 'tensor', 'shape': (12, 576, 1024), 'dtype': 'torch.float32', 'device': 'cuda:0', 'finite': True, 'min': -8.948944091796875, 'max': 10.216102600097656}, {'type': 'tensor', 'shape': (12, 576, 2), 'dtype': 'torch.int64', 'device': 'cuda:0', 'finite': True, 'min': 0.0, 'max': 31.0}]
[decoder]
[{'type': 'tensor', 'shape': (1, 6912, 1024), 'dtype': 'torch.float32', 'device': 'cuda:0', 'finite': True, 'min': -8.948944091796875, 'max': 10.216102600097656

## 视角规模与性能验证

用相同场景的不同视角数量运行推理，记录前向时间和显存峰值。这个实验用于验证模型的多视图输入行为，不等同于论文完整 benchmark。

In [16]:
import gc
import time

benchmark_results = []
for view_count in [4, 8, 12]:
    benchmark_images = images[:view_count]
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
    start_time = time.perf_counter()
    benchmark_result = inference(
        benchmark_images,
        model,
        device,
        dtype=inference_precision,
        verbose=False,
        profiling=profiling_enabled,
    )
    benchmark_output, benchmark_profile = (
        benchmark_result if profiling_enabled else (benchmark_result, None)
    )
    elapsed = time.perf_counter() - start_time
    peak_memory_gb = (
        torch.cuda.max_memory_allocated(device) / 1024**3
        if device.type == "cuda"
        else float("nan")
    )
    benchmark_results.append(
        {
            "views": view_count,
            "total_time": (
                float(benchmark_profile["total_time"])
                if benchmark_profile is not None
                else float("nan")
            ),
            "wall_time": elapsed,
            "peak_memory_gb": peak_memory_gb,
            "output_count": len(benchmark_output["preds"]),
        }
    )
    del benchmark_images, benchmark_output, benchmark_profile
    gc.collect()

for result in benchmark_results:
    print(
        f"视角={result['views']:>2} | "
        f"模型时间={result['total_time']:.3f}s | "
        f"墙钟时间={result['wall_time']:.3f}s | "
        f"峰值显存={result['peak_memory_gb']:.2f}GB | "
        f"输出={result['output_count']}"
    )

assert all(result["output_count"] == result["views"] for result in benchmark_results)
print("视角规模验证通过")

encode_images time: 0.12180614471435547
pos emb time: 0.0004839897155761719
decoder time: 0.08476042747497559
head prepare input time: 0.0002491474151611328
head forward time: 0.07056307792663574
total Fast3R forward time: 0.27797818183898926
encode_images time: 0.15106964111328125
pos emb time: 0.0005750656127929688
decoder time: 0.16988062858581543
head prepare input time: 0.0001475811004638672
head forward time: 0.14459848403930664
total Fast3R forward time: 0.4664151668548584
encode_images time: 0.27707791328430176
pos emb time: 0.0007290840148925781
decoder time: 0.2924051284790039
head prepare input time: 0.00024580955505371094
head forward time: 0.30933308601379395
total Fast3R forward time: 0.8799545764923096
视角= 4 | 模型时间=0.278s | 墙钟时间=0.287s | 峰值显存=4.56GB | 输出=4
视角= 8 | 模型时间=0.466s | 墙钟时间=0.481s | 峰值显存=5.31GB | 输出=8
视角=12 | 模型时间=0.880s | 墙钟时间=0.900s | 峰值显存=6.13GB | 输出=12
视角规模验证通过


## 点云导出与结果检查

使用 global prediction 的点图和置信度生成彩色 PLY。置信度阈值采用全体点的百分位数，避免低置信度区域淹没可视化结果。

In [17]:
import numpy as np
import open3d as o3d

all_points = []
all_colors = []
all_confidences = []

for prediction, view in zip(output_dict["preds"], images):
    points = prediction["pts3d_in_other_view"].detach().cpu().numpy().squeeze(0)
    confidence = prediction["conf"].detach().cpu().numpy().squeeze(0)
    image_tensor = view["img"].detach().cpu().squeeze(0).permute(1, 2, 0).numpy()

    if image_tensor.min() < 0:
        image_tensor = (image_tensor + 1.0) / 2.0
    if image_tensor.max() > 1:
        image_tensor = image_tensor / 255.0
    colors = np.clip(image_tensor, 0.0, 1.0)

    all_points.append(points.reshape(-1, 3))
    all_colors.append(colors.reshape(-1, 3))
    all_confidences.append(confidence.reshape(-1))

all_points = np.concatenate(all_points, axis=0)
all_colors = np.concatenate(all_colors, axis=0)
all_confidences = np.concatenate(all_confidences, axis=0)

confidence_percentile = 45
confidence_threshold = np.percentile(all_confidences, confidence_percentile)
valid = (
    np.isfinite(all_points).all(axis=1)
    & np.isfinite(all_colors).all(axis=1)
    & np.isfinite(all_confidences)
    & (all_confidences >= confidence_threshold)
)

filtered_points = all_points[valid]
filtered_colors = all_colors[valid]
max_points = 1_000_000
if len(filtered_points) > max_points:
    sample_indices = np.linspace(0, len(filtered_points) - 1, max_points, dtype=int)
    filtered_points = filtered_points[sample_indices]
    filtered_colors = filtered_colors[sample_indices]

point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(filtered_points.astype(np.float64))
point_cloud.colors = o3d.utility.Vector3dVector(filtered_colors.astype(np.float64))
ply_path = output_dir / "reconstruction_pointcloud.ply"
written = o3d.io.write_point_cloud(str(ply_path), point_cloud)

if not written:
    raise RuntimeError(f"PLY 写入失败: {ply_path}")
print("置信度阈值:", float(confidence_threshold))
print("过滤后点数:", len(filtered_points))
print("PLY 已保存:", ply_path)
assert ply_path.exists() and ply_path.stat().st_size > 0

置信度阈值: 9.571455001831055
过滤后点数: 973464
PLY 已保存: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_pointcloud.ply


## 复现结果汇总与论文级评估边界

到这里完成的是本地 checkpoint 的组件级和端到端推理验证。论文中的 DTU、7-Scenes、Neural-RGBD 等指标还需要按照项目文档准备预处理数据，并运行 `configs/eval/` 下对应的评估配置。

In [18]:
summary = {
    "environment": {
        "conda": os.environ.get("CONDA_DEFAULT_ENV", "unknown"),
        "device": str(device),
        "cuda": bool(torch.cuda.is_available()),
    },
    "input_views": len(images),
    "predictions": len(output_dict["preds"]),
    "module_hooks": sorted(module_outputs.keys()),
    "pose_count": len(camera_poses),
    "pointcloud_points": len(filtered_points),
    "results_file": str(result_path),
    "ply_file": str(ply_path),
}

print("=== Fast3R 复现汇总 ===")
print("环境检查:", summary["environment"])
print("输入/预测视角:", summary["input_views"], "/", summary["predictions"])
print("已捕获模块:", ", ".join(summary["module_hooks"]))
print("相机位姿数量:", summary["pose_count"])
print("PLY 点数:", summary["pointcloud_points"])
print("NPZ:", summary["results_file"])
print("PLY:", summary["ply_file"])

assert summary["input_views"] == summary["predictions"] == summary["pose_count"]
assert {"patch_embed", "encoder", "decoder", "global_head"}.issubset(
    set(summary["module_hooks"])
)
assert Path(summary["results_file"]).exists()
assert Path(summary["ply_file"]).exists()
print("本地组件级复现全部通过")

=== Fast3R 复现汇总 ===
环境检查: {'conda': 'fast3r', 'device': 'cuda', 'cuda': True}
输入/预测视角: 12 / 12
已捕获模块: decoder, encoder, global_head, local_head, patch_embed
相机位姿数量: 12
PLY 点数: 973464
NPZ: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_results.npz
PLY: /home/yyz/fast3r/demo_outputs/notebook_inference/reconstruction_pointcloud.ply
本地组件级复现全部通过


## DTU 代表性数据集评测

DTU 是第一个正式评测集。下面保留数据检查单元；下载默认关闭。已有的22场景结果见末尾的论文核对单元；公开 HF 权重通过 `scripts/fast3r_hf_dtu_eval.py` 使用官方指标实现评测。

In [24]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

project_root = Path.cwd()
data_root = project_root / "data"
dtu_root = data_root / "dtu_test_mvsnet_release"
dtu_drive_url = "https://drive.google.com/drive/folders/1bqtcVf8lK4VC8LgG-SIGRBECcrFqM7Wy"
ALLOW_DATA_DOWNLOAD = False  # 改为 True 才会在 Notebook 中安装 gdown 并下载大文件

free_gb = shutil.disk_usage(project_root).free / 1024**3
print(f"当前磁盘可用空间: {free_gb:.1f} GB")
print("DTU 目标目录:", dtu_root)

if dtu_root.exists() and any(dtu_root.iterdir()):
    entries = [path for path in dtu_root.iterdir()]
    print(f"DTU 目录已存在，包含 {len(entries)} 个顶层条目。")
else:
    print("DTU 预处理数据尚未下载。")
    print("官方预处理数据来源:", dtu_drive_url)
    if not ALLOW_DATA_DOWNLOAD:
        print("为避免误触发大文件下载，默认不自动下载。请按 README 手动准备数据，或把 ALLOW_DATA_DOWNLOAD 改为 True。")
    elif free_gb < 12:
        print("警告：可用空间低于 12 GB，建议先清理磁盘后再下载。")
    else:
        try:
            import gdown
        except ImportError:
            print("正在将 gdown 安装到当前 Notebook 内核...")
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "gdown"],
                cwd=project_root,
                check=True,
            )
            importlib.invalidate_caches()
            import gdown

        data_root.mkdir(parents=True, exist_ok=True)
        print("开始下载 DTU 预处理数据，这可能需要较长时间。")
        try:
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "gdown",
                    "--folder",
                    dtu_drive_url,
                    "-O",
                    str(dtu_root),
                ],
                cwd=project_root,
                check=True,
            )
        except subprocess.CalledProcessError as error:
            print("DTU 下载失败，退出码:", error.returncode)
            print("可能原因是当前网络无法访问 Google Drive。")
            print("请在可访问该链接的网络中手动下载并解压到:", dtu_root)
        else:
            print("DTU 下载完成。")

当前磁盘可用空间: 9.0 GB
DTU 目标目录: /home/yyz/fast3r/data/dtu_test_mvsnet_release
DTU 目录已存在，包含 22 个顶层条目。


### 运行 DTU 评测

下面保留原始 Lightning 入口检查；缺少 last.ckpt 时该入口不运行。公开 HF 权重同样可以评测：使用 `scripts/fast3r_hf_dtu_eval.py`；Notebook 末尾读取已完成的结果。文件格式本身不能证明某项指标无法复现。

In [25]:
import shlex
import subprocess
from pathlib import Path

project_root = Path.cwd()
dtu_root = project_root / "data" / "dtu_test_mvsnet_release"
lightning_checkpoint = project_root / "checkpoints" / "last.ckpt"

if not dtu_root.exists():
    print("找不到 DTU 数据目录，请先运行上一个下载单元。")
elif not lightning_checkpoint.exists():
    print("找不到 Lightning checkpoint:", lightning_checkpoint)
    print("当前的 checkpoints/Fast3R_ViT_Large_512/model.safetensors 不能直接用于 fast3r/eval.py。")
    print("请将训练得到的 last.ckpt 放到上述路径，或修改 lightning_checkpoint 变量。")
else:
    dtu_dataset = (
        "[DTU(split='test', "
        f"ROOT='{dtu_root}', resolution=512, num_seq=1, "
        "full_video=True, kf_every=5)]"
    )
    command = [
        sys.executable,
        "fast3r/eval.py",
        "eval=ablation_recon_better_inference_hp/ablation_recon_better_inference_hp",
        f"ckpt_path={lightning_checkpoint}",
        "slurm_job_id=local",
        f"data.data_root={data_root}",
        f"data.data_module.validation_datasets={dtu_dataset}",
        "trainer.limit_val_batches=1",
    ]
    print("即将执行:")
    print(" ".join(shlex.quote(part) for part in command))
    subprocess.run(command, cwd=project_root, check=True)

找不到 Lightning checkpoint: /home/yyz/fast3r/checkpoints/last.ckpt
当前的 checkpoints/Fast3R_ViT_Large_512/model.safetensors 不能直接用于 fast3r/eval.py。
请将训练得到的 last.ckpt 放到上述路径，或修改 lightning_checkpoint 变量。


## 教学可视化：先看输入，再看模型信心

把原始图片和模型输出并排画出来，有助于理解一个关键事实：Fast3R 的输出不是一张普通深度图，而是每个像素对应的三维坐标 `(x, y, z)` 以及一个置信度。

In [ ]:
if "images" not in globals():
    print("尚未加载图片，请先运行前面的图片加载单元。")
else:
    import matplotlib.pyplot as plt
    from fast3r.dust3r.utils.image import rgb

    gallery_count = min(6, len(images))
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    for axis, view, path in zip(axes.flat, images[:gallery_count], selected_filelist[:gallery_count]):
        axis.imshow(rgb(view["img"][0]))
        axis.set_title(Path(path).name)
        axis.axis("off")
    for axis in axes.flat[gallery_count:]:
        axis.axis("off")
    fig.suptitle("输入的多视图图片")
    fig.tight_layout()
    plt.show()

In [ ]:
if "output_dict" not in globals():
    print("尚未完成推理，请先运行模型推理单元。")
else:
    import matplotlib.pyplot as plt
    from fast3r.dust3r.utils.image import rgb

    show_count = min(4, len(output_dict["preds"]))
    fig, axes = plt.subplots(show_count, 2, figsize=(9, 3 * show_count), squeeze=False)
    for row in range(show_count):
        image = rgb(images[row]["img"][0])
        confidence = output_dict["preds"][row]["conf"].detach().cpu().squeeze().numpy()
        low, high = np.percentile(confidence, [5, 95])
        axes[row, 0].imshow(image)
        axes[row, 0].set_title(f"view {row}: RGB")
        axes[row, 1].imshow(confidence, cmap="turbo", vmin=low, vmax=high)
        axes[row, 1].set_title(f"view {row}: confidence")
        axes[row, 0].axis("off")
        axes[row, 1].axis("off")
    fig.tight_layout()
    plt.show()

## 相机轨迹：从 3D 点图估计 camera-to-world 位姿

`estimate_camera_poses` 使用 Fast3R 的点图和 fast-PnP 估计每个视角的相机到世界变换。下面只画平移部分，因此它是一个直观的轨迹检查，不是带尺度对齐的相机姿态 benchmark。

In [ ]:
if "camera_poses" not in globals():
    print("尚未估计相机位姿，请先运行 PnP 单元。")
else:
    import matplotlib.pyplot as plt

    camera_centers = np.stack([np.asarray(pose)[:3, 3] for pose in camera_poses])
    fig = plt.figure(figsize=(8, 6))
    axis = fig.add_subplot(111, projection="3d")
    axis.plot(camera_centers[:, 0], camera_centers[:, 1], camera_centers[:, 2], "o-", linewidth=2)
    for index, center in enumerate(camera_centers):
        axis.text(*center, str(index), fontsize=8)
    axis.set_xlabel("x")
    axis.set_ylabel("y")
    axis.set_zlabel("z")
    axis.set_title("estimated camera trajectory")
    plt.tight_layout()
    plt.show()

## 点云预览：浏览器内查看稀疏采样

PLY 是给 Open3D / MeshLab 使用的持久化产物；这里用 Plotly 抽样最多 20,000 个点，避免 Notebook 输出过大。

In [ ]:
if "filtered_points" not in globals():
    print("尚未生成点云，请先运行 PLY 导出单元。")
else:
    import plotly.graph_objects as go

    rng = np.random.default_rng(7)
    preview_count = min(20_000, len(filtered_points))
    preview_indices = rng.choice(len(filtered_points), size=preview_count, replace=False)
    preview_points = filtered_points[preview_indices]
    preview_colors = (filtered_colors[preview_indices] * 255).astype(np.uint8)
    color_strings = [f"rgb({r},{g},{b})" for r, g, b in preview_colors]
    figure = go.Figure(go.Scatter3d(
        x=preview_points[:, 0], y=preview_points[:, 1], z=preview_points[:, 2],
        mode="markers", marker={"size": 2, "color": color_strings, "opacity": 0.75},
    ))
    figure.update_layout(title=f"Fast3R point cloud preview ({preview_count:,} points)", height=650,
                          scene_aspectmode="data")
    figure.show()

## 视角规模实验图

这个小实验记录的是本机前向耗时和显存随视角数的变化。它能支撑‘一次前向处理多视图’的工程观察，但不能替代论文中的完整数据集指标。

In [ ]:
if not globals().get("benchmark_results"):
    print("尚未运行视角规模实验。")
else:
    import matplotlib.pyplot as plt
    import pandas as pd
    from IPython.display import display

    benchmark_table = pd.DataFrame(benchmark_results)
    display(benchmark_table)
    axis = benchmark_table.plot(x="views", y="wall_time", marker="o", figsize=(7, 4), legend=False)
    axis.set_ylabel("wall time (s)")
    axis.set_title("inference time vs. number of views")
    plt.tight_layout()
    plt.show()

## 生成可交付的复现卡片

把 checkpoint、输入视角数、推理精度、输出文件和本地 benchmark 一起保存，面试时比只给一张截图更容易说明实验是如何复现的。

In [ ]:
import json
from datetime import datetime, timezone

if "output_dir" not in globals():
    print("尚未完成主线推理，暂不生成复现卡片。")
else:
    repro_card = {
        "project": "Fast3R",
        "purpose": "functional reproduction of multi-view inference",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "checkpoint": str(checkpoint_dir),
        "input_views": len(images),
        "image_size": list(images[0]["img"].shape[-2:]),
        "device": str(device),
        "precision": inference_precision,
        "forward_time_seconds": (float(profiling_info["total_time"]) if profiling_info else None),
        "point_count_after_confidence_filter": int(len(filtered_points)),
        "artifacts": {"npz": str(result_path), "ply": str(ply_path)},
        "benchmark": benchmark_results,
        "scope_note": "This card describes functional inference. Dataset experiments and paper comparisons are recorded separately in the paper audit cells.",
    }
    repro_card_path = output_dir / "reproducibility_card.json"
    repro_card_path.write_text(json.dumps(repro_card, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(repro_card, ensure_ascii=False, indent=2))
    print("复现卡片已保存:", repro_card_path)

## 面试 / 简历表述模板

完成主线后，可以用下面这段作为简历初稿，再把自己的实际硬件和复现卡片中的数字替换进去：

- **Fast3R 多视图 3D 重建复现（PyTorch / CUDA）**：基于官方 ViT-L/512 checkpoint，搭建从视频抽帧、图像归一化、多视图一次前向到 PnP 相机估计的端到端推理流程；导出带 RGB 与置信度筛选的 `.ply` 点云和 `.npz` 结果。
- 设计模块级 hook 与视角规模实验，检查 patch embedding、encoder、decoder、global/local head 的张量形状和有限性，并记录 4/8/12 视角的前向时间与显存。

面试时应说明实际覆盖范围：公开权重推理、DTU 22 场景评测，以及后面的推理视角数和点图头消融。论文数值存在差距，其余数据集和重新训练实验未完成。

## 论文逐项核对：复现到了哪里？

对应 [Fast3R arXiv v2](https://arxiv.org/html/2501.13928v2)，完整清单见 [PAPER_REPRODUCTION.md](PAPER_REPRODUCTION.md)。以下单元可独立运行，直接读取已保存的实测 JSON，无需 GPU。

| 步骤 | 论文位置 | 本项目验证 |
| --- | --- | --- |
| 1 | §3.1、§3.3、Figure 2 | 共享 encoder、全视角融合 Transformer、global/local 点图与置信度；上面的 hook 已运行 |
| 2 | §3.2，Eq. (1)–(3) | 阅读归一化回归和置信度损失；未重新训练 |
| 3 | §4.3、Table 4 | 22 场景 DTU，比较每场景 median 的均值 |
| 4 | §5.4、Table 5 | 同一次预测的 aligned local 与 global 对比 |
| 5 | §5.1、Figure 5 | 3/5/10/20 视角的均匀采样适配实验 |
| 6 | §4.1、Table 2 | 本机预热后重复计时、allocated/reserved 显存 |

完整复现仍缺 §4.2/Table 1、§4.3/Table 3、训练消融和附录实验。本机结果与论文环境不同，不能将脚本运行成功当成所有结论验证成功。

### 步骤 2：从论文公式找到代码

每个像素输出一个 3D 点。归一化回归先把预测点和 GT 点分别除以各自点到原点距离的平均值，再计算 L2 点误差，这样可减少整体尺度差异的影响。置信度损失让模型对可靠点赋予更高权重。

打开 `configs/model/fast3r.yaml`：训练配置连接 `ConfLossMultiviewV2(Regr3DMultiviewV4(L21Loss))`，alpha=0.2。实现位于 `fast3r/dust3r/losses.py`。代码使用 `conf * loss - alpha * log(conf)`，同时归一化 global/local 项；阅读论文 Eq. (3) 时注意 HTML 中 log 项符号与实际代码不同，以已发布实现为实验依据。合成点图的数值与梯度检查通过，但尚未从头训练。

`fast3r/models/fast3r.py` 中 `_get_random_image_pos` 固定第一视角为参考坐标系，对其他视角随机采样 image-index embedding。因此新实验必须保存 seed 与输入标签。

In [1]:
import json
from pathlib import Path
loss_checks = json.loads(Path('results/loss_checks.json').read_text())
assert loss_checks['status'] == 'passed' and loss_checks['finite_gradients']
print(json.dumps(loss_checks, indent=2, ensure_ascii=False))
print('这里是损失数值/反向传播验证，没有进行模型训练。')

{
  "paper_section": "3.2, Equations 1-3",
  "status": "passed",
  "scope": "Synthetic CPU loss/backprop invariants only; no model training or dataset benchmark.",
  "seed": 42,
  "views": 3,
  "confidence_alpha": 0.2,
  "exact_prediction_loss": -0.13862943649291992,
  "uniformly_scaled_prediction_loss": -0.13862931728363037,
  "perturbed_prediction_loss": -0.05105750262737274,
  "finite_gradients": true,
  "confidence_sign": "released implementation: conf * regression - alpha * log(conf)"
}
这里是损失数值/反向传播验证，没有进行模型训练。


In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

paper_root = Path.cwd()
assert (paper_root / 'README.md').exists(), '请在仓库根目录运行'
dtu_history = json.loads((paper_root / 'demo_outputs/paper_eval/dtu_all.json').read_text())
assert dtu_history['scene_count'] == len(dtu_history['scenes']) == 22
assert len({r['scene'] for r in dtu_history['scenes']}) == 22
for metric, value in dtu_history['aggregate_mean'].items():
    recomputed = np.mean([r['metrics'][metric] for r in dtu_history['scenes']])
    assert np.isfinite(value) and np.isclose(value, recomputed), metric
print('历史 DTU：22 场景完整，所有 aggregate_mean 与逐场景均值一致')
dtu_seeded = json.loads((paper_root / 'results/dtu_seed42_stride5.json').read_text())
assert dtu_seeded['scene_count'] == len(dtu_seeded['scenes']) == 22
assert dtu_seeded['seed'] == 42 and dtu_seeded['kf_every'] == 5
assert dtu_seeded['head_chunk_size'] == 2
for metric, value in dtu_seeded['aggregate_mean'].items():
    assert np.isclose(value, np.mean([r['metrics'][metric] for r in dtu_seeded['scenes']]))
comparison = pd.DataFrame([
    {'指标': 'Accuracy median', '本地历史运行': dtu_history['aggregate_mean']['accuracy_median'], '论文 Table 4': 1.706},
    {'指标': 'Completion median', '本地历史运行': dtu_history['aggregate_mean']['completion_median'], '论文 Table 4': 0.857},
])
comparison['seed=42 stride=5'] = [dtu_seeded['aggregate_mean']['accuracy_median'], dtu_seeded['aggregate_mean']['completion_median']]
comparison['新运行相对论文高出 (%)'] = (comparison['seed=42 stride=5'] / comparison['论文 Table 4'] - 1) * 100
display(comparison.round(4))
print('这两个指标越低越好；seeded 运行仍有数值差距。历史报告未记录 seed；新运行同时改了seed、遍历顺序与DPT分块，不能把差异全归因于seed。')
figure_dir = paper_root / 'results/figures'
figure_dir.mkdir(parents=True, exist_ok=True)

历史 DTU：22 场景完整，所有 aggregate_mean 与逐场景均值一致


,指标,本地历史运行,论文 Table 4,seed=42 stride=5,新运行相对论文高出 (%)
0,Accuracy median,2.9306,1.706,2.0827,22.0792
1,Completion median,1.8200,0.857,1.0311,20.3142


这两个指标越低越好；seeded 运行仍有数值差距。历史报告未记录 seed；新运行同时改了seed、遍历顺序与DPT分块，不能把差异全归因于seed。


### 步骤 4–5：消融控制变量

运行 `python scripts/fast3r_paper_experiments.py --output-json results/dtu_paper_experiments.json` 生成新实验。每次结果写入前先保存临时文件，再原子替换，避免把写了一半的 JSON 当成完整结果。

local/global 分支复用 10 视角的同一次预测。改变的只有用于指标的点图和相应置信度；两种分支都实际计算了两个 head，所以其比较不等于测量移除 local head 的速度。

视角数实验均匀抽取完整 49 帧中的 N 张，保留第一帧，属于 Figure 5 的本机适配。不同 N 也会改变用于评价的 GT 点集；它与官方 stride 采样不同。若观测到某指标没有单调改善，应如实报告。

In [3]:
ablation = json.loads((paper_root / 'results/dtu_paper_experiments.json').read_text())
assert ablation['status'] == 'completed', '等待实验完成；如有 OOM 请先检查报告'
expected = len(ablation['scenes_expected']) * len(ablation['view_counts_expected'])
assert len(ablation['rows']) == expected == 88
assert len({(r['scene'], r['view_count']) for r in ablation['rows']}) == expected
assert all(r['status'] == 'completed' for r in ablation['rows'])
records = []
for r in ablation['rows']:
    assert len(r['input_labels']) == r['view_count']
    for head, metrics in r['metrics'].items():
        assert all(np.isfinite(v) for v in metrics.values())
        records.append({'scene': r['scene'], 'views': r['view_count'], 'head': head, **metrics})
ablation_df = pd.DataFrame(records)
ablation_means = ablation_df.groupby(['views', 'head']).mean(numeric_only=True)
display(ablation_means.round(4))
for (views, head), row in ablation_means.iterrows():
    stored = ablation['aggregate'][f'{views}_views_{head}']
    assert stored['scene_count'] == 22
    for k, v in stored['aggregate_mean'].items():
        assert np.isclose(v, row[k]), (views, head, k)
paired = ablation_means.loc[10]
display(pd.DataFrame({'local': paired.loc['local'], 'global': paired.loc['global'],
                      'global - local': paired.loc['global'] - paired.loc['local']}).round(4))
print('全部 88 次前向与 22 组配对 head 评测通过；差值>0表示global距离误差更大。')

accuracy  accuracy_median  completion  completion_median  \
views head                                                               
3     local     6.3791           3.3735      5.6077             2.9197   
5     local     4.8601           2.6129      3.5799             1.8621   
10    global    4.3514           2.0541      2.7941             1.0239   
      local     4.1451           1.9145      2.3889             0.9070   
20    local     4.0098           1.7020      2.5146             0.9653   

                 nc1  nc1_median     nc2  nc2_median  
views head                                            
3     local   0.7058      0.7894  0.6841      0.7638  
5     local   0.6944      0.7771  0.6675      0.7411  
10    global  0.6600      0.7384  0.6283      0.6930  
      local   0.6769      0.7588  0.6432      0.7124  
20    local   0.6415      0.7152  0.6071      0.6643

,local,global,global - local
accuracy,4.1451,4.3514,0.2063
accuracy_median,1.9145,2.0541,0.1397
completion,2.3889,2.7941,0.4052
completion_median,0.9070,1.0239,0.1169
nc1,0.6769,0.6600,-0.0169
nc1_median,0.7588,0.7384,-0.0204
nc2,0.6432,0.6283,-0.0150
nc2_median,0.7124,0.6930,-0.0194


全部 88 次前向与 22 组配对 head 评测通过；差值>0表示global距离误差更大。


In [4]:
local_means = ablation_means.xs('local', level='head')
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric in zip(axes, ['accuracy_median', 'completion_median']):
    ax.plot(local_means.index, local_means[metric], 'o-')
    ax.set(xlabel='Input views', ylabel=metric, title='DTU: 22-scene mean of medians')
    ax.grid(alpha=0.3)
fig.suptitle('Figure 5 adaptation: uniform views, released HF weights')
fig.tight_layout()
fig.savefig(figure_dir / 'dtu_view_scaling.png', dpi=180)
plt.show()
for metric in ['accuracy_median', 'completion_median']:
    print(metric, '随视角数单调不增:', bool(np.all(np.diff(local_means[metric]) <= 0)))

<Figure size 1000x400 with 2 Axes>

accuracy_median 随视角数单调不增: True
completion_median 随视角数单调不增: False


### 步骤 6：解释性能测量（§4.1 / Table 2）

首个场景预热一次，再测量三次，记录模型内部分段计时以及包含 CPU 输出拷贝的 inference wall time。两种时间不是同一个概念。显存同时报告 PyTorch allocated 和 reserved；它们都不是 `nvidia-smi` 中含桌面与驱动开销的总占用。

论文 Table 2 是 A100 上 512×384 的输入；本实验是 RTX 5070 Ti Laptop 上的 512×512 DTU 图片，且 DPT head 分块为2。这里复现的是测量方法，不能拿绝对时间声称超越论文速度；也尚未运行 DUSt3R 对照。

In [5]:
timing_rows = []
for trial in ablation['performance']:
    assert trial['warmup'] == 1 and len(trial['samples']) == trial['repeats'] == 3
    timing_rows.append({'views': trial['view_count'],
        'forward_mean_s': np.mean([s['forward_seconds'] for s in trial['samples']]),
        'forward_std_s': np.std([s['forward_seconds'] for s in trial['samples']]),
        'wall_mean_s': np.mean([s['inference_wall_seconds'] for s in trial['samples']]),
        'peak_allocated_GiB': max(s['peak_allocated_gib'] for s in trial['samples']),
        'peak_reserved_GiB': max(s['peak_reserved_gib'] for s in trial['samples'])})
timing_df = pd.DataFrame(timing_rows).sort_values('views')
display(timing_df.round(4))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].errorbar(timing_df.views, timing_df.forward_mean_s, yerr=timing_df.forward_std_s, fmt='o-', capsize=4)
axes[0].set(xlabel='Input views', ylabel='Forward seconds (mean ± std)')
axes[1].plot(timing_df.views, timing_df.peak_allocated_GiB, 'o-', label='allocated')
axes[1].plot(timing_df.views, timing_df.peak_reserved_GiB, 's--', label='reserved')
axes[1].set(xlabel='Input views', ylabel='Peak GPU memory (GiB)')
axes[1].legend()
fig.suptitle('Table 2 methodology adaptation: RTX 5070 Ti Laptop, DPT chunk=2')
fig.tight_layout()
fig.savefig(figure_dir / 'inference_efficiency.png', dpi=180)
plt.show()

,views,forward_mean_s,forward_std_s,wall_mean_s,peak_allocated_GiB,peak_reserved_GiB
0,3,0.3108,0.0400,0.3189,4.4305,4.8730
1,5,0.5636,0.0659,0.5757,4.5649,5.1172
2,10,1.3030,0.0682,1.3254,4.9089,5.2969
3,20,2.9958,0.0083,3.0375,5.5827,5.9258


<Figure size 1000x400 with 2 Axes>

### 已生成的科研图

下面的 PNG 来自上述实测结果单元，GitHub 上也可直接查看：

![DTU view scaling](results/figures/dtu_view_scaling.png)

![Inference efficiency](results/figures/inference_efficiency.png)

### 已完成全量运行：Neural RGB-D（§4.3 / Table 3）

2026-10-01全量9场景评测正常退出，共278个采样视角；实际结果为 `results/nrgbd_seed42_stride40.json`。每40帧选一帧，使用完整有效轨迹，公开HF权重、seed42、512分辨率、aligned local点图、DPT chunk2。

Table 3 将距离乘100；加载器将毫米深度转为米，因此先在9场景间平均median距离再乘100。本地实测4.0165/1.2001，相比论文参考3.40/1.01分别高18.13%/18.82%，尚未匹配论文数值。公开权重与论文具体训练权重对应关系及原始下载revision未验证；chunk2是12GiB显存适配，不能认定其造成指标差距。

In [6]:
nrgbd_path = paper_root / 'results/nrgbd_seed42_stride40.json'
if not nrgbd_path.exists():
    print('Neural RGB-D：数据准备/评测尚未完成，当前没有本地指标。')
else:
    nrgbd_report = json.loads(nrgbd_path.read_text())
    assert nrgbd_report['dataset'] == 'nrgbd' and nrgbd_report['kf_every'] == 40
    assert nrgbd_report['seed'] == 42 and nrgbd_report['head'] == 'local'
    assert nrgbd_report['paper_distance_multiplier'] == 100
    nrgbd_manifest = json.loads((paper_root / 'results/nrgbd_data_manifest.json').read_text())
    assert nrgbd_report['scene_count'] == len(nrgbd_report['scenes']) == len(nrgbd_manifest['scene_names'])
    assert {r['scene'] for r in nrgbd_report['scenes']} == set(nrgbd_manifest['scene_names'])
    assert nrgbd_report['scene_count'] == len({r['scene'] for r in nrgbd_report['scenes']}) == 9
    for k, v in nrgbd_report['aggregate_mean'].items():
        values = [r['metrics'][k] for r in nrgbd_report['scenes']]
        assert np.isfinite(values).all() and np.isclose(v, np.mean(values), rtol=1e-12, atol=1e-12)
    nrgbd_comparison = pd.DataFrame([
        {'metric': 'Accuracy median', 'local x100': nrgbd_report['aggregate_mean']['accuracy_median'] * 100, 'paper Table 3': 3.40},
        {'metric': 'Completion median', 'local x100': nrgbd_report['aggregate_mean']['completion_median'] * 100, 'paper Table 3': 1.01},
    ])
    nrgbd_comparison['error increase %'] = (nrgbd_comparison['local x100'] / nrgbd_comparison['paper Table 3'] - 1) * 100
    display(nrgbd_comparison.round(4))
    nrgbd_scenes = pd.DataFrame([{'scene': r['scene'], 'views': r['views'],
        'Accuracy median x100': r['metrics']['accuracy_median'] * 100,
        'Completion median x100': r['metrics']['completion_median'] * 100}
        for r in nrgbd_report['scenes']])
    display(nrgbd_scenes.round(4))
    print(f"已核验 {len(nrgbd_scenes)} 个场景，共 {nrgbd_scenes['views'].sum()} 个采样视角；未匹配论文指标。")

,metric,local x100,paper Table 3,error increase %
0,Accuracy median,4.0165,3.40,18.1331
1,Completion median,1.2001,1.01,18.8220


,scene,views,Accuracy median x100,Completion median x100
0,breakfast_room,30,2.1250,0.4672
1,complete_kitchen,31,5.5303,1.9673
2,green_room,37,2.2289,0.4938
3,grey_white_room,38,7.5022,2.5886
4,kitchen,38,3.1274,1.2414
5,morning_apartment,23,1.0980,0.5411
6,staircase,29,5.0105,1.2555
7,thin_geometry,10,8.0700,1.8468
8,whiteroom,42,1.4563,0.3993


已核验 9 个场景，共 278 个采样视角；未匹配论文指标。


### 后续真实实验：配对点图消融与7-Scenes

当前归档：NRGBD全部9场景配对已完成。local/global的Accuracy mean×100为9.7108/9.2594，Completion mean×100为3.1292/3.1789；local提高normal consistency并降低Completion，但Accuracy没有改善，因此没有全面复现论文local优势。所有local指标与历史seed42/stride40运行逐项完全相等；这一重复验证不保证跨硬件逐bit一致。7-Scenes于2026-10-01 12:44:47（Asia/Shanghai）正常结束，覆盖全部7类场景、18条测试轨迹、850视角。Table3 median×100为3.3035/3.1058；Table5 mean×100为local6.3613/7.0516、global6.9567/6.3510，local不改善Completion，未匹配论文数值。

§5.4/Table 5：新增 `--head both`，同一输入、同一次网络预测依次测 aligned local 和 global，不能把独立随机前向的两个结果当作严格配对。选择metric点图不跳过local head计算，因此本实验不证明移除head的速度收益。下表使用mean距离，与Table 3/4的median不是同一指标。

§4.3/Table 3：7-Scenes由Microsoft官方TestSplit选择全部测试轨迹；仅存储原始stride20帧以节省磁盘，并在加载器中使用原始帧总数恢复完全相同的采样编号。非full_video或其他stride将报错。深度使用固定版本SimpleRecon的depth-to-RGB标定投影与z-buffer，不是TSDF重建深度，也不是把原始depth直接改名。保留测试序列ZIP的CRC/SHA256、源ETag与预处理源码SHA256；数据仅供非商业学术复现，不上传GitHub。

这里只在完整JSON存在且场景/轨迹集合和汇总指标通过核验后显示结果；运行中不以论文参考值填表。

![NRGBD paired head ablation](results/figures/nrgbd_paired_heads.png)

![7-Scenes paired head ablation](results/figures/7scenes_paired_heads.png)

In [7]:
paired_rows = []
for dataset_name, filename in [('nrgbd', 'nrgbd_paired_seed42_stride40.json'), ('7scenes', '7scenes_paired_seed42_stride20.json')]:
    path = paper_root / 'results' / filename
    if not path.exists():
        print(f'{dataset_name}：全量配对结果尚未生成。')
        continue
    report = json.loads(path.read_text())
    assert report['dataset'] == dataset_name and report['head'] == 'both' and report['paired_same_forward']
    assert report['aggregate_mean_head'] == 'local' and report['aggregate_mean'] == report['aggregate_by_head']['local']
    assert report['scene_count'] == len(report['scenes']) == len({r['scene'] for r in report['scenes']})
    if dataset_name == 'nrgbd':
        manifest = json.loads((paper_root / 'results/nrgbd_data_manifest.json').read_text())
        expected = set(manifest['scene_names'])
        assert len(expected) == 9
        old = json.loads((paper_root / 'results/nrgbd_seed42_stride40.json').read_text())
        old_rows = {r['scene']: r for r in old['scenes']}
        assert all(r['metrics_by_head']['local'] == old_rows[r['scene']]['metrics'] and r['seed'] == old_rows[r['scene']]['seed'] and r['views'] == old_rows[r['scene']]['views'] for r in report['scenes'])
    else:
        manifest = json.loads((paper_root / 'results/7scenes_data_manifest.json').read_text())
        assert manifest['status'] == 'prepared' and len(manifest['scenes_expected']) == 7
        expected = {r['scene'] + '/' + r['sequence'] for r in manifest['sequences']}
    assert {r['scene'] for r in report['scenes']} == expected
    for head in ['local', 'global']:
        for metric, value in report['aggregate_by_head'][head].items():
            values = [r['metrics_by_head'][head][metric] for r in report['scenes']]
            assert np.isfinite(values).all() and np.isclose(value, np.mean(values), rtol=1e-12, atol=1e-12)
        paired_rows.append({'dataset': dataset_name, 'head': head, 'trajectories': len(expected),
            'Accuracy mean x100': report['aggregate_by_head'][head]['accuracy'] * 100,
            'Completion mean x100': report['aggregate_by_head'][head]['completion'] * 100})
    assert all(r['paired_same_forward'] for r in report['scenes'])
if paired_rows:
    display(pd.DataFrame(paired_rows).round(4))
    print('这是Table 5 mean口径的本地配对实验，不是Table 3 median或整篇论文完成。')
nrgbd_paired_path = paper_root / 'results/nrgbd_paired_seed42_stride40.json'
if nrgbd_paired_path.exists():
    nrgbd_paired = json.loads(nrgbd_paired_path.read_text())
    deltas = pd.DataFrame([{'scene': r['scene'],
        'Accuracy': (r['metrics_by_head']['global']['accuracy'] - r['metrics_by_head']['local']['accuracy']) * 100,
        'Completion': (r['metrics_by_head']['global']['completion'] - r['metrics_by_head']['local']['completion']) * 100}
        for r in nrgbd_paired['scenes']]).set_index('scene')
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for ax, metric in zip(axes, ['Accuracy', 'Completion']):
        values = deltas[metric]
        ax.barh(deltas.index.str.replace('_', ' '), values, color=['#2563eb' if v < 0 else '#d97706' for v in values])
        ax.axvline(0, color='black', linewidth=1)
        ax.set_title(metric + ' mean distance x100')
        ax.set_xlabel('Global minus aligned local (negative: global better)')
        ax.grid(axis='x', alpha=.25)
    fig.suptitle('NRGBD paired head ablation: 9 scenes, same forward predictions')
    fig.tight_layout()
    fig.savefig(paper_root / 'results/figures/nrgbd_paired_heads.png', dpi=160, bbox_inches='tight')
    plt.show()

,dataset,head,trajectories,Accuracy mean x100,Completion mean x100
0,nrgbd,local,9,9.7108,3.1292
1,nrgbd,global,9,9.2594,3.1789
2,7scenes,local,18,6.3613,7.0516
3,7scenes,global,18,6.9567,6.3510


这是Table 5 mean口径的本地配对实验，不是Table 3 median或整篇论文完成。


<Figure size 1300x500 with 2 Axes>

In [8]:
seven_path = paper_root / 'results/7scenes_paired_seed42_stride20.json'
if not seven_path.exists():
    print('7-Scenes：完整评测尚未结束，暂不报告本地指标。')
    prepared_path = paper_root / 'results/7scenes_data_manifest.json'
    if prepared_path.exists():
        prepared = json.loads(prepared_path.read_text())
        if prepared['status'] == 'prepared':
            assert len(prepared['scenes_expected']) == 7 and prepared['trajectory_count'] == len(prepared['sequences']) == 18
            assert len({(r['scene'], r['sequence']) for r in prepared['sequences']}) == 18
            assert all(r['stored_indices'] == list(range(0, r['original_frame_count'], 20)) and r['selected_member_crc_checked'] and r['sequence_zip_crc_checked'] for r in prepared['sequences'])
            print(f"归档清单：7类场景、18测试轨迹、{sum(len(r['stored_indices']) for r in prepared['sequences'])}个stride20视角已准备。")
else:
    seven = json.loads(seven_path.read_text())
    from scripts.validate_7scenes_report import validate as validate_seven_report
    seven_validation = validate_seven_report(paper_root)
    print('完整集合、原始采样编号、seed、有限逐项aggregate及same-forward核验：', seven_validation['status'])
    assert seven['dataset'] == '7scenes' and seven['kf_every'] == 20 and seven['seed'] == 42
    assert seven['aggregate_mean_head'] == 'local' and seven['paper_distance_multiplier'] == 100
    seven_comparison = pd.DataFrame([
        {'metric': 'Accuracy median', 'local x100': seven['aggregate_mean']['accuracy_median'] * 100, 'paper Table 3': 1.58},
        {'metric': 'Completion median', 'local x100': seven['aggregate_mean']['completion_median'] * 100, 'paper Table 3': 0.93}])
    seven_comparison['error increase %'] = (seven_comparison['local x100'] / seven_comparison['paper Table 3'] - 1) * 100
    display(seven_comparison.round(4))
    print(f"7类场景、{seven['scene_count']}条测试轨迹、{seven_validation['sampled_views']}视角：先按官方实现逐轨迹median再平均、乘100。")
    seven_rows = pd.DataFrame([{'trajectory': r['scene'], 'views': r['views'],
        'Accuracy median x100': r['metrics_by_head']['local']['accuracy_median'] * 100,
        'Completion median x100': r['metrics_by_head']['local']['completion_median'] * 100}
        for r in seven['scenes']]).set_index('trajectory')
    display(seven_rows.round(4))
    deltas = pd.DataFrame([{'trajectory': r['scene'],
        'Accuracy': (r['metrics_by_head']['global']['accuracy'] - r['metrics_by_head']['local']['accuracy']) * 100,
        'Completion': (r['metrics_by_head']['global']['completion'] - r['metrics_by_head']['local']['completion']) * 100}
        for r in seven['scenes']]).set_index('trajectory')
    fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
    for ax, metric in zip(axes, ['Accuracy', 'Completion']):
        values = deltas[metric]
        ax.barh(deltas.index, values, color=['#2563eb' if v < 0 else '#d97706' for v in values])
        ax.axvline(0, color='black', linewidth=1)
        ax.set_title(metric + ' mean distance x100')
        ax.set_xlabel('Global minus aligned local (negative: global better)')
        ax.grid(axis='x', alpha=.25)
    fig.suptitle('7-Scenes paired head ablation: all 18 test trajectories, same forward')
    fig.tight_layout()
    fig.savefig(paper_root / 'results/figures/7scenes_paired_heads.png', dpi=160, bbox_inches='tight')
    plt.show()
    print('local改善汇总Accuracy，但Completion更差；公开权重与完整协议对应关系仍需核查，不能宣称整篇论文完成。')

完整集合、原始采样编号、seed、有限逐项aggregate及same-forward核验： passed


,metric,local x100,paper Table 3,error increase %
0,Accuracy median,3.3035,1.58,109.0794
1,Completion median,3.1058,0.93,233.9550


7类场景、18条测试轨迹、850视角：先按官方实现逐轨迹median再平均、乘100。


,views,Accuracy median x100,Completion median x100
trajectory,,,
chess/seq-03,50,0.4338,0.3685
chess/seq-05,50,1.0184,0.4722
fire/seq-03,50,0.2279,0.2514
fire/seq-04,50,0.1922,0.2228
heads/seq-01,50,1.3783,0.4870
office/seq-02,50,5.7091,1.8128
office/seq-06,50,17.9334,33.2884
office/seq-07,50,5.5626,3.5821
office/seq-09,50,5.7412,3.0902


<Figure size 1400x800 with 2 Axes>

local改善汇总Accuracy，但Completion更差；公开权重与完整协议对应关系仍需核查，不能宣称整篇论文完成。


### 哪些仍未完成？

1. §4.2/Table 1：CO3D 与 RealEstate10K 位姿 GT、划分与 RRA/RTA/mAA。PnP 输出或一张相机轨迹图不能替代这些 benchmark。
2. §4.3/Table 3/4：DTU22场景、Neural RGB-D9场景与7-Scenes全部18条测试轨迹已运行核验；论文数值仍未对齐，下一步做权重与协议差异审计。
3. §5.1 训练视角、§5.2 模型/数据 scaling、§5.3 无位置插值：需要各组独立训练模型。论文 §4 使用128张 A100-80GB，当前单卡无法按原规格训练。
4. 附录 C/D/E：Gaussian splatting、BA 与多视图深度；以及 §4.1 的大规模视角和 DUSt3R 基线。

合理的简历表述：基于官方公开权重搭建多视图重建流程，完成DTU22场景与Neural RGB-D9场景评测，记录论文指标差距，并设计视角规模与local/global点图消融。查看上面的实测表后再补入具体数字。

## 步骤6：公开权重与位姿数据预检查（§3.2、§4.2–4.3、§5.4）

三个重建数据集的实测已经保存，但整篇论文仍未完成。[协议审计](PROTOCOL_AUDIT.md)记录了当前公开HF权重的哈希匹配、公式与代码的置信度项符号差异、同像素RoMa注册与论文ICP文字的差异。它们是已核实事项，不是已证明的误差根因。

下面读取审计快照并再次检查位姿数据入口，不加载模型、不占用GPU、不下载数据。RealEstate10K规定文件实际包含1,832个唯一ID；预检查拒绝静默跳过缺失视频。CO3D需要可核验来源的selected_seqs_test.json。数据检查ready也只说明给定清单文件可读，不能证明论文协议一致。

上述是步骤6的历史资源快照；2026-10-02用户已扩容，下面步骤7更新数据准备状态。独立训练消融仍需不同训练模型的权重/算力。没有Table 1实测，不用PnP Demo填充论文成绩。

In [9]:
from pathlib import Path
import json
import sys
import pandas as pd

snapshot = json.loads(Path("results/protocol_audit_20261001.json").read_text())
checkpoint = snapshot["checkpoint"]
manifest = json.loads(Path("results/checkpoint_manifest.json").read_text())
for entry in manifest["files"]:
    key = "config" if entry["path"].endswith("config.json") else "weight"
    assert entry["sha256"] == checkpoint[f"local_{key}_sha256"]
    assert checkpoint[f"local_{key}_sha256"] == checkpoint[f"remote_{'lfs_' if key == 'weight' else ''}{key}_sha256"]
print("审计快照：", snapshot["checked_at_utc"])
print("公开HF文件哈希匹配；原始下载revision及论文实验权重映射仍未验证。")
print("当前公开revision：", checkpoint["current_remote_revision"])

sys.path.insert(0, str(Path("scripts").resolve()))
from check_pose_data import check_co3d, check_re10k
paths = snapshot["pose_data_preflight"]
co3d = check_co3d(paths["co3d_processed_root"])
re10k = check_re10k(paths["re10k_video_root"], paths["re10k_metadata_root"], paths["re10k_split_file"])
assert re10k["split_sha256"] == paths["re10k_split_sha256"]
print(pd.DataFrame([
    {"数据": "CO3D", "预检查": co3d["status"], "缺失/失败项": len(co3d["issues"])},
    {"数据": "RealEstate10K", "预检查": re10k["status"], "缺失/失败项": len(re10k["issues"])},
]).to_string(index=False))
print(f"RE10K指定{re10k['expected_scene_count']}个唯一视频，已准备{re10k['ready_scene_count']}个。")
print("10项合成预检查测试通过，不是RRA/RTA/mAA测评结果。")
print("本节不运行训练/位姿评测，整篇论文复现未完成。")

审计快照： 2026-10-01T05:07:25Z
公开HF文件哈希匹配；原始下载revision及论文实验权重映射仍未验证。
当前公开revision： a2c770b768ceb3a53c36c4f7a3619db0413dc3a1


           数据        预检查  缺失/失败项
         CO3D incomplete       1
RealEstate10K incomplete     523
RE10K指定1832个唯一视频，已准备1309个。
10项合成预检查测试通过，不是RRA/RTA/mAA测评结果。
本节不运行训练/位姿评测，整篇论文复现未完成。


## 步骤7：Table 1数据准备与可恢复位姿入口（2026-10-02）

用户已扩容，之前的磁盘限制解除。官方RE10K规定1832个相机TXT已完整准备，RGB归档由独立systemd服务下载；元数据齐备或归档下载完成不代表RGB/GT通过核验，更不代表位姿指标完成。

新单卡HF入口复用官方principal-point crop、focal/PnP与相对位姿指标，固定10视角采样，逐scene保存输入/协议SHA与真实poses。恢复时重新计算已有结果；缺场景不得静默跳过，PnP失败保留并计数。16-mixed、DPT chunk2、顺序固定OpenCV seed是本机适配，不冒充未公布论文采样。

下单元只验证相机记录及下载状态，不加载模型、不启动网络下载。[续接检查点](CONTINUATION.md)记录后台服务、日志、恢复命令与后续GT审计。额度不足时后台下载仍可运行；定时任务只能在后续触发且额度可用时尝试接续，不能保证重置瞬间恢复。

In [10]:
import hashlib
from check_pose_data import read_split, read_re10k_metadata
from prepare_re10k_metadata import checksum, EXPECTED_MD5
metadata_path = Path('results/re10k_metadata_manifest.json')
if not metadata_path.exists():
    print('官方相机元数据未完成；不填报Table 1分数。')
else:
    meta = json.loads(metadata_path.read_text())
    names = read_split('scripts/re10k_test_1800.txt')
    assert meta['status'] == 'metadata_prepared'
    assert meta['split_sha256'] == checksum('scripts/re10k_test_1800.txt')
    assert meta['archive_md5'] == EXPECTED_MD5
    assert set(meta['records']) == set(names)
    for name in names:
        path = Path(meta['metadata_root']) / (name + '.txt')
        assert checksum(path) == meta['records'][name]['sha256']
        assert len(read_re10k_metadata(path)) == meta['records'][name]['camera_record_count']
    print(f'官方相机TXT：{len(names)}/1832全部哈希与解析通过。')
    print('archive SHA256:', meta['archive_sha256'])
    partial = Path('data/re10k_test_only.zip.part')
    print('RGB归档当前字节：', partial.stat().st_size if partial.exists() else 0)
    print('RGB覆盖/GT与裁剪协议仍待审计；没有正式RRA/RTA/mAA分数。')
    print('新恢复入口的合成测试通过不等于完成真实GPU位姿评测。')
recovery_path = Path('results/re10k_rgb_recovery_20261002.json')
if recovery_path.exists():
    recovery = json.loads(recovery_path.read_text())
    assert recovery['expected_bytes'] - recovery['preserved_bytes'] == recovery['missing_bytes']
    assert not recovery['formal_pose_metrics_available']
    print('下载恢复历史检查点：', recovery['restart_time'], recovery['status'])
    print('保留字节/当时缺少字节：', recovery['preserved_bytes'], recovery['missing_bytes'])
    print('先重新传输并核对前缀，不代表下载完成；具体网络根因未确定。')
smoke_path = Path('results/re10k_smoke_seed42.json')
if smoke_path.exists():
    smoke = json.loads(smoke_path.read_text())
    assert smoke['protocol']['benchmark_scope'] == 'adaptation_smoke'
    assert smoke['scene_count'] == 1
    from fast3r_hf_re10k_pose_eval import aggregate, pose_metrics
    assert aggregate(smoke['scenes'], read_split('results/re10k_smoke_split.txt')) == smoke['aggregate_mean']
    for row in smoke['scenes']:
        recalculated, errors = pose_metrics(row['predicted_c2w'], row['gt_c2w'])
        assert recalculated == row['metrics']
        assert len(errors['rotation_deg']) == row['pair_count'] == 45
    print('真实GPU接线smoke：1场景/10视角/45对，PnP失败=', smoke['pnp_failed_view_count'])
    print(pd.Series(smoke['aggregate_mean']).to_string())
    print('以上只说明入口运行成功，不是Table1全量成绩，不说明达到了论文指标。')

官方相机TXT：1832/1832全部哈希与解析通过。
archive SHA256: af18e4d560ee9f73120128e53a56da215e118f3fb52018c14a66aa0f8ed691db
RGB归档当前字节： 55604889849
RGB覆盖/GT与裁剪协议仍待审计；没有正式RRA/RTA/mAA分数。
新恢复入口的合成测试通过不等于完成真实GPU位姿评测。
下载恢复历史检查点： 2026-10-02T03:52:39+08:00 recovery_started_not_download_complete
保留字节/当时缺少字节： 54818855664 786034185
先重新传输并核对前缀，不代表下载完成；具体网络根因未确定。


真实GPU接线smoke：1场景/10视角/45对，PnP失败= 0
RRA_at_5     0.555556
RRA_at_15    1.000000
RRA_at_30    1.000000
RTA_at_5     0.000000
RTA_at_15    0.088889
RTA_at_30    0.466667
mAA_30       0.162724
以上只说明入口运行成功，不是Table1全量成绩，不说明达到了论文指标。


## 步骤8：完整归档与不完整测试覆盖（§4.2 / Table 1）

55.60GB RGB归档下载服务已正常结束；ZIP索引7286个视频中，只含规定1832个ID里的1756个，缺76个。这里保持原测试清单，不能把交集称为论文全量成绩。下单元验证真实索引集合与下载清单，不加载网络模型。

后台准备已正常结束0，1756场景265447帧通过CRC、safe weights-only tensor、官方完整timestamp/camera与图片核验，保存全部候选帧和输入SHA。独立全部已保存RGB字节/GT/清单哈希核验通过。原图存在640×338、640×272等尺寸，保留实际宽高和编码字节，不强制拉伸至640×360。首个固定尺寸检查安全停止后修正了过强的格式假设，旧清单保留；58项离线测试通过。完整RGB/裁剪与论文权重/采样等价性仍未证实。

缺失来源的匿名可行性检查未使用cookies/账户，也未绕过限制。其他镜像或有界流式归档审计待继续；不把两个URL的429/登录报错推广为全部76视频永久不可用。

In [1]:
from pathlib import Path
import json, hashlib, zipfile, sys
import numpy as np
sys.path.insert(0, str(Path.cwd() / 'scripts'))
from check_pose_data import read_split
from prepare_re10k_metadata import checksum
archive_report = json.loads(Path('results/re10k_rgb_archive_manifest.json').read_text())
index_audit = json.loads(Path('results/re10k_rgb_index_audit.json').read_text())
assert Path(archive_report['archive_path']).stat().st_size == archive_report['archive_bytes'] == 55604889849
assert checksum('scripts/re10k_test_1800.txt') == index_audit['split_sha256']
assert index_audit['archive_sha256'] == archive_report['archive_sha256']
with zipfile.ZipFile(archive_report['archive_path']) as z:
    raw_index = z.read(index_audit['index_member'])
assert hashlib.sha256(raw_index).hexdigest() == index_audit['index_sha256']
mirror_ids = set(json.loads(raw_index))
prescribed_ids = set(read_split('scripts/re10k_test_1800.txt'))
covered = prescribed_ids & mirror_ids
missing = prescribed_ids - mirror_ids
assert covered == set(index_audit['covered_scene_ids'])
assert missing == set(index_audit['missing_scene_ids'])
assert len(covered) == index_audit['covered_scene_count'] == 1756
assert len(missing) == index_audit['missing_scene_count'] == 76
assert len(prescribed_ids) == 1832 and not index_audit['full_scene_coverage_verified']
print('归档字节/观测SHA256：', archive_report['archive_bytes'], archive_report['archive_sha256'])
print(f'规定覆盖 {len(covered)}/{len(prescribed_ids)}，缺 {len(missing)}；缺失集合完整保留。')
prepared_path = Path('results/re10k_rgb_prepared_manifest.json')
if prepared_path.exists():
    prepared = json.loads(prepared_path.read_text())
    assert set(prepared['scenes']) == covered
    assert set(prepared['missing_scene_ids']) == missing
    assert prepared['frame_count'] == sum(r['frames'] for r in prepared['scenes'].values())
    assert all(np.isfinite(r['maximum_camera_abs_delta']) and r['maximum_camera_abs_delta'] <= 1e-4 for r in prepared['scenes'].values())
    print('已覆盖部分准备结果：', prepared['status'], prepared['prepared_scene_count'], '场景', prepared['frame_count'], '帧')
else:
    print('逐块GT/CRC/RGB准备仍pending；本单元不把索引存在当作RGB就绪。')
verified_path = Path('results/re10k_rgb_verification_20261002.json')
if verified_path.exists():
    verified = json.loads(verified_path.read_text())
    assert verified['prepared_manifest_sha256'] == checksum(prepared_path)
    assert verified['verified_scene_count'] == len(covered) and verified['verified_frame_count'] == prepared['frame_count']
    assert not verified['full_scene_coverage_verified']
    print('独立RGB字节/GT SHA/候选集合核验通过：', verified['verified_frame_count'], '帧')
    print('原图H×W场景数：', verified['shape_h_w_scene_counts'])
print('正式Table1尚无成绩；未改变规定清单，也未运行缺76场景的全量评测。')

归档字节/观测SHA256： 55604889849 ce351771c966fb25ef41efc561a313ef40607c9aa8ea904ed8d582b361408097
规定覆盖 1756/1832，缺 76；缺失集合完整保留。
已覆盖部分准备结果： covered_prepared_split_incomplete 1756 场景 265447 帧
独立RGB字节/GT SHA/候选集合核验通过： 265447 帧
原图H×W场景数： {'272x640': 1, '338x640': 2, '360x640': 1753}
正式Table1尚无成绩；未改变规定清单，也未运行缺76场景的全量评测。


## 步骤9：缺失RGB来源与CO3D测试选择（§4.2 / Table 1）

真实8MiB HTTP Range/gzip探测通过，仅观察到1个场景，尚未观察76缺失ID；前缀SHA不是整包SHA。独立后台扫描只将缺失ID的匹配timestamp PNG原字节保存在候选目录，单图8MiB、总保存16GiB、磁盘留1GiB，不覆盖原JPEG、不自动认定图片/GT几何等价。

CO3D先准备51类官方元数据ZIP，以固定DUSt3R来源的fewview_train/test、quality>0.5、seed42+category index、每类最多50序列重建测试选择；未验证作者processed清单等价性，尚未获得RGB/depth/mask/NPZ。论文描述41类物体中的未见轨迹，不是41个未见类别；100 @配置是采样长度包装，不代表全序列遍历。以下只核验真实数据准备报告，不产生位姿成绩。

In [1]:
from pathlib import Path
import json, hashlib
probe = json.loads(Path('results/re10k_missing_prefix_probe_20261002.json').read_text())
missing_index = json.loads(Path('results/re10k_rgb_index_audit.json').read_text())
assert probe['status'] == 'bounded_prefix_only' and probe['compressed_bytes_read'] == 8 * 1024**2
assert not probe['full_archive_sha_verified'] and not probe['formal_pose_metrics_available']
assert set(probe['target_ids_observed']) | set(probe['target_ids_not_observed']) == set(missing_index['missing_scene_ids'])
assert probe['missing_target_scene_count'] == 76 and probe['staged_frame_count'] == 0
print('真实源前缀审计：', probe['compressed_bytes_read'], 'bytes；观测', probe['observed_scene_count'], '场景；完整来源SHA未通过（未完成遍历）。')
print('76缺失ID仍未核验补齐；候选PNG与已准备JPEG隔离，不自动生成Table1分数。')
metadata_path = Path('results/co3d_test_selection_summary_20261002.json')
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text())
    assert metadata['status'] == 'official_metadata_selection_reconstructed_not_rgb_ready'
    independent = json.loads(Path('results/co3d_test_selection_verified_20261002.json').read_text())
    assert independent['selection_report_sha256'] == metadata['full_selection_report_sha256']
    assert independent['nonempty_category_count'] == metadata['nonempty_selected_category_count']
    assert independent['selected_sequence_count'] == metadata['selected_sequence_count']
    assert independent['candidate_frame_count'] == metadata['candidate_frame_count']
    selection_path = Path('data/co3d_test_metadata/selected_seqs_test_reconstructed.json')
    if selection_path.exists():
        assert hashlib.sha256(selection_path.read_bytes()).hexdigest() == metadata['selected_manifest_sha256']
        selected = json.loads(selection_path.read_text())
        assert sum(bool(v) for v in selected.values()) == metadata['nonempty_selected_category_count']
        assert sum(map(len, selected.values())) == metadata['selected_sequence_count']
        assert sum(len(f) for s in selected.values() for f in s.values()) == metadata['candidate_frame_count']
        assert all(len(f) >= 10 and len(set(f)) == len(f) for s in selected.values() for f in s.values())
    assert not metadata['rgb_depth_mask_and_camera_npz_ready'] and not metadata['formal_pose_metrics_available']
    print('CO3D公开规则重建选择：', metadata['nonempty_selected_category_count'], '非空类别，', metadata['selected_sequence_count'], '序列，', metadata['candidate_frame_count'], '候选帧。')
    print('原作者选择等价性：', metadata['original_fast3r_scene_selection_equivalence'])
    print('公开默认规则得到51类，不等于论文41类；必须继续核验类别选择，不能直接当Table1全量。')
else:
    print('CO3D官方元数据准备pending；不将下载启动当作划分或RGB就绪。')
print('论文§4.2/Table1：数据前提推进，尚无新增正式RRA/RTA/mAA。')

真实源前缀审计： 8388608 bytes；观测 1 场景；完整来源SHA未通过（未完成遍历）。
76缺失ID仍未核验补齐；候选PNG与已准备JPEG隔离，不自动生成Table1分数。
CO3D公开规则重建选择： 51 非空类别， 2511 序列， 498757 候选帧。
原作者选择等价性： unverified; author processed manifest absent
公开默认规则得到51类，不等于论文41类；必须继续核验类别选择，不能直接当Table1全量。
论文§4.2/Table1：数据前提推进，尚无新增正式RRA/RTA/mAA。


### CO3D：从51类到有来源的41类候选（§4.2 / Table 1）

PoseDiffusion官方固定版本的 `TRAINING_CATEGORIES` 是41类，`default_test.yaml` 用 `category: seen`、10视角。这与论文“41类物体中的未见轨迹”相符，是类别层面的证据，不是Fast3R逐轨迹清单确认。排除的10类见下面真实输出；保留原51类规则的种子/类别索引/帧顺序，不重抽序列。

apple ZIP的HTTP Range目录探测只统计名称和标示大小，不证明图像CRC/解码或相机几何已通过。全41类空间审计在独立后台服务进行，本单元不下载数据也不启动GPU。没有新的正式RRA/RTA/mAA。

In [1]:
from pathlib import Path
import json, hashlib
protocol = json.loads(Path('results/co3d_seen41_protocol_20261002.json').read_text())
probe = json.loads(Path('results/co3d_apple_range_probe_20261002.json').read_text())
assert protocol['category_count'] == 41 and len(set(protocol['seen_categories'])) == 41
assert len(protocol['excluded_unseen_categories']) == 10
assert not (set(protocol['seen_categories']) & set(protocol['excluded_unseen_categories']))
assert sum(protocol['sequence_counts_by_category'].values()) == protocol['selected_sequence_count']
assert sum(protocol['frame_counts_by_category'].values()) == protocol['candidate_frame_count']
assert protocol['candidate_manifest_sha256'] == probe['candidate_manifest_sha256']
local = Path('data/co3d_test_metadata/selected_seqs_test_seen41_candidate.json')
if local.exists():
    assert hashlib.sha256(local.read_bytes()).hexdigest() == protocol['candidate_manifest_sha256']
    selected = json.loads(local.read_text())
    assert list(selected) == protocol['seen_categories']
    assert sum(map(len, selected.values())) == protocol['selected_sequence_count']
    assert sum(len(f) for s in selected.values() for f in s.values()) == protocol['candidate_frame_count']
for report in (protocol, probe):
    assert not report['formal_pose_metrics_available']
assert not protocol['original_fast3r_split_equivalence_verified']
assert not probe['member_crc_verified'] and not probe['all_41_category_inventory_complete']
assert all(v == probe['expected_candidate_frame_count'] for v in probe['matched_member_counts'].values())
print('固定PoseDiffusion revision：', protocol['posediffusion_revision'])
print('41类候选：', protocol['selected_sequence_count'], '轨迹；', protocol['candidate_frame_count'], '候选帧。')
print('排除的PoseDiffusion unseen类别：', ', '.join(protocol['excluded_unseen_categories']))
print('选择操作：只过滤原51类候选，保留各类原始seed+index与帧顺序。')
print('apple真实目录Range探测：', probe['index_transport_bytes'], '传输bytes；', probe['matched_member_counts'])
print('apple目录标示RGB/depth/mask合计：', probe['advertised_selected_uncompressed_bytes'], 'bytes（不是已下载量）。')
print('局限：类别映射为有证据推断，作者逐轨迹/100@采样等价性未验证；CRC、解码和相机几何未验证。')
print('全41类预算：后台审计进行中；本单元只分析已归档apple探测，不动态覆盖历史输出。')
print('论文§4.2/Table1：没有新增正式位姿成绩；整篇复现尚未完成。')

固定PoseDiffusion revision： b138198e2891a0f1a1c3435614b9490adb7fd4d6
41类候选： 2011 轨迹； 399204 候选帧。
排除的PoseDiffusion unseen类别： ball, book, couch, frisbee, hotdog, kite, remote, sandwich, skateboard, suitcase
选择操作：只过滤原51类候选，保留各类原始seed+index与帧顺序。
apple真实目录Range探测： 83941069 传输bytes； {'images': 9563, 'depths': 9563, 'masks': 9563}
apple目录标示RGB/depth/mask合计： 6073570411 bytes（不是已下载量）。
局限：类别映射为有证据推断，作者逐轨迹/100@采样等价性未验证；CRC、解码和相机几何未验证。
全41类预算：后台审计进行中；本单元只分析已归档apple探测，不动态覆盖历史输出。
论文§4.2/Table1：没有新增正式位姿成绩；整篇复现尚未完成。


### CO3D目录预算的失败诊断与安全恢复（§4.2 / Table 1数据前提）

banana官方ZIP有229525个成员，超过脚本初版20万保护阈值。只读诊断核实目录字节仍在32MiB范围内；有界提高成员上限，保留Range、ETag和路径检查。此前完成的apple/backpack经过旧版本指纹和统计核验导入v2，旧报告不覆盖。下面执行的是已归档恢复快照分析，不下载数据、不查询当前服务、不把恢复当作全量完成或模型成绩。

In [1]:
from pathlib import Path
import json, hashlib
r = json.loads(Path('results/co3d_storage_recovery_20261002.json').read_text())
assert r['exit_status'] == 1 and r['observed_zip_member_count'] > r['old_member_limit']
assert r['observed_zip_member_count'] < r['new_member_limit']
assert r['diagnostic_index_transport_bytes'] < r['index_read_byte_limit_unchanged'] == 32 * 1024**2
assert r['v1_journals_preserved'] and r['active_running_verified']
assert set(r['reused_completed_categories']) == {'apple', 'backpack'}
assert not r['full_41_budget_complete'] and not r['formal_pose_metrics_available']
for c, info in r['reused_completed_categories'].items():
    assert info['no_network_replay']
    for version in ('v1', 'v2'):
        directory = 'results/co3d_seen41_storage_progress' if version == 'v1' else r['new_progress_directory']
        p = Path(directory) / (c + '.json')
        if p.exists():
            assert hashlib.sha256(p.read_bytes()).hexdigest() == info[version + '_journal_sha256']
            entry = json.loads(p.read_text())
            assert all(v == info['expected_frame_count'] for v in entry['matched_member_counts'].values())
            assert entry['matched_uncompressed_bytes'] == info['matched_uncompressed_bytes']
print('失败原因：', r['failure'], '；实际', r['observed_zip_member_count'], '成员，原上限', r['old_member_limit'])
print('独立只读目录诊断：', r['diagnostic_index_transport_bytes'], 'bytes；32MiB字节上限保留。')
print('新成员上限：', r['new_member_limit'], '；不取消路径/Range/ETag检查，不改变样本。')
print('恢复时刻：', r['resume_time_local'], '；此时已确认active/running，非当前实时状态。')
for c, info in r['reused_completed_categories'].items():
    print(c, '复用', info['expected_frame_count'], '组目录项；旧文件SHA核验、未重放网络。')
print('新的最终预算路径：', r['new_final_report'])
print('论文§4.2/Table1：恢复的是数据预算流程，全41类预算/CRC/GT/正式位姿成绩仍未完成。')

失败原因： ZIP member count exceeds budget ；实际 229525 成员，原上限 200000
独立只读目录诊断： 29035659 bytes；32MiB字节上限保留。
新成员上限： 400000 ；不取消路径/Range/ETag检查，不改变样本。
恢复时刻： 2026-10-02T06:54:25+08:00 ；此时已确认active/running，非当前实时状态。
apple 复用 9563 组目录项；旧文件SHA核验、未重放网络。
backpack 复用 10055 组目录项；旧文件SHA核验、未重放网络。
新的最终预算路径： results/co3d_seen41_storage_budget_v2_20261002.json
论文§4.2/Table1：恢复的是数据预算流程，全41类预算/CRC/GT/正式位姿成绩仍未完成。


### 全部CO3D ZIP尾部预检与v3有界恢复（§4.2 / Table 1数据前提）

第二次失败不是GPU或模型错误：bowl_001.zip中央目录超过原32MiB字节预算。下面核验41类235包的真实尾部预检、预算界限与8类历史记录复用。这里的ETag/尾部SHA只能标识本次读取，不能替代完整ZIP SHA或图片CRC。恢复记录是历史快照，不是当前服务状态；完整目录预算与正式位姿成绩仍待完成。

In [1]:
from pathlib import Path
import json, hashlib
footer_path = Path('results/co3d_zip_footer_preflight_20261002.json')
f = json.loads(footer_path.read_text())
r = json.loads(Path('results/co3d_storage_footer_recovery_20261002.json').read_text())
assert hashlib.sha256(footer_path.read_bytes()).hexdigest() == r['footer_preflight_report_sha256']
archives = f['archives']
assert len(archives) == f['archive_count'] == r['footer_archive_count'] == 235
assert len({a['url'] for a in archives}) == 235
categories = {a['url'].rsplit('/', 1)[1].rsplit('_', 1)[0] for a in archives}
assert len(categories) == f['category_count'] == r['footer_category_count'] == 41
assert sum(a['footer_transport_bytes'] for a in archives) == f['footer_transport_bytes'] == r['footer_transport_bytes'] == 15405895
assert max(a['member_count'] for a in archives) == f['maximum_member_count'] == r['maximum_member_count'] == 300491
assert max(a['central_directory_bytes'] for a in archives) == f['maximum_central_directory_bytes'] == r['maximum_central_directory_bytes'] == 37498281
for a in archives:
    assert a['etag'] and len(a['footer_sha256']) == 64
    assert 0 < a['footer_transport_bytes'] <= f['footer_byte_limit_per_zip'] == 65557
    assert 0 < a['central_directory_bytes'] + 65557 + 128 < r['new_hard_index_byte_limit'] == 64 * 1024**2
    assert 0 < a['member_count'] < r['unchanged_member_count_limit'] == 400000
    assert 0 <= a['central_directory_offset'] < a['archive_bytes']
    assert a['central_directory_offset'] + a['central_directory_bytes'] < a['archive_bytes']
assert r['observed_bowl_directory_bytes'] > r['old_index_byte_limit'] == 32 * 1024**2
assert r['active_running_verified'] and r['prior_v1_v2_journals_preserved']
assert len(r['reused_v2_completed_categories']) == 8
checked = 0
for c, info in r['reused_v2_completed_categories'].items():
    assert info['no_central_directory_replay']
    for version, key in [('v2', 'prior_journal_sha256'), ('v3', 'v3_journal_sha256')]:
        path = Path('results/co3d_seen41_storage_' + version + '_progress') / (c + '.json')
        if path.exists():
            assert hashlib.sha256(path.read_bytes()).hexdigest() == info[key]
            saved = json.loads(path.read_text())
            assert all(v == info['expected_frame_count'] for v in saved['matched_member_counts'].values())
            checked += 1
bowl = Path(r['new_progress_directory']) / 'bowl.json'
if bowl.exists():
    assert hashlib.sha256(bowl.read_bytes()).hexdigest() == r['bowl_journal_sha256']
    entry = json.loads(bowl.read_text())
    assert entry['matched_member_counts'] == r['bowl_matched_member_counts']
    assert entry['matched_uncompressed_bytes'] == r['bowl_advertised_uncompressed_bytes']
assert all(v == 10098 for v in r['bowl_matched_member_counts'].values())
assert not any(r[k] for k in ['full_41_member_inventory_and_storage_budget_complete', 'member_crc_verified', 'full_archive_sha_verified', 'formal_pose_metrics_available'])
print('官方ZIP尾部预检：', len(categories), '类 /', len(archives), '包；真实传输', f['footer_transport_bytes'], 'bytes。')
print('最大目录', f['maximum_central_directory_bytes'], 'bytes；最大成员', f['maximum_member_count'], '。')
print('bowl失败：目录超过32MiB；新硬上限64MiB，每包预算=实测目录+65557+128，成员仍限制40万。')
print('核验复用8类历史记录；本机存在并通过SHA复核的v2/v3文件数：', checked)
print('bowl已通过原失败点：10098组图像/深度/掩码目录项，不代表图片CRC/解码完成。')
print('恢复时刻：', r['resume_time_local'], '；历史active/running快照，不是当前服务查询。')
print('新的最终预算路径：', r['new_final_report'])
print('论文§4.2/Table1：已完成全部尾部大小预检；全成员预算、RGB/GT准备与正式位姿成绩仍未完成。')

官方ZIP尾部预检： 41 类 / 235 包；真实传输 15405895 bytes。
最大目录 37498281 bytes；最大成员 300491 。
bowl失败：目录超过32MiB；新硬上限64MiB，每包预算=实测目录+65557+128，成员仍限制40万。
核验复用8类历史记录；本机存在并通过SHA复核的v2/v3文件数： 16
bowl已通过原失败点：10098组图像/深度/掩码目录项，不代表图片CRC/解码完成。
恢复时刻： 2026-10-02T07:29:59+08:00 ；历史active/running快照，不是当前服务查询。
新的最终预算路径： results/co3d_seen41_storage_budget_v3_20261002.json
论文§4.2/Table1：已完成全部尾部大小预检；全成员预算、RGB/GT准备与正式位姿成绩仍未完成。


### CO3D完整目录预算、名义采样与相机元数据（§4.2 / Table 1前提）

这一阶段不是模型指标：完整候选的目录汇总和相机元数据核验已结束，原始成员约297.20GB，尚不适合当前空间直接落盘。调用真实仓库采样方法的all-valid追踪揭示100@范围与重复视角，但没有读取RGB/深度，不能把名义请求当实际论文输入。相机按官方set-list连接编号与文件名，保留car的大坐标；数学残差通过不等于float32指标、图像裁剪或深度就绪。下面仅分析已保存真实JSON，不查询GPU或重新下载。

In [1]:
from pathlib import Path
import json, hashlib

def file_sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

bpath = Path('results/co3d_seen41_storage_budget_v3_20261002.json')
b = json.loads(bpath.read_text())
v = json.loads(Path('results/co3d_storage_budget_verified_20261002.json').read_text())
s = json.loads(Path('results/co3d_sampling_trace_20261002.json').read_text())
g = json.loads(Path('results/co3d_camera_metadata_audit_v2_20261002.json').read_text())
f = json.loads(Path('results/co3d_storage_feasibility_20261002.json').read_text())
assert file_sha(bpath) == v['budget_report_sha256'] == f['budget_report_sha256']
assert file_sha('scripts/verify_co3d_storage_budget.py') == v['verifier_sha256']
assert file_sha('scripts/audit_co3d_sampling_trace.py') == s['audit_script_sha256']
assert file_sha('scripts/audit_co3d_camera_metadata.py') == g['fingerprint']['auditor_sha256']
assert b['category_count'] == v['category_count'] == g['category_count'] == 41
assert b['selected_sequence_count'] == v['selected_sequence_count'] == g['selected_sequence_count'] == 2011
assert b['candidate_frame_count'] == v['candidate_frame_count'] == g['candidate_frame_count'] == 399204
assert v['data_archive_count'] == 235
assert b['fingerprint'] == v['fingerprint']
assert g['fingerprint']['candidate_manifest_sha256'] == s['candidate_manifest_sha256'] == v['fingerprint']['candidate_manifest_sha256']
assert [r['category'] for r in b['category_summaries']] == [r['category'] for r in v['categories']] == [r['category'] for r in g['categories']]
for kind in ('images', 'depths', 'masks'):
    assert b['matched_member_counts'][kind] == v['matched_member_counts'][kind] == 399204
    size = sum(r['matched_uncompressed_bytes'][kind] for r in b['category_summaries'])
    assert size == b['advertised_uncompressed_bytes_by_kind'][kind] == v['advertised_uncompressed_bytes_by_kind'][kind]
assert sum(b['advertised_uncompressed_bytes_by_kind'].values()) == b['advertised_total_uncompressed_bytes'] == v['advertised_total_uncompressed_bytes'] == 297197632174
assert b['index_transport_bytes'] == sum(r['index_transport_bytes'] for r in b['category_summaries']) == v['recorded_index_transport_bytes_including_reused_journals']
assert not f['full_raw_members_fit_with_reserve'] and not f['sparse_download_ready']
assert f['advertised_raw_total_bytes'] + f['minimum_reserve_bytes'] > f['filesystem_free_bytes_snapshot']
for row in v['categories']:
    local = Path('results/co3d_seen41_storage_v3_progress') / (row['category'] + '.json')
    if local.exists(): assert file_sha(local) == row['journal_sha256']

rows = s['nominal_100_rows']
stats = s['nominal_100_wrapper']
assert len(rows) == stats['nominal_sample_count'] == s['wrapper_length'] == 100
assert len({(r['category'], r['scene']) for r in rows}) == stats['nominal_distinct_trajectory_count'] == 99
assert len({r['category'] for r in rows}) == stats['nominal_distinct_category_count'] == 38
assert len({(r['category'], r['scene'], frame) for r in rows for frame in r['frame_numbers']}) == stats['nominal_distinct_frame_count'] == 918
assert sum(len(set(r['frame_numbers'])) < 10 for r in rows) == stats['samples_with_duplicate_views'] == 58
assert all(len(r['frame_numbers']) == 10 and r['distinct_frame_count'] == len(set(r['frame_numbers'])) for r in rows)
assert s['generated_combination_count'] * 2011 == s['base_dataset_length'] == 20110000
assert s['all_valid_stub'] and not s['realized_samples_verified'] and not s['paper_sampling_equivalence_verified']
all_path = Path('data/co3d_test_metadata/nominal_all_sequences_draws_seed42.json')
if all_path.exists():
    assert file_sha(all_path) == s['nominal_all_sequence_manifest_sha256']
    all_rows = json.loads(all_path.read_text())['rows']
    assert len(all_rows) == 2011
    assert len({(r['category'], r['scene'], frame) for r in all_rows for frame in r['frame_numbers']}) == s['nominal_all_sequence_adaptation']['nominal_distinct_frame_count'] == 18588

assert sum(r['candidate_frame_count'] for r in g['categories']) == 399204
assert sum(r['selected_sequence_count'] for r in g['categories']) == 2011
assert sum(r['filename_annotation_number_mismatch_count'] for r in g['categories']) == g['filename_annotation_number_mismatch_count'] == 38869
assert sum(r['source_camera_count_abs_translation_above_1e6'] for r in g['categories']) == g['source_camera_count_abs_translation_above_1e6'] == 12
assert max(r['maximum_abs_source_translation'] for r in g['categories']) == g['maximum_abs_source_translation']
for key, value in g['max_errors'].items(): assert max(r[key] for r in g['categories']) == value
assert g['max_errors']['w2c_c2w_max_component_scaled_error'] < 1e-12
for row in g['categories']:
    local = Path('results/co3d_camera_metadata_v2_progress') / (row['category'] + '.json')
    if local.exists(): assert json.loads(local.read_text()) == row
    assert row['unique_candidate_annotation_count'] == row['candidate_frame_count']
    assert row['intrinsics_format'] == 'ndc_isotropic' and row['depth_scale_adjustment'] == 1.0
    assert not row['rgb_depth_mask_decode_verified'] and not row['processed_crop_equivalence_verified']
assert not g['float32_pose_metric_stability_verified'] and not g['formal_pose_metrics_available']
assert not b['member_crc_verified'] and not b['full_archive_sha_verified'] and not b['formal_pose_metrics_available']
print('目录独立核验：41类 / 2011轨迹 / 399204组每种成员 / 235包；没有图像CRC或正式位姿分数。')
for kind, size in b['advertised_uncompressed_bytes_by_kind'].items(): print(kind, '原始目录标示', size, 'bytes =', round(size / 1024**3, 4), 'GiB')
print('原始总量：', round(b['advertised_total_uncompressed_bytes'] / 1024**3, 4), 'GiB；空间快照', round(f['filesystem_free_bytes_snapshot'] / 1024**3, 4), 'GiB，原始全量不适配。')
print('历史目录传输含复用记录：', b['index_transport_bytes'], 'bytes，不是本轮新增网络量。')
print('名义100@：100次 / 99轨迹 / 38类 / 918唯一帧；58次重复，假设所有深度有效。')
print('名义2011轨迹适配：18588唯一帧；不替换原100@，不冒充论文全量。')
print('全候选相机metadata核验：399204项；38869项文件名编号不同于annotation编号，按官方set-list连接。')
print('原始大坐标：12项平移>1e6，最大', g['maximum_abs_source_translation'], '；绝对逆残差', g['max_errors']['w2c_c2w_max_abs_error'], '。')
print('最大component-scaled逆残差：', g['max_errors']['w2c_c2w_max_component_scaled_error'], '；不是float32指标稳定性证明。')
print('论文§4.2/Table1：目录/相机元数据前提推进；按需图像/深度/crop与正式RRA/RTA/mAA尚未完成。')

目录独立核验：41类 / 2011轨迹 / 399204组每种成员 / 235包；没有图像CRC或正式位姿分数。
images 原始目录标示 36297726723 bytes = 33.8049 GiB
depths 原始目录标示 246352518781 bytes = 229.4337 GiB
masks 原始目录标示 14547386670 bytes = 13.5483 GiB
原始总量： 276.7869 GiB；空间快照 45.9665 GiB，原始全量不适配。
历史目录传输含复用记录： 2832716261 bytes，不是本轮新增网络量。
名义100@：100次 / 99轨迹 / 38类 / 918唯一帧；58次重复，假设所有深度有效。
名义2011轨迹适配：18588唯一帧；不替换原100@，不冒充论文全量。
全候选相机metadata核验：399204项；38869项文件名编号不同于annotation编号，按官方set-list连接。
原始大坐标：12项平移>1e6，最大 8.296149380025549e+16 ；绝对逆残差 16.0 。
最大component-scaled逆残差： 6.758143057927225e-16 ；不是float32指标稳定性证明。
论文§4.2/Table1：目录/相机元数据前提推进；按需图像/深度/crop与正式RRA/RTA/mAA尚未完成。


### 相对位姿的数值精度检查（§4.2 / Table 1数学前提）

相机位姿写成 $C_i=[R_i,t_i;0,1]$，相对位姿为 $C_i^{-1}C_j$，平移项涉及两个大数相减。前面的尺度逆残差很小，并不能保证单精度保留小基线。这里使用固定DUSt3R的float32 world-to-camera矩阵构造和求逆，再调用公开Fast3R相对位姿函数，与float64对应计算比较。覆盖全部候选的同轨迹相机对；不改变GT原点或尺度、不删除异常轨迹，不输入RGB或模型。方向比较对零向量没有定义，所以另计零基线而非静默丢弃。GT与自身比较仅是指标自检，绝不是模型预测精度；CPU结果不替代CUDA/TF32和真实预测数值检查。

In [1]:
import json, hashlib, math
from pathlib import Path
r = json.loads(Path('results/co3d_pose_precision_20261002.json').read_text())
file_sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
paths = {'auditor_sha256':'scripts/audit_co3d_pose_precision.py', 'metric_sha256':'fast3r/eval/cam_pose_metric.py', 'so3_sha256':'fast3r/utils/so3_utils.py', 'annotation_mapper_sha256':'scripts/audit_co3d_camera_metadata.py', 'protocol_report_sha256':'results/co3d_seen41_protocol_20261002.json', 'camera_report_sha256':'results/co3d_camera_metadata_audit_v2_20261002.json'}
for key, p in paths.items(): assert file_sha(p) == r['fingerprint'][key], p
assert r['category_count'] == 41 and r['selected_sequence_count'] == 2011
assert len(r['categories']) == 41 and len({c['category'] for c in r['categories']}) == 41
assert sum(c['selected_sequence_count'] for c in r['categories']) == 2011
a = r['aggregate']
assert a['candidate_frame_count'] == 399204
sum_keys = ['candidate_frame_count','pair_count','float64_zero_translation_pairs','float32_zero_translation_pairs','nonzero64_collapsed32_pairs','zero64_nonzero32_pairs','direction_defined_pairs','direction_disagreement_above_1deg_pairs','released_float32_identical_gt_translation_above_1deg_pairs','float64_identical_camera_center_pairs','baseline_above_diagnostic_scale_threshold_pairs','resolved_baseline_collapsed32_pairs','resolved_baseline_direction_above_1deg_pairs']
for k, v in a.items():
    assert math.isfinite(v)
    expected = sum(c[k] for c in r['categories']) if k in sum_keys else max(c[k] for c in r['categories'])
    assert v == expected, k
for k in ['predicted_poses_available','formal_pose_metrics_available','gpu_precision_verified','image_geometry_verified','paper_sampling_equivalence_verified']: assert r[k] is False
print('GT数值检查：41类 / 2011轨迹 / 399204帧；同轨迹相机对', a['pair_count'])
print('float64/float32零平移对：', a['float64_zero_translation_pairs'], '/', a['float32_zero_translation_pairs'])
print('计算中float64非零而float32变零：', a['nonzero64_collapsed32_pairs'], '；不自动等于物理基线被舍入。')
print('可定义方向的对数：', a['direction_defined_pairs'], '；方向差>1°：', a['direction_disagreement_above_1deg_pairs'], '；最大差', a['direction_disagreement_max_deg'], '°')
print('GT与自身比较：float32平移角>1°的对数', a['released_float32_identical_gt_translation_above_1deg_pairs'], '；最大角', a['released_float32_identical_gt_translation_max_deg'], '°')
print('相对平移最大分量绝对差：', a['relative_translation_max_component_abs_difference'], '；它不能单独作为方向误差大小的判断。')
print('直接float64中心相同的对数：', a['float64_identical_camera_center_pairs'])
print('基线>1e-6×max(1,中心范数)诊断分组：', a['baseline_above_diagnostic_scale_threshold_pairs'], '对；方向差>1°', a['resolved_baseline_direction_above_1deg_pairs'], '；最大', a['resolved_baseline_direction_max_deg'], '°；float32变零', a['resolved_baseline_collapsed32_pairs'])
print('风险轨迹总数：', r['flagged_sequence_count'], '；列表仅前20条：', [(s['category'],s['sequence']) for s in r['flagged_sequences']])
print('CPU诊断，不是预测RRA/RTA/mAA；GPU/真实输入、作者采样等价性尚未验证。')

GT数值检查：41类 / 2011轨迹 / 399204帧；同轨迹相机对 39624443
float64/float32零平移对： 1181 / 1375
计算中float64非零而float32变零： 194 ；不自动等于物理基线被舍入。
可定义方向的对数： 39623068 ；方向差>1°： 3301 ；最大差 89.9803869121981 °
GT与自身比较：float32平移角>1°的对数 1375 ；最大角 90.0 °
相对平移最大分量绝对差： 13369065328.0 ；它不能单独作为方向误差大小的判断。
直接float64中心相同的对数： 1181
基线>1e-6×max(1,中心范数)诊断分组： 39619868 对；方向差>1° 340 ；最大 5.984502155375689 °；float32变零 0
风险轨迹总数： 201 ；列表仅前20条： [('car', '411_55952_107659'), ('toytrain', '624_103166_207224'), ('stopsign', '435_61888_121383'), ('orange', '508_73226_142557'), ('orange', '501_72014_140081'), ('handbag', '397_50090_98524'), ('toyplane', '386_46204_92245'), ('apple', '506_72986_141860'), ('toytrain', '618_100442_200991'), ('chair', '373_41549_82956'), ('toaster', '396_49593_97742'), ('mouse', '581_86235_170995'), ('microwave', '618_100748_201835'), ('teddybear', '508_73233_142600'), ('banana', '438_62674_124082'), ('handbag', '404_53452_104225'), ('toilet', '408_55424_106535'), ('toilet', '557_80271_157845'), ('baseballbat', '2

### 十帧真实CO3D预处理接线（§4.2 / Table 1数据前提）

这是apple一条轨迹前十个候选帧的明确小探测，不是100@采样或全量benchmark。原始PNG深度的uint16是float16位模式，需要reinterpret后转float32，不能直接当整数深度归一化。参考预处理保存 $q=\lfloor65535d/d_{max}\rfloor$，加载器再恢复 $d'=q d_{max}/65535$，所以量化精度受该帧最大深度影响。裁剪先移除主点偏移，缩放K还包含OpenCV/Colmap的半像素约定：$c'_x=s(c_x+0.5)-o_x-0.5$，而非仅将主点乘s。

本轮有界读取官方ZIP目录和30个RGB/depth/mask成员，验证精确Range/ETag、成员CRC、解码，再将适配输出与固定DUSt3R原prepare_sequences在相同运行库上的输出比较：JPEG/PNG字节一致，NPZ数组一致（不比较ZIP时间戳）。成员CRC不代表整个ZIP SHA。独立复核调用公开_load_view_data，仍传入原202帧候选池，只读取已准备的十帧；不将稀疏目录改成采样清单。接近方形的图像可按固定rng产生两种输出朝向，保留公开loader行为，不拉伸图像。以下只分析真实报告，没有模型推理分数。

In [1]:
import json, hashlib
from pathlib import Path
file_sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
r = json.loads(Path('results/co3d_preprocess_probe_20261002.json').read_text())
v = json.loads(Path('results/co3d_preprocess_probe_verified_20261002.json').read_text())
assert v['probe_report_sha256'] == file_sha('results/co3d_preprocess_probe_20261002.json')
for k, p in {'probe_sha256':'scripts/probe_co3d_preprocess.py', 'cache_sha256':'scripts/co3d_range_cache.py', 'local_cropping_sha256':'fast3r/dust3r/datasets/utils/cropping.py'}.items(): assert r['fingerprint'][k] == file_sha(p)
assert v['verifier_sha256'] == file_sha('scripts/verify_co3d_preprocess_probe.py')
assert v['loader_sha256'] == file_sha('fast3r/dust3r/datasets/co3d_multiview.py')
assert v['base_loader_sha256'] == file_sha('fast3r/dust3r/datasets/base/base_stereo_view_dataset.py')
assert r['frame_count'] == v['prepared_probe_frame_count'] == len(r['records']) == len(v['rows']) == 10
assert r['frame_numbers'] == [x['frame_number'] for x in v['rows']] == [1,2,3,4,5,11,12,13,14,15]
assert r['candidate_manifest_sha256'] == v['candidate_manifest_sha256']
assert v['original_candidate_pool_length'] == 202 and r['all_candidate_frame_pool_unchanged']
assert all(len(x['raw_members']) == 3 and x['reference_image_depth_mask_bytes_equal'] and x['reference_npz_arrays_equal'] for x in r['records'])
assert all(m['member_crc_verified'] and not m['full_archive_sha_verified'] for x in r['records'] for m in x['raw_members'])
assert v['all_loader_probe_views_valid'] and all(x['loader_depth_positive_fraction'] > 0 for x in v['rows'])
for k in ['source_zip_full_sha_verified','formal_pose_metrics_available','full_candidate_data_ready','paper_selection_and_sampling_equivalence_verified']: assert r[k] is False
for k in ['actual_sampling_and_retries_verified','formal_pose_metrics_available','full_candidate_data_ready']: assert v[k] is False
print('真实接线：', r['category'], r['sequence'], '；十帧编号', r['frame_numbers'])
print('30个原始成员CRC/解码通过；保存原始字节', sum(m['bytes'] for x in r['records'] for m in x['raw_members']))
print('首次成功探测Range传输：', r['network_bytes_first_successful_probe'], 'bytes，含apple完整候选中央目录；不是完整ZIP下载。')
print('十帧与原DUSt3R输出：RGB/depth/mask逐文件字节相同，K/float32 pose/maximum_depth数组相同。')
print('原始/processed第一帧(H,W)：', r['records'][0]['raw_size_hw'], '/', r['records'][0]['processed_size_hw'])
print('公开loader原候选池：202帧，未替换成10帧稀疏清单；实际读取十帧均有效。')
print('公开loader输出(W,H)朝向：', sorted({tuple(x['image_size_wh']) for x in v['rows']}))
print('mask后正深度像素比例范围：', min(x['loader_depth_positive_fraction'] for x in v['rows']), max(x['loader_depth_positive_fraction'] for x in v['rows']))
print('§4.2/Table1前提推进；没有执行100@采样/补采重试/模型预测或全量RRA/RTA/mAA。')

真实接线： apple 608_95658_192033 ；十帧编号 [1, 2, 3, 4, 5, 11, 12, 13, 14, 15]
30个原始成员CRC/解码通过；保存原始字节 7738425
首次成功探测Range传输： 91655026 bytes，含apple完整候选中央目录；不是完整ZIP下载。
十帧与原DUSt3R输出：RGB/depth/mask逐文件字节相同，K/float32 pose/maximum_depth数组相同。
原始/processed第一帧(H,W)： [1011, 1036] / [499, 512]
公开loader原候选池：202帧，未替换成10帧稀疏清单；实际读取十帧均有效。
公开loader输出(W,H)朝向： [(384, 512), (512, 384)]
mask后正深度像素比例范围： 0.052225748697916664 0.056859334309895836
§4.2/Table1前提推进；没有执行100@采样/补采重试/模型预测或全量RRA/RTA/mAA。


### 完整候选池的一个真实映射样本（§4.2 / Table 1前提）

100@是固定长度包装器，不表示已经评测100个轨迹。本次只执行epoch0第0项，映射至base index12303633。原41类/2011轨迹/399204候选帧不变，按需缓存不是新的稀疏split。公开采样的jitter/clamping会重复视图，10张输入实际只有9张不同图片，不能擅自去重。

严格adapter只移除文件读取异常被吞掉的catch-all：下载/CRC/相机失败直抛，裁剪后真正零有效深度仍按原规则失效、补采或重试场景。后者通过合成测试，本次真实样本全部有效，没有发生补采或换场景。base最终还用深度与相机产生 $X_c=d K^{-1}[u,v,1]^T$、$X_w=R X_c+t$，所以需要逐项比较3D点、有效掩码、归一化tensor和rng，不仅看图片尺寸。重复相机的平移基线为零，方向不可定义；之后的45个相机对需保留这一事实，不能删除它让指标变好。

本次全部base输出与公开loader精确相同，独立执行固定DUSt3R预处理后9帧的JPEG/PNG字节、NPZ数组也一致。Python补采seed按请求固定，属于可复核的单样本适配，不证明作者draws或连续100@共享状态相同。第一次结果JSON因NumPy idx序列化失败已保留并用v2安全恢复。没有模型推理或正式位姿分数。

In [1]:
import json, hashlib
from pathlib import Path
file_sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
r = json.loads(Path('results/co3d_lazy_draw0_v2_20261002.json').read_text())
v = json.loads(Path('results/co3d_lazy_draw0_verified_20261002.json').read_text())
e = json.loads(Path('results/co3d_lazy_serialization_recovery_20261002.json').read_text())
assert v['probe_report_sha256'] == e['successful_report_sha256'] == file_sha('results/co3d_lazy_draw0_v2_20261002.json')
assert e['independent_reference_report_sha256'] == file_sha('results/co3d_lazy_draw0_verified_20261002.json')
assert r['fingerprint']['preparer'] == v['preparer_fingerprint']
assert r['fingerprint']['preparer']['lazy_code_sha256'] == file_sha('scripts/co3d_lazy_dataset.py')
assert r['fingerprint']['probe_sha256'] == file_sha('scripts/probe_co3d_lazy_sample.py')
assert v['verifier_sha256'] == file_sha('scripts/verify_co3d_lazy_sample.py')
for key, path in {'released_loader_sha256':'fast3r/dust3r/datasets/co3d_multiview.py', 'base_loader_sha256':'fast3r/dust3r/datasets/base/base_stereo_view_dataset.py', 'wrapper_sha256':'fast3r/dust3r/datasets/base/easy_dataset.py'}.items(): assert r['fingerprint'][key] == file_sha(path)
assert [r['scope'][k] for k in ['candidate_categories','candidate_trajectories','candidate_frames']] == [41,2011,399204]
assert r['scope']['executed_wrapper_indices'] == [0] and r['scope']['base_index'] == 12303633
assert len(r['returned_views']) == len(r['actual_load_trace']) == 10
assert [x['frame_number'] for x in r['actual_load_trace']] == [144,42,41,151,20,28,52,144,10,1]
assert all(x['status'] == 'valid' for x in r['actual_load_trace'])
assert len(r['pool_attempts']) == 1 and r['pool_attempts'][0]['candidate_pool_length'] == 133
assert len(r['prepared_frames']) == v['checked_frame_count'] == len(v['frames']) == 9
assert all(x['image_depth_mask_bytes_equal'] and x['npz_arrays_equal'] for x in v['frames'])
assert r['original_loader_base_outputs_exactly_equal'] and not r['invalid_scene_tracker']
assert all(not any(indices for indices in by_res.values()) for by_res in r['invalidated_pool_indices'].values())
for k in ['formal_pose_metrics_available','full_candidate_data_ready','all_100_draws_executed','source_zip_full_sha_verified','paper_selection_and_sampling_equivalence_verified']: assert r[k] is False
assert not v['formal_pose_metrics_available'] and not e['data_deleted_or_overwritten']
assert r['network_bytes_first_successful_draw'] == v['network_bytes'] == e['successful_retry_network_bytes'] == 0
print('真实执行范围：100@中的第0项，base index', r['scope']['base_index'], '；不是全部100次。')
print('完整原候选池：41类 / 2011轨迹 / 399204帧；本轨迹原133帧未变。')
print('输入帧顺序：', [x['frame_number'] for x in r['actual_load_trace']], '；10views / 9unique，保留重复。')
print('实际加载10次均有效，pool尝试1次；没有真实补采/scene retry。')
print('完整base输出与原loader精确一致；9unique帧与固定原预处理JPEG/PNG bytes、NPZ arrays一致。')
print('保存raw原字节：', sum(m['bytes'] for f in r['prepared_frames'] for m in f['raw_members']))
print('GT有效深度覆盖范围：', min(x['gt_valid_fraction'] for x in r['returned_views']), max(x['gt_valid_fraction'] for x in r['returned_views']), '；不是预测准确率。')
print('首次JSON序列化失败已保留未完整文件，修复后v2缓存重试网络0bytes；不是首轮下载总流量。')
print('138项离线测试通过；没有模型/RRA/RTA/mAA；作者采样与连续100@共享状态等价仍未确认。')

真实执行范围：100@中的第0项，base index 12303633 ；不是全部100次。
完整原候选池：41类 / 2011轨迹 / 399204帧；本轨迹原133帧未变。
输入帧顺序： [144, 42, 41, 151, 20, 28, 52, 144, 10, 1] ；10views / 9unique，保留重复。
实际加载10次均有效，pool尝试1次；没有真实补采/scene retry。
完整base输出与原loader精确一致；9unique帧与固定原预处理JPEG/PNG bytes、NPZ arrays一致。
保存raw原字节： 4541640
GT有效深度覆盖范围： 0.1325225830078125 0.4758809407552083 ；不是预测准确率。
首次JSON序列化失败已保留未完整文件，修复后v2缓存重试网络0bytes；不是首轮下载总流量。
138项离线测试通过；没有模型/RRA/RTA/mAA；作者采样与连续100@共享状态等价仍未确认。


### CO3D公开HF位姿入口：真实输入预检查（§4.2 / Table 1前提）

本节仅复核已归档第0个mapped draw，不重做下载或旧GPU实验。网络只接收归一化RGB和原true_shape；GT K参与公开crop预处理，但GT深度、点图、位姿和K不输入网络或focal/PnP。公开基类把portrait图转为landscape tensor，所以HF head的landscape_only=False输出portrait，发布版评测按原true_shape转置一次回到loader landscape，再估计focal和PnP；不改发布算法。

位姿链路为多视图Transformer的global点图 → 首视图置信度过滤的focal → 每视图2D/3D对应PnP → 45对相机的公开旋转/平移方向角与mAA。PnP求解像素与投影3D点的重投影误差；GT只在后续评分使用。重复视图产生的零基线pair仍保留，GT-self角度仅诊断数值行为，不是预测准确率。单样本不能替代连续100@共享状态，更不能代入Table 1全量成绩。

In [1]:
import json, hashlib
from pathlib import Path
r = json.loads(Path('results/co3d_pose_draw0_input_preflight_v2_20261002.json').read_text())
p = r['protocol']
file_sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert r['protocol_sha256'] == hashlib.sha256(json.dumps(p, sort_keys=True).encode()).hexdigest()
assert p['runner_sha256'] == file_sha('scripts/fast3r_hf_co3d_pose_smoke.py')
assert p['draw_report_sha256'] == file_sha('results/co3d_lazy_draw0_v2_20261002.json')
assert p['reference_report_sha256'] == file_sha('results/co3d_lazy_draw0_verified_20261002.json')
assert p['checkpoint_weight_sha256'] == '1357d42e74539ba03510ea1986c7464a7facf17157779e2b7ebd6284e79e0427'
assert p['checkpoint_config_sha256'] == '88edc53ba652ad835b86af27d980a46808d722b4d8f9d0d4360d05c789ada92b'
assert p['network_input_keys'] == ['img','true_shape'] and not p['gt_used_by_network_or_PnP']
assert r['input_tensor_shapes'] == [[1,3,384,512]]*10
assert r['original_true_shapes'] == [[[512,384]]]*10 and r['gt_shape'] == [10,4,4]
assert p['pair_count'] == 45 and p['inference_seed'] == 12303675
assert not r['model_loaded'] and not r['formal_pose_metrics_available'] and r['network_bytes'] == 0
assert p['paper_checkpoint_and_draw_mapping_verified'] is False
print('已核验一个真实mapped draw，10张输入、9个独立帧，原完整候选池未改变。')
print('网络输入10 × [1,3,384,512]；原portrait true_shape [512,384]；GT [10,4,4]只供评分。')
print('发布版方向校正一次后global points → focal → PnP → 45pairs；重复/identity fallback不静默删除。')
print('公开权重与配置SHA一致；seed12303675、16-mixed、DPT chunk2。')
print('本单元仅CPU输入/代码校验，网络0bytes、未加载模型；144项离线测试通过。')
print('首次GPU方向断言错误已诊断：portrait输出经发布版校正成landscape；v1保存，修复后单样本重试。')
print('未执行连续100@共享状态；作者draw/checkpoint对应未确认，不是Table1全量成绩。')

已核验一个真实mapped draw，10张输入、9个独立帧，原完整候选池未改变。
网络输入10 × [1,3,384,512]；原portrait true_shape [512,384]；GT [10,4,4]只供评分。
发布版方向校正一次后global points → focal → PnP → 45pairs；重复/identity fallback不静默删除。
公开权重与配置SHA一致；seed12303675、16-mixed、DPT chunk2。
本单元仅CPU输入/代码校验，网络0bytes、未加载模型；144项离线测试通过。
首次GPU方向断言错误已诊断：portrait输出经发布版校正成landscape；v1保存，修复后单样本重试。
未执行连续100@共享状态；作者draw/checkpoint对应未确认，不是Table1全量成绩。


### 一个真实CO3D样本的位姿输出与失败恢复（§4.2/Table 1接线，不是全量）

服务11:05:17正常结束。首次失败只来自新入口错误的形状断言，发布版head/correction均未改；旧预检查和失败代码、日志保留，修复后另写v2预检查再重试。10张RGB同次前向，经global点图/focal/PnP得到10个c2w，保留重复视图和45个相机对。以下重算保存位姿的公开指标，不重跑模型。

该样本PnP未失败不代表准确：RRA@30和mAA@30均为0，RTA@30仅约13.33%，旋转误差50.13°–171.65°。这是当前声明适配的真实低分，不是Table 1或整篇复现完成；不删坏样本，也不把合成测试通过称作论文指标通过。首视图估计focal约27.98px；GT-self的重复零基线对产生90°方向角，是数值诊断，不是网络误差。下一步需固定同输入/同前向的方向与focal单变量诊断，不能未经证据把差距归因于某个因素，更不能悄悄换公式提高成绩。

In [1]:
import json, hashlib, sys
from pathlib import Path
import numpy as np, torch
sys.path.insert(0, str(Path('scripts').resolve()))
from fast3r_hf_re10k_pose_eval import pose_metrics
file_sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
r = json.loads(Path('results/co3d_pose_draw0_seed42_20261002.json').read_text())
v = json.loads(Path('results/co3d_pose_smoke_recovery_verified_20261002.json').read_text())
f = json.loads(Path('results/co3d_pose_draw0_input_preflight_v2_20261002.json').read_text())
assert v['result_sha256'] == file_sha(v['result_path'])
assert v['new_preflight_sha256'] == file_sha(v['new_preflight_path'])
assert v['initial_preflight_sha256'] == file_sha(v['initial_preflight_path'])
assert v['successful_runner_sha256'] == r['protocol']['runner_sha256'] == file_sha('scripts/fast3r_hf_co3d_pose_smoke.py')
assert r['protocol'] == f['protocol'] and r['protocol_sha256'] == f['protocol_sha256'] == v['protocol_sha256']
assert v['service_properties'] == {'MainPID':'0','ExecMainStatus':'0','ActiveState':'inactive','SubState':'dead'}
assert r['model_inference_completed'] and not r['formal_pose_metrics_available'] and not r['all_100_draws_executed']
assert not v['existing_data_or_results_deleted_or_overwritten'] and not v['other_GPU_jobs_terminated']
assert len(r['inputs']['frames']) == 10 and r['pair_count'] == 45
assert r['repeated_camera_view_pairs'] == [[0,7]] and r['pnp_failed_view_indices'] == []
assert r['raw_prediction_shapes'] == [[1,512,384]]*10 and r['corrected_prediction_shapes'] == [[1,384,512]]*10
for field in ['predicted_c2w','gt_c2w']:
    a = np.asarray(r[field]); assert a.shape == (10,4,4) and np.isfinite(a).all()
    assert np.allclose(a[:,3,:], [0,0,0,1], atol=1e-5)
    assert np.allclose(np.linalg.det(a[:,:3,:3]), 1, atol=1e-4)
torch.set_num_threads(2)
metrics, errors = pose_metrics(r['predicted_c2w'], r['gt_c2w'])
assert all(abs(metrics[k]-r['metrics'][k]) < 1e-7 for k in metrics)
assert all(len(errors[k]) == 45 and np.allclose(errors[k],r['relative_errors'][k],atol=1e-7,rtol=0) for k in errors)
self_metrics, self_errors = pose_metrics(r['gt_c2w'], r['gt_c2w'])
assert all(abs(self_metrics[k]-r['gt_self_metric_diagnostic']['metrics'][k]) < 1e-7 for k in self_metrics)
print('一个真实mapped draw完成且重新验证：10views / 9unique / 45pairs；不是Table1全量。')
print('PnP失败0/10；发布版portrait → landscape一次；预检查v1及失败证据保留，v2安全恢复。')
for name in ['RRA_at_5','RRA_at_15','RRA_at_30','RTA_at_5','RTA_at_15','RTA_at_30','mAA_30']:
    print(name, f'{100*metrics[name]:.4f}%')
print('旋转误差范围(deg)：', min(errors['rotation_deg']), max(errors['rotation_deg']))
print('预测首视图focal(px)：', r['estimated_focal'], '；不是GT focal。')
print('GT-self RTA@30诊断：', f'{100*self_metrics["RTA_at_30"]:.4f}%', '；重复零基线保留，不是预测指标。')
print('真实前向秒数/峰值allocated bytes：', r['runtime']['forward_seconds'], r['runtime']['cuda_max_memory_allocated_bytes'])
print('144项离线测试通过；模型指标很差需诊断；连续100@、作者划分/权重匹配与训练仍未完成。')

一个真实mapped draw完成且重新验证：10views / 9unique / 45pairs；不是Table1全量。
PnP失败0/10；发布版portrait → landscape一次；预检查v1及失败证据保留，v2安全恢复。
RRA_at_5 0.0000%
RRA_at_15 0.0000%
RRA_at_30 0.0000%
RTA_at_5 0.0000%
RTA_at_15 8.8889%
RTA_at_30 13.3333%
mAA_30 0.0000%
旋转误差范围(deg)： 50.126380920410156 171.6511688232422
预测首视图focal(px)： 27.98291778564453 ；不是GT focal。
GT-self RTA@30诊断： 97.7778% ；重复零基线保留，不是预测指标。
真实前向秒数/峰值allocated bytes： 1.0859267720006756 4970486784
144项离线测试通过；模型指标很差需诊断；连续100@、作者划分/权重匹配与训练仍未完成。


### 同一次前向的方向诊断（§4.2/Table 1前提，不是新正式协议）

只改变global点图/置信度的空间朝向：发布版portrait→landscape与未转置raw portrait，网络仅前向一次，预测tensor哈希校验未被分支修改；两分支保持同一GT/seed/公开focal函数/conf>1 PnP/45pairs/metric。方向改变后focal由同一函数重估，像素轴也改变，因此这不是固定focal的独立实验。旧发布流程低分报告保留，不挑较高分替代它。

原始方向的focal变成553.66px，但RRA@30仍仅1/45≈2.22%，mAA@30仍0。此单样本证据不支持只取消转置就解决低分，也不证明所有场景同因；下一步需更多受控几何接线/预测坐标与focal检查，不静默修改正式指标。

In [1]:
import json, hashlib, sys
from pathlib import Path
import numpy as np
sys.path.insert(0,str(Path('scripts').resolve()))
from fast3r_hf_re10k_pose_eval import pose_metrics
r=json.loads(Path('results/co3d_orientation_diagnostic_20261002.json').read_text())
p=r['protocol']; sha=lambda path:hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert p['diagnostic_runner_sha256']==sha('scripts/diagnose_co3d_pose_orientation.py')
assert p['historical_baseline_sha256']==sha('results/co3d_pose_draw0_seed42_20261002.json')
assert p['runner_sha256']==sha('scripts/fast3r_hf_co3d_pose_smoke.py')
assert r['protocol_sha256']==hashlib.sha256(json.dumps(p,sort_keys=True).encode()).hexdigest()
assert r['paired_same_forward'] and r['forward_count']==1 and len(r['global_tensor_sha256'])==10
assert r['historical_baseline_metrics_exact_match']
assert all(x==0 for x in r['historical_baseline_metric_deltas'].values())
assert not r['formal_pose_metrics_available'] and not r['full_paper_completed'] and not p['GT_for_focal_or_PnP']
assert set(r['branches'])=={'released_landscape_correction','raw_portrait_no_correction'}
for name,row in r['branches'].items():
    assert row['pair_count']==45 and len(row['inputs']['frames'])==10
    m,e=pose_metrics(row['predicted_c2w'],row['gt_c2w'])
    assert all(abs(m[k]-row['metrics'][k])<1e-7 for k in m)
    assert all(len(e[k])==45 and np.allclose(e[k],row['relative_errors'][k],atol=1e-7,rtol=0) for k in e)
    print(name)
    print('focal(px):',row['estimated_focal'],'PnP failures:',row['pnp_failed_view_indices'])
    print('RRA@30 / RTA@30 / mAA@30 (%):',*[round(m[k]*100,4) for k in ['RRA_at_30','RTA_at_30','mAA_30']])
print('同一次前向，发布版分支与旧报告指标逐项完全相等；原低分未覆盖。')
print('raw方向RRA@30仅1/45，mAA仍0：取消转置不足以解决该样本低分。')
print('focal随方向重估，非固定focal因子；仅单样本诊断，不是Table1成绩或全场景根因证明。')
print('145项离线测试通过；原21个Notebook分析单元不重复执行。')

released_landscape_correction
focal(px): 27.98291778564453 PnP failures: []
RRA@30 / RTA@30 / mAA@30 (%): 0.0 13.3333 0.0
raw_portrait_no_correction
focal(px): 553.660400390625 PnP failures: []
RRA@30 / RTA@30 / mAA@30 (%): 2.2222 11.1111 0.0
同一次前向，发布版分支与旧报告指标逐项完全相等；原低分未覆盖。
raw方向RRA@30仅1/45，mAA仍0：取消转置不足以解决该样本低分。
focal随方向重估，非固定focal因子；仅单样本诊断，不是Table1成绩或全场景根因证明。
145项离线测试通过；原21个Notebook分析单元不重复执行。


### RE10K补缺来源完整扫描结束（§4.2 / Table 1数据前提）

固定公开tar.gz的全部205,763,619,478压缩字节已流式读取，扫描器完成gzip尾部/CRC读取且SHA256与已审计LFS值一致。未落盘整个205GB包。独立核验只复核保存报告、规定1832-ID集合及76缺失GT的原SHA/记录数，不重新下载并重算全包SHA。

在已核对的tar路径/PNG命名协议下识别4137个source场景，76缺失ID均未观测，0帧暂存、0场景补齐。没有全量审计被忽略的tar路径，不把此结果夸大成所有来源永久不可用。现有1756/1832准备仍未覆盖76个测试ID；不能用交集当正式Table 1，也没有新图片几何或模型位姿成绩。

In [1]:
import json, hashlib
from pathlib import Path
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
r=json.loads(Path('results/re10k_missing_candidates_full_source.json').read_text())
v=json.loads(Path('results/re10k_missing_full_source_verified_20261002.json').read_text())
i=json.loads(Path('results/re10k_rgb_index_audit.json').read_text())
assert v['source_report_sha256']==sha('results/re10k_missing_candidates_full_source.json')
assert v['verifier_sha256']==sha('scripts/verify_re10k_missing_full_source.py')
assert v['scanner_sha256']==sha('scripts/stream_re10k_missing_candidates.py')
assert v['index_sha256']==sha('results/re10k_rgb_index_audit.json')
assert r['full_archive_sha_verified'] and r['compressed_bytes_read']==r['source_expected_bytes']==205763619478
assert r['compressed_sha256']==r['source_expected_sha256']==v['source_stream_sha256']=='f6055cd8ea1ccce642482ca623f98c21af1c449210760d23d78a43b09e546523'
assert sorted(i['missing_scene_ids'])==r['target_ids_not_observed']==v['missing_scene_ids']
assert len(v['missing_scene_ids'])==76 and v['missing_GT_camera_record_count']==14163
assert r['target_ids_observed']==[] and r['complete_candidate_timestamp_scene_ids']==[] and r['frames']=={}
assert r['staged_frame_count']==v['staged_frame_count']==v['recovered_scene_count']==0
assert v['recognized_source_scene_count']==r['observed_scene_count']==4137
assert v['prescribed_scene_count']==1832 and v['previously_verified_prepared_scene_count']==1756
assert not v['archive_independently_rehashed_this_verification'] and v['network_bytes_this_verification']==0
assert not v['formal_pose_metrics_available'] and not v['full_prescribed_split_ready'] and not v['new_image_geometry_verified']
print('完整源扫描压缩字节：',r['compressed_bytes_read'],'；scanner全流SHA与已审计LFS相同。')
print('该路径协议识别4137场景；76缺失ID未观测，暂存0帧、补齐0场景。')
print('独立核验76份官方GT的SHA与14163条相机记录；保存报告集合与规定split一致。')
print('独立核验网络0bytes；未再次下载/独立重hash205GB，不混淆两种证据。')
print('现有1756/1832覆盖未改善；没有新图片几何或Table1位姿成绩。')
print('仅该来源/路径协议无补缺，不证明所有视频永久不可用；148项离线测试通过。')

完整源扫描压缩字节： 205763619478 ；scanner全流SHA与已审计LFS相同。
该路径协议识别4137场景；76缺失ID未观测，暂存0帧、补齐0场景。
独立核验76份官方GT的SHA与14163条相机记录；保存报告集合与规定split一致。
独立核验网络0bytes；未再次下载/独立重hash205GB，不混淆两种证据。
现有1756/1832覆盖未改善；没有新图片几何或Table1位姿成绩。
仅该来源/路径协议无补缺，不证明所有视频永久不可用；148项离线测试通过。


In [1]:
import json, hashlib
from pathlib import Path
import sys
sys.path.insert(0, 'scripts')
from diagnose_co3d_pose_focal import validate, BRANCHES
from fast3r_hf_co3d_pose_smoke import load_archived_draw, protocol, signature, sha
import numpy as np
r=json.loads(Path('results/co3d_focal_diagnostic_20261002.json').read_text())
a,views=load_archived_draw()
gt=np.stack([v['camera_pose'] for v in views]).astype(np.float32)
p=protocol(Path('checkpoints/Fast3R_ViT_Large_512'),42+int(a['scope']['base_index']),2)
for k in ('diagnostic_runner_sha256','historical_baseline_sha256','benchmark_scope','changed_variable','branches','GT_for_focal_or_PnP'): p[k]=r['protocol'][k]
assert p['diagnostic_runner_sha256']==sha('scripts/diagnose_co3d_pose_focal.py')
assert p['historical_baseline_sha256']==sha('results/co3d_pose_draw0_seed42_20261002.json')
validate(r,p,{'frames':a['returned_views']},gt)
assert r['historical_baseline_metrics_exact_match']
old=json.loads(Path('results/co3d_orientation_diagnostic_20261002.json').read_text())
assert r['branches']['raw_global_focal']['metrics']==old['branches']['raw_portrait_no_correction']['metrics']
assert all(row['metrics']['mAA_30']==0 for row in r['branches'].values())
print('§4.2/Table 1接线诊断；一次前向，四组45pairs，保留重复帧和旧低分。')
for name,row in r['branches'].items():
    m=row['metrics']
    print(f"{name}: focal={row['estimated_focal']:.4f}px, RRA30={m['RRA_at_30']*100:.4f}%, RTA30={m['RTA_at_30']*100:.4f}%, mAA30={m['mAA_30']*100:.4f}%, PnP失败={len(row['pnp_failed_view_indices'])}/10")
print('同方向下保持global PnP点/掩码/seed相同；只改变预测焦距来源，GT不用于focal或PnP。')
print('local焦距未经过local→global对齐，不是发布版aligned-local评测，也不是Table 5消融。')
print('替换焦距来源未解决此样本旋转低分；不推广到全部场景，不以调参挑最好值。')
print('151项离线测试通过；尚未完成连续100@共享采样状态、作者划分和权重对应。')

§4.2/Table 1接线诊断；一次前向，四组45pairs，保留重复帧和旧低分。
released_global_focal: focal=27.9829px, RRA30=0.0000%, RTA30=13.3333%, mAA30=0.0000%, PnP失败=0/10
released_local_focal: focal=14.2643px, RRA30=0.0000%, RTA30=15.5556%, mAA30=0.0000%, PnP失败=0/10
raw_global_focal: focal=553.6604px, RRA30=2.2222%, RTA30=11.1111%, mAA30=0.0000%, PnP失败=0/10
raw_local_focal: focal=488.1309px, RRA30=2.2222%, RTA30=8.8889%, mAA30=0.0000%, PnP失败=0/10
同方向下保持global PnP点/掩码/seed相同；只改变预测焦距来源，GT不用于focal或PnP。
local焦距未经过local→global对齐，不是发布版aligned-local评测，也不是Table 5消融。
替换焦距来源未解决此样本旋转低分；不推广到全部场景，不以调参挑最好值。
151项离线测试通过；尚未完成连续100@共享采样状态、作者划分和权重对应。


In [1]:
import json, sys
from pathlib import Path
import numpy as np
sys.path.insert(0,'scripts')
from diagnose_co3d_pose_aligned_focal import validate
from fast3r_hf_co3d_pose_smoke import load_archived_draw, protocol, sha
from verify_co3d_sampling_resume import audit
r=json.loads(Path('results/co3d_aligned_focal_diagnostic_20261002.json').read_text())
a,views=load_archived_draw();gt=np.stack([v['camera_pose'] for v in views]).astype(np.float32)
p=protocol(Path('checkpoints/Fast3R_ViT_Large_512'),42+int(a['scope']['base_index']),2)
for k in ('diagnostic_runner_sha256','historical_baseline_sha256','benchmark_scope','changed_variable','branches','GT_for_focal_or_PnP','GT_valid_mask_for_alignment','alignment_conf_percentile','prior_focal_runner_sha256'):
    p[k]=r['protocol'][k]
assert p['diagnostic_runner_sha256']==sha('scripts/diagnose_co3d_pose_aligned_focal.py')
assert p['prior_focal_runner_sha256']==sha('scripts/diagnose_co3d_pose_focal.py')
assert p['historical_baseline_sha256']==sha('results/co3d_pose_draw0_seed42_20261002.json')
assert not p['GT_valid_mask_for_alignment'] and p['alignment_conf_percentile']==0
validate(r,p,{'frames':a['returned_views']},gt)
old=json.loads(Path('results/co3d_focal_diagnostic_20261002.json').read_text())
for name in ('released_global_focal','raw_global_focal'):
    assert r['branches'][name]['metrics']==old['branches'][name]['metrics']
print('§4.2/Table 1：发布local→global相似变换代码接线诊断；无GT valid_mask，不是正式评测。')
for name,row in r['branches'].items():
    m=row['metrics']
    print(f"{name}: focal={row['estimated_focal']:.4f}px, RRA30={m['RRA_at_30']*100:.4f}%, RTA30={m['RTA_at_30']*100:.4f}%, mAA30={m['mAA_30']*100:.4f}%, PnP失败={len(row['pnp_failed_view_indices'])}/10")
print('同方向global PnP点/掩码/seed保持相同；旧global基线完全一致。发布方向mAA仍0。')
v=json.loads(Path('results/co3d_sampling_resume_synthetic_20261002.json').read_text())
assert v==audit()
assert v['sample_count']==100 and v['resume_boundary']==37 and v['suffix_request_count']==63
assert v['suffix_exact_equal'] and v['final_state_exact_equal']
assert not v['real_RGB_or_depth_loaded'] and v['model_forward_count']==0
print('合成连续100请求：第37请求后保存并恢复；余下63请求和最终状态逐项相同。')
print('无效场景：',v['invalid_scenes'],'；无效帧标记数：',v['invalid_frame_flags'])
print('JSON保存Python补采RNG、dataset RNG、invalidate和scene tracker；绑定pool/组合/映射/protocol。')
print('仅单worker完整请求边界采样恢复；不是GPU RNG或模型checkpoint，也未接入真实100@ runner。')
print('159项离线测试通过；未完成作者划分/权重对应或正式全量Table 1。')

§4.2/Table 1：发布local→global相似变换代码接线诊断；无GT valid_mask，不是正式评测。
released_global_focal: focal=27.9829px, RRA30=0.0000%, RTA30=13.3333%, mAA30=0.0000%, PnP失败=0/10
released_aligned_local_focal: focal=24.9949px, RRA30=0.0000%, RTA30=31.1111%, mAA30=0.0000%, PnP失败=0/10
raw_global_focal: focal=553.6604px, RRA30=2.2222%, RTA30=11.1111%, mAA30=0.0000%, PnP失败=0/10
raw_aligned_local_focal: focal=498.6280px, RRA30=6.6667%, RTA30=11.1111%, mAA30=0.6452%, PnP失败=0/10
同方向global PnP点/掩码/seed保持相同；旧global基线完全一致。发布方向mAA仍0。
合成连续100请求：第37请求后保存并恢复；余下63请求和最终状态逐项相同。
无效场景： [['toy', 'bad']] ；无效帧标记数： 23
JSON保存Python补采RNG、dataset RNG、invalidate和scene tracker；绑定pool/组合/映射/protocol。
仅单worker完整请求边界采样恢复；不是GPU RNG或模型checkpoint，也未接入真实100@ runner。
159项离线测试通过；未完成作者划分/权重对应或正式全量Table 1。


In [1]:
import json,sys
from pathlib import Path
sys.path.insert(0,'scripts')
from fast3r_hf_co3d_pose_smoke import load_archived_draw,sha
from prepare_co3d_continuous import inputs
r=json.loads(Path('results/co3d_continuous_prefix1_verified_20261002.json').read_text())
root=Path('results/co3d_continuous_prepare_20261002')
assert r['verified_request_count']==1 and r['expected_request_count']==100
assert r['input_tensors_GT_rng_and_shared_state_exact_equal'] and r['network_bytes']==0
assert sha('scripts/verify_co3d_continuous_prefix.py')==r['verifier_sha256']
assert sha(root/'initial.json')==r['initial_sha256']
assert sha(root/'request_000.json')==r['request_sha256']['request_000.json']
row=json.loads((root/'request_000.json').read_text())
a,views=load_archived_draw()
assert row['result']['returned_views']==inputs(views)
assert row['result']['base_index']==12303633 and row['after_state']['next_request']==1
assert not r['all_100_requests_prepared'] and not r['formal_pose_metrics_available']
print('§4.2/Table 1数据与采样前提：真实100@连续准备入口已接通；此快照仅核验第1请求。')
print('原41类/2011轨迹候选池、epoch0映射不变；draw0输入tensor/GT/RNG与旧归档完全相等。')
print('第1请求9个唯一帧、10views，保留重复帧；只读重放输入与共享after-state完全相等，网络0bytes。')
print('结果与after-state同一不可变JSON事务提交，恢复拒绝间断/链损坏/身份变化；162项离线测试通过。')
print('宿主fast3r-co3d-continuous-prepare.service已启动，接续第2请求准备；日志results/co3d_continuous_prepare.log。')
print('服务启动是归档快照，不代表当前实时状态；不占用GPU，保留raw2GiB/processed512MiB及空闲1GiB上限。')
print('尚无新模型前向/位姿成绩；100请求不是2011轨迹全枚举，也未确认作者划分/权重对应。')

§4.2/Table 1数据与采样前提：真实100@连续准备入口已接通；此快照仅核验第1请求。
原41类/2011轨迹候选池、epoch0映射不变；draw0输入tensor/GT/RNG与旧归档完全相等。
第1请求9个唯一帧、10views，保留重复帧；只读重放输入与共享after-state完全相等，网络0bytes。
结果与after-state同一不可变JSON事务提交，恢复拒绝间断/链损坏/身份变化；162项离线测试通过。
宿主fast3r-co3d-continuous-prepare.service已启动，接续第2请求准备；日志results/co3d_continuous_prepare.log。
服务启动是归档快照，不代表当前实时状态；不占用GPU，保留raw2GiB/processed512MiB及空闲1GiB上限。
尚无新模型前向/位姿成绩；100请求不是2011轨迹全枚举，也未确认作者划分/权重对应。


In [1]:
import json,sys
from pathlib import Path
sys.path.insert(0,'scripts')
from diagnose_co3d_nonfinite_depth import diagnose
from fast3r_hf_re10k_pose_eval import sha
r=json.loads(Path('results/co3d_nonfinite_depth_failure_20261002.json').read_text())
assert r==diagnose()
v=json.loads(Path('results/co3d_continuous_prefix3_verified_20261002.json').read_text())
assert v['verified_request_count']==3 and v['network_bytes']==0
assert v['input_tensors_GT_rng_and_shared_state_exact_equal']
root=Path('results/co3d_continuous_prepare_20261002')
assert sha(root/'initial.json')==v['initial_sha256']
for name,h in v['request_sha256'].items():assert sha(root/name)==h
assert r['raw_depth_positive_inf_count']==5 and r['raw_depth_nan_count']==0
assert r['reference_uint16_all_zero'] and r['reference_loader_effective_depth_all_zero']
assert not r['live_preprocessing_changed'] and not r['retry_started']
print('§4.2/Table 1前提：真实连续准备在第4请求失败，3/100完整事务全部只读重放一致。')
print('失败帧：',r['member'],'；5个正Inf，RGB/深度/掩码尺寸和scale正常。')
print('raw成员CRC/SHA核验通过；参考裁剪/缩放后maximum仍Inf，当前有限性守卫拒绝。')
print('只读算术诊断：参考uint16全部为0，原加载有效深度也全0；不是手动将Inf像素补成0。')
print('该证据尚非新版预处理+原加载器端到端等价验证；无代码语义修改、无服务重启。')
print('164项离线测试通过；旧数据/失败日志/前三个事务保留，无新增模型或正式位姿成绩。')
print('下步做版本化处理与补采等价验证，不能静默删帧、换scene或覆盖旧已绑定runner。')

§4.2/Table 1前提：真实连续准备在第4请求失败，3/100完整事务全部只读重放一致。
失败帧： stopsign/599_92267_182752/images/frame000081.jpg ；5个正Inf，RGB/深度/掩码尺寸和scale正常。
raw成员CRC/SHA核验通过；参考裁剪/缩放后maximum仍Inf，当前有限性守卫拒绝。
只读算术诊断：参考uint16全部为0，原加载有效深度也全0；不是手动将Inf像素补成0。
该证据尚非新版预处理+原加载器端到端等价验证；无代码语义修改、无服务重启。
164项离线测试通过；旧数据/失败日志/前三个事务保留，无新增模型或正式位姿成绩。
下步做版本化处理与补采等价验证，不能静默删帧、换scene或覆盖旧已绑定runner。


In [1]:
# §4.2 / Table 1: Inf帧参考文件等价与v2三请求迁移，不是位姿成绩
import json
from pathlib import Path
proof = json.loads(Path("results/co3d_inf_reference_v2_offline_verified_20261002.json").read_text())
migration = json.loads(Path("results/co3d_v2_prefix3_migration_verified_20261002.json").read_text())
assert all(proof[k] for k in ("image_depth_mask_bytes_equal", "npz_arrays_and_dtypes_equal", "original_loader_returns_none_and_invalidates", "strict_loader_returns_none_and_invalidates"))
assert proof["maximum_depth_preserved"] == "positive_infinity"
assert proof["network_bytes"] == proof["model_forward_count"] == 0
assert migration["verified_request_count"] == 3
assert migration["input_tensors_GT_rng_trace_and_shared_state_equal"]
assert migration["modality_bytes_and_npz_arrays_equal"]
assert not proof["formal_pose_metrics_available"] and not migration["full_paper_completed"]
print("§4.2/Table 1 preparation: actual Inf frame reference bytes/NPZ/original-loader equality verified.")
print("V2 first three requests: tensor/GT/trace/shared state equal frozen V1; independent identity and journals.")
print("Maximum NPZ preserved as +Inf; reference quantization zero, original loader invalidates; no manual fill or skipped scene.")
print("V2 preparation service restarted from request 4; this archived proof is NOT live completion status or formal pose scores.")
failure = json.loads(Path("results/co3d_v2_request4_failure_20261002.json").read_text())
assert failure["completed_request_count"] == 3 and not failure["failed_request_committed"]
assert failure["load_trace"][0]["status"] == "zero_masked_depth_after_crop"
assert failure["load_trace"][1]["status"] == "hard_error"
assert failure["network_bytes"] == failure["model_forward_count"] == 0
print("LATEST: v2 service exited 1; request4 uncommitted. Frame42:", failure["failure"]["depth_positive_inf_count"], "positive Inf;", failure["failure"]["depth_negative_count"], "negative depth; NaN=", failure["failure"]["depth_nan_count"])
print("Do not restart unchanged v2. Next: exact negative bit patterns/reference output, versioned proof; no silent skip or formal score.")


§4.2/Table 1 preparation: actual Inf frame reference bytes/NPZ/original-loader equality verified.
V2 first three requests: tensor/GT/trace/shared state equal frozen V1; independent identity and journals.
Maximum NPZ preserved as +Inf; reference quantization zero, original loader invalidates; no manual fill or skipped scene.
V2 preparation service restarted from request 4; this archived proof is NOT live completion status or formal pose scores.
LATEST: v2 service exited 1; request4 uncommitted. Frame42: 900 positive Inf; 3 negative depth; NaN= 0
Do not restart unchanged v2. Next: exact negative bit patterns/reference output, versioned proof; no silent skip or formal score.


In [1]:
# §4.2/Table 1: signed Inf reference equivalence; not pose scores
import json
from pathlib import Path
diag = json.loads(Path("results/co3d_negative_reference_diagnostic_20261002.json").read_text())
proof = json.loads(Path("results/co3d_signed_inf_reference_v3_verified_20261002.json").read_text())
migration = json.loads(Path("results/co3d_v3_prefix3_migration_verified_20261002.json").read_text())
assert diag["prepared_record"]["raw_negative_values"] == [
    {"position_yx": [510,80], "uint16_bits": 64512, "value": "negative_infinity"},
    {"position_yx": [552,961], "uint16_bits": 62917, "value": -23632.0},
    {"position_yx": [552,962], "uint16_bits": 62940, "value": -24000.0}]
assert all(proof[k] for k in ("image_depth_mask_bytes_equal", "npz_arrays_and_dtypes_equal", "original_loader_returns_none_and_invalidates", "strict_loader_returns_none_and_invalidates"))
assert proof["maximum_depth_preserved"] == "positive_infinity"
assert proof["prepared_record"]["stats"]["raw_negative_depth_count"] == 3
assert proof["network_bytes"] == proof["model_forward_count"] == 0
assert migration["verified_request_count"] == 3
assert migration["input_tensors_GT_rng_trace_and_shared_state_equal"]
assert migration["modality_bytes_and_npz_arrays_equal"]
assert not proof["formal_pose_metrics_available"] and not migration["full_paper_completed"]
print("§4.2/Table 1 input preparation: frame42 values -Inf, -23632, -24000 retained, not filled.")
print("Actual reference RGB/depth/mask bytes and NPZ arrays/dtypes equal v3; maximum +Inf preserved, loader invalidates zero depth.")
print("V3 first three requests independently replayed: inputs/GT/trace/shared state equal frozen v1; immutable identities separate.")
print("V3 service resumed request4 preparation. This archived analysis is not live completion or formal pose metrics.")
failure = json.loads(Path("results/co3d_v3_request4_failure_20261002.json").read_text())
assert failure["completed_request_count"] == 3 and not failure["failed_request_committed"]
assert failure["failure"]["depth_positive_inf_count"] == failure["failure"]["depth_nan_count"] == 0
assert failure["failure"]["depth_negative_count"] == 4
assert [t["status"] for t in failure["load_trace"][:8]] == ["zero_masked_depth_after_crop"]*8
assert len(failure["pool_attempts"]) == 2
assert failure["network_bytes"] == failure["model_forward_count"] == 0
print("LATEST: v3 stopped safely, 3/100 committed. Original sampler retried after 8 zero-depth candidates, not all 202 frames audited.")
print("New frame45: 4 finite negative depths, no +Inf/NaN; requires separate actual-reference quantization/loader proof, no unchanged v3 restart.")


§4.2/Table 1 input preparation: frame42 values -Inf, -23632, -24000 retained, not filled.
Actual reference RGB/depth/mask bytes and NPZ arrays/dtypes equal v3; maximum +Inf preserved, loader invalidates zero depth.
V3 first three requests independently replayed: inputs/GT/trace/shared state equal frozen v1; immutable identities separate.
V3 service resumed request4 preparation. This archived analysis is not live completion or formal pose metrics.
LATEST: v3 stopped safely, 3/100 committed. Original sampler retried after 8 zero-depth candidates, not all 202 frames audited.
New frame45: 4 finite negative depths, no +Inf/NaN; requires separate actual-reference quantization/loader proof, no unchanged v3 restart.


In [1]:
# §4.2/Table 1: finite signed depth reference semantics; not pose metrics
import json
from pathlib import Path
diag = json.loads(Path("results/co3d_finite_negative_reference_diagnostic_20261002.json").read_text())
proof = json.loads(Path("results/co3d_finite_signed_reference_v4_verified_20261002.json").read_text())
migration = json.loads(Path("results/co3d_v4_prefix3_migration_verified_20261002.json").read_text())
assert all(proof[k] for k in ("image_depth_mask_bytes_equal", "npz_arrays_and_dtypes_equal", "strict_and_original_loader_outputs_equal"))
assert proof["maximum_depth_preserved"] == 61952
assert proof["loaded_geometry"]["valid"] and not proof["loaded_geometry"]["invalidated"]
assert proof["loaded_geometry"]["positive_depth_count"] == 12985
assert proof["prepared_record"]["stats"]["processed_negative_depth_count"] == 4
assert proof["prepared_record"]["stats"]["negative_quantized_positive_count"] == 4
assert proof["network_bytes"] == proof["model_forward_count"] == 0
assert migration["verified_request_count"] == 3
assert migration["input_tensors_GT_rng_trace_and_shared_state_equal"] and migration["modality_bytes_and_npz_arrays_equal"]
assert not proof["formal_pose_metrics_available"] and not migration["full_paper_completed"]
print("§4.2/Table 1 input only: four finite negative depths retained; actual reference uint16 cast yields four positive quantized pixels.")
print("Reference/v4 bytes, NPZ and original-loader geometry equal. Loader valid; positive-depth pixels:", proof["loaded_geometry"]["positive_depth_count"])
print("Maximum metadata:", proof["maximum_depth_preserved"], "; loaded maximum:", proof["loaded_geometry"]["maximum_loaded_depth"], "; not claimed physical accuracy.")
print("V4 three-request replay equals frozen v1; separate identity/journals. Service resumed request4; archived proof is not live completion or formal scores.")
prefix = json.loads(Path("results/co3d_v4_prefix_snapshot_20261002T1200_verified.json").read_text())
assert prefix["verified_request_count"] == 4
assert prefix["input_tensors_GT_rng_and_shared_state_exact_equal"]
assert prefix["network_bytes"] == prefix["model_forward_count"] == 0
assert not prefix["all_100_requests_prepared"] and not prefix["formal_pose_metrics_available"]
print("NEW completed boundary: 4/100 requests independently replayed, including original scene retry/oversampling; snapshot, not live count.")


§4.2/Table 1 input only: four finite negative depths retained; actual reference uint16 cast yields four positive quantized pixels.
Reference/v4 bytes, NPZ and original-loader geometry equal. Loader valid; positive-depth pixels: 12985
Maximum metadata: 61952.0 ; loaded maximum: 166.37753295898438 ; not claimed physical accuracy.
V4 three-request replay equals frozen v1; separate identity/journals. Service resumed request4; archived proof is not live completion or formal scores.
NEW completed boundary: 4/100 requests independently replayed, including original scene retry/oversampling; snapshot, not live count.


### CO3D候选100请求的完整位姿实测（§4.2 / Table 1，2026-10-03）

100请求均已评测并保存，独立核验器逐项连接prepared输入tensor/GT、逐请求checkpoint与完整JSON，从c2w重算全部4500个相机对和aggregate。该候选100@实际覆盖99轨迹/38类、884唯一返回RGB；作者processed split及公开checkpoint与论文实验对应仍未确认。本单元保留候选实测身份，论文数字仅作参照。

模型输入是RGB与true_shape；预测global点图用于焦距估计和PnP。对每个像素，PnP寻找相机参数，使3D点投影接近对应2D像素：$\tilde u\sim K(RX+t)$。本机采用发布版首视图global焦距估计及conf>1的RANSAC-PnP；论文描述随机焦距猜测及top15%置信度，需后续受控诊断。GT pose/K不参与预测焦距或PnP。

保存的相机矩阵是camera-to-world $T_i$，相对位姿为$T_{ij}=T_i^{-1}T_j$。RRA与RTA分别统计旋转角误差、平移方向角误差小于给定阈值的相机对比例；本机沿用发布版平移公式的符号二义性及零向量处理，不改公式。mAA@30沿用仓库PyTorch函数：对$\max(e_R,e_t)$建立0–30°的31-bin直方图，以全部45对为分母，累计后取均值，再平均100请求。不能把重建的距离mean/median当作这些角度指标。

64请求含重复视角，共160个重复相机对，全部保留；8请求估计焦距为0，80/1000视角PnP失败，公开identity fallback全部保留和计数。它们仍参与最终成绩，不能解释为80次成功的identity位姿估计。v2保护误停的日志/断点、完整本地v3报告及所有历史GPU结果保留。

本单元只读保存证据、打印真实对照表和绘图。完整本地报告存在时执行全部独立重算；仅持有Git小摘要时明确只核验归档摘要。结果图：`results/figures/co3d_candidate100_pose.png`。正式Table1、RE10K缺失场景、独立训练消融及完整训练仍未完成。

In [1]:
import json, sys, statistics
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path("scripts").resolve()))
from verify_co3d_candidate_pose_report import verify, sha, KEYS, REPORT, SUMMARY

summary = json.loads(SUMMARY.read_text())
assert summary["request_count"] == 100 and summary["pair_count"] == 4500
assert [row["request"] for row in summary["requests"]] == list(range(100))
assert not summary["formal_Table1_result"] and not summary["full_paper_completed"]
assert sha("scripts/verify_co3d_candidate_pose_report.py") == summary["verifier_sha256"]
for key in KEYS:
    assert abs(statistics.mean(row["metrics"][key] for row in summary["requests"])
               - summary["macro_mean_request_metrics"][key]) < 1e-7
assert sum(len(row["pnp_failed_view_indices"]) for row in summary["requests"]) == 80
if REPORT.exists():
    assert sha(REPORT) == summary["complete_report_sha256"]
    assert verify() == summary
    print("VERIFIED: all 100 saved pose requests / 4500 pairs, input/GT links, bound hashes, identity fallbacks and every aggregate; no network/model forward.")
else:
    print("ARCHIVED SUMMARY ONLY: compact request metrics verified; full local pose/GT report is absent, so no independent pose recomputation was performed.")
order = ["RRA_at_5", "RRA_at_15", "RTA_at_5", "RTA_at_15", "mAA_30"]
actual = summary["macro_mean_request_metrics"]
reference = summary["paper_Table1_Fast3R_reference_only"]
table = pd.DataFrame([{"metric": k, "candidate_percent": 100*actual[k],
                       "paper_reference_percent": 100*reference[k],
                       "reference_minus_candidate_pp": 100*(reference[k]-actual[k])}
                      for k in order]).set_index("metric")
print(table.round(4).to_string())
print("Candidate coverage:", {k: summary[k] for k in ("unique_trajectories", "unique_categories",
    "unique_returned_frames", "requests_with_duplicate_views", "duplicate_pair_count")})
print("PnP failures: 80/1000 views across 8 requests; zero focal requests: 8. All retained.")
print("Paper split/checkpoint equivalence unresolved; formal Table 1 and full paper remain incomplete.")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
positions = np.arange(len(order))
axes[0].bar(positions-.18, [100*actual[k] for k in order], .36, label="Candidate adaptation")
axes[0].bar(positions+.18, [100*reference[k] for k in order], .36, label="Paper reference")
axes[0].set_xticks(positions, ["RRA@5", "RRA@15", "RTA@5", "RTA@15", "mAA@30"])
axes[0].set_ylim(0, 100); axes[0].set_ylabel("Percent (higher is better)")
axes[0].set_title("Protocols / weights are not proven equivalent"); axes[0].legend(fontsize=8)
failed = np.array([bool(row["pnp_failed_view_indices"]) for row in summary["requests"]])
scores = np.array([100*row["metrics"]["mAA_30"] for row in summary["requests"]])
axes[1].scatter(np.arange(100)[~failed], scores[~failed], s=14, label="No recorded PnP failure")
axes[1].scatter(np.arange(100)[failed], scores[failed], s=30, marker="x", color="tab:orange",
                label="PnP identity fallback retained")
axes[1].set_xlabel("Fixed request index"); axes[1].set_ylabel("Request mAA@30 (%)")
axes[1].set_ylim(-3, 103); axes[1].set_title("All 100 requests, including failed views")
axes[1].legend(fontsize=8)
fig.suptitle("CO3D candidate 100-request pose evaluation, seed 42")
figure = Path("results/figures/co3d_candidate100_pose.png")
figure.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure, dpi=180); plt.close(fig)
print("Saved scientific figure:", figure)


VERIFIED: all 100 saved pose requests / 4500 pairs, input/GT links, bound hashes, identity fallbacks and every aggregate; no network/model forward.
           candidate_percent  paper_reference_percent  reference_minus_candidate_pp
metric                                                                             
RRA_at_5             28.0444                     90.2                       62.1556
RRA_at_15            31.2889                     96.2                       64.9111
RTA_at_5             21.9778                     68.2                       46.2222
RTA_at_15            28.5778                     81.6                       53.0222
mAA_30               23.8007                     75.0                       51.1993
Candidate coverage: {'unique_trajectories': 99, 'unique_categories': 38, 'unique_returned_frames': 884, 'requests_with_duplicate_views': 64, 'duplicate_pair_count': 160}
PnP failures: 80/1000 views across 8 requests; zero focal requests: 8. All retained.
Paper spl

## §4.2 / Table 1：作者协议纠正与固定输入PnP 2×2诊断

作者[公开回复 #78](https://github.com/facebookresearch/fast3r/issues/78#issuecomment-2844393603)描述实际1000次CO3D test采样、可能超过41类；发布配置仍100次。旧41类/100请求只是候选适配，不等于作者benchmark。精确processed清单、RNG及HF权重组别映射仍未知。

神经网络由RGB预测global 3D点图和confidence，之后PnP解相机：$u\approx\pi(K(f)[R\mid t]X)$。本次不改网络、权重、GT或输出方向，每个固定request的一次前向供四分支共享，只改变焦距估计和confidence筛点。baseline位姿须先复现原报告，再运行其他分支。

- 发布版：第一帧global点图p10估焦距，PnP用conf>1。
- 仅mask：相同焦距，conf>quantile(.85)；并列阈值严格排除，实际点数不一定15%。
- 仅搜索：发布fast_pnp(None)在[S/2,3S]作100个geomspace猜测，根据inliers选第一帧焦距再共享。
- 两项：搜索+top15。搜索是确定性网格，不冒充论文随机猜测；第一帧搜索失败则全部identity回退，无GT替代。

request0/2/3根据原报告挑选（低分/高分/零焦距），不是无偏样本，**不报告三者平均或正式Table1成绩**。零焦距request3的搜索分支没有PnP回退，却仍mAA0，说明求解返回与位姿正确性不同。其他低分request0也未被这两项修复。[作者#76](https://github.com/facebookresearch/fast3r/issues/76#issuecomment-2842609184)的portrait/强制landscape裁剪建议是下一步线索，不是本机已证实根因；改变输入crop的实验必须明确使用不同网络输入。

下面只读新JSON并重算pose指标/绘图，不重新加载模型或GPU；旧93个单元保留。独立证明不重推原预测点图，mask/预测哈希只验证冻结代码的生成记录。

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from fast3r.eval.cam_pose_metric import camera_to_rel_deg, calculate_auc
import torch

report_path = Path('results/co3d_candidate_pnp_diagnostic_v1_20261003.json')
r = json.loads(report_path.read_text())
v = json.loads(Path('results/co3d_candidate_pnp_diagnostic_verified_20261003.json').read_text())
a = json.loads(Path('results/co3d_author_protocol_update_20261003.json').read_text())
assert hashlib.sha256(report_path.read_bytes()).hexdigest() == v['report_sha256']
assert r['formal_Table1_result'] is False and r['full_paper_completed'] is False
assert r['model_forward_count'] == 3 and r['request_count'] == 3
assert a['author_stated_request_count'] == 1000 and a['public_config']['resized_request_count'] == 100
assert a['author_exact_processed_sequence_JSON_verified'] is False
baseline = json.loads(Path('results/co3d_pose_100_seed42_adaptation_v3.json').read_text())
assert hashlib.sha256(Path('results/co3d_pose_100_seed42_adaptation_v3.json').read_bytes()).hexdigest() == v['baseline_report_sha256']
branches = ['released', 'top15_only', 'search_only', 'search_top15']
records = []
for row in r['requests']:
    assert row['paired_same_forward'] and row['forward_count'] == 1
    assert row['prediction_before'] == row['prediction_after']
    assert row['baseline_pose_max_abs_difference'] == 0
    gt = torch.tensor(row['gt_c2w'], dtype=torch.float32)
    for name in branches:
        b = row['branches'][name]
        pred = torch.tensor(b['predicted_c2w'], dtype=torch.float32)
        rotation, translation = camera_to_rel_deg(pred, gt, 'cpu', 10)
        assert len(rotation) == len(translation) == 45
        values = {f'RRA_at_{k}': float((rotation < k).float().mean()) for k in (5, 15, 30)}
        values.update({f'RTA_at_{k}': float((translation < k).float().mean()) for k in (5, 15, 30)})
        values['mAA_30'] = float(calculate_auc(rotation, translation, max_threshold=30))
        assert all(abs(values[k] - b['metrics'][k]) <= 1e-7 for k in values)
        assert all(torch.equal(pred[i], torch.eye(4)) for i in b['pnp_failed_view_indices'])
        masks = b['confidence_masks']
        records.append({'request': row['request'], 'scene': row['scene'], 'branch': name,
            'RRA15 %': 100*values['RRA_at_15'], 'RTA15 %': 100*values['RTA_at_15'],
            'mAA30 %': 100*values['mAA_30'], 'focal px': b['estimated_focal'],
            'PnP fallback /10': len(b['pnp_failed_view_indices']),
            'mask min %': 100*min(x['retained_points']/x['total_points'] for x in masks),
            'mask max %': 100*max(x['retained_points']/x['total_points'] for x in masks)})
frame = pd.DataFrame(records)
display(frame.round(4))
print('Author statement: 1000 requests; released config: 100. Exact split/checkpoint mapping still unverified.')
print('3 selected probes, 12 branches; baseline exact, all 45 pairs and identity fallbacks retained. No benchmark average.')
fig, axes = plt.subplots(1, 3, figsize=(12.4, 4.0), sharey=True)
colors = ['#4c78a8', '#59a14f', '#f28e2b', '#b279a2']
for axis, row in zip(axes, r['requests']):
    vals = [100*row['branches'][name]['metrics']['mAA_30'] for name in branches]
    axis.bar(range(4), vals, color=colors)
    axis.set_xticks(range(4), ['Released', 'Top15', 'Search', 'Both'])
    axis.set_title(f"Request {row['request']}: {row['scene'].split('/')[0]}")
    axis.set_ylim(0, 108)
    axis.grid(axis='y', alpha=.2)
    for j, name in enumerate(branches):
        failures = len(row['branches'][name]['pnp_failed_view_indices'])
        axis.text(j, vals[j]+2, f'{vals[j]:.1f}%\n{failures} fail', ha='center', fontsize=8)
axes[0].set_ylabel('mAA@30 (%) — selected diagnostic probes')
fig.suptitle('Same-forward focal × confidence diagnostic; not Table 1, no probe average')
fig.tight_layout(rect=[0, 0, 1, .93])
figure = Path('results/figures/co3d_pnp_factorial_diagnostic.png')
fig.savefig(figure, dpi=160, bbox_inches='tight')
plt.show()
print('Scientific figure:', figure)


,request,scene,branch,RRA15 %,RTA15 %,mAA30 %,focal px,PnP fallback /10,mask min %,mask max %
0,0,bicycle/374_41967_84033,released,0.0000,11.1111,0.0000,28.1469,0,75.5076,99.6597
1,0,bicycle/374_41967_84033,top15_only,0.0000,0.0000,0.0000,28.1469,0,14.9933,15.0004
2,0,bicycle/374_41967_84033,search_only,2.2222,2.2222,0.0000,976.9861,0,75.5076,99.6597
3,0,bicycle/374_41967_84033,search_top15,0.0000,4.4444,0.0000,1508.4506,0,14.9933,15.0004
4,2,cup/400_51462_101232,released,100.0000,97.7778,92.2581,515.7850,0,99.9954,100.0000
5,2,cup/400_51462_101232,top15_only,100.0000,97.7778,94.2652,515.7850,0,14.9124,14.9943
6,2,cup/400_51462_101232,search_only,100.0000,97.7778,93.0466,500.1084,0,99.9954,100.0000
7,2,cup/400_51462_101232,search_top15,100.0000,97.7778,92.4731,465.1829,0,14.9124,14.9943
8,3,stopsign/249_26596_53531,released,24.4444,0.0000,0.0000,0.0000,10,9.7092,99.8433
9,3,stopsign/249_26596_53531,top15_only,24.4444,0.0000,0.0000,0.0000,10,9.7092,15.0004


Author statement: 1000 requests; released config: 100. Exact split/checkpoint mapping still unverified.
3 selected probes, 12 branches; baseline exact, all 45 pairs and identity fallbacks retained. No benchmark average.


<Figure size 1240x400 with 3 Axes>

Scientific figure: results/figures/co3d_pnp_factorial_diagnostic.png


## §4.2 / Table 1：portrait 的神经网络接口与像素几何（2026-10-03）

这一阶段不是再调阈值，而是检查“像素对应哪个 token、哪个 3D 点、哪个投影坐标”。作者[Issue #76](https://github.com/facebookresearch/fast3r/issues/76#issuecomment-2842609184)回忆评测时裁剪成横向512×384；我们先检查实际公开HF架构，再用同一组帧做受控对照。原100请求和上次PnP消融全部保留。

### 两个不同的接口问题

1. **神经网络的空间顺序**：HF配置使用`PatchEmbedDust3R`，不是`ManyAR_PatchEmbed`，也不是DINO。前者对loader的384×512张量按16×16 patch生成24×32 token，忽略`true_shape`。但`landscape_only=False`的DPT用portrait的`true_shape=[512,384]`把同一token序列reshape成32×24。**reshape不等于transpose**。实际发布patch类的合成编号测试有99.7396%的编号位置不等于真正空间转置；这不是模型预测错误率。
2. **投影的像素顺序**：loader把portrait点图/图像网格转置，并把GT内参行交换为$K'=SK$，其中$S=\begin{bmatrix}0&1&0\\1&0&0\\0&0&1\end{bmatrix}$。3D坐标和GT相机位姿没有对应旋转。PnP却重新构造$K_f=\begin{bmatrix}f&0&W/2\\0&f&H/2\\0&0&1\end{bmatrix}$。因此$\tilde u'=S\tilde u\sim SK[R|t]X$不能直接当成原位姿的$K_f[R|t]X$；像素转置不是保持正深度的普通相机旋转。$\det S=-1$，不能悄悄把反射矩阵当SO(3)旋转。数学上还存在翻转z的负深度投影歧义，不能把PnP返回success当成正确几何。

合成完美3D点的发布PnP测试：原坐标旋转误差0°、最大重投影误差约6.76e-6px；仅转置像素却使用标准K，PnP仍返回成功，但旋转误差约179.84°。这证明接口可以制造错误，不证明两个问题完全解释实际模型误差。

原候选100请求中63全portrait、29全landscape、8混合；1000输入视角的661个portrait内参行列式均负。这只是历史候选分布，不是作者全量分布。

### 同帧对照如何保持公平

固定request0/2/3，帧编号、顺序、重复帧、GT相机、公开权重、每次前向seed42+request、精度、PnP/焦距/mask不变。GT仅用于原数据预处理/裁剪和评分，不进入网络、估计焦距或PnP。

- 原始分支：直接引用已归档结果，不重跑原100请求。
- 仅形状标记：RGB字节不变，只令`true_shape`等于loader张量的384×512。这会改变DPT执行，**是新的前向**，且不修复转置后的像素几何。
- 强制横向裁剪：只从固定基类方法中去除portrait/square目标分辨率反转；保留主点crop、Lanczos/BICUBIC resize、深度最近邻与半像素K更新，不旋转图像、不修改源数据。portrait的输入像素改变，视野也改变，**不是同输入/同次前向消融**。固定帧不补采/不删失败帧，改变crop RNG的标记单独记录，不冒称仍是原sampler after-state。

共5个新前向：两个portrait各2次，原本横向cup的两个新条件输入完全一致，复用1次预测，并与历史poses/focal/metrics匹配。6分支×45对相机全部保留，独立核验输入/GT/crop K/RNG与270个pair的指标。

**真实性边界**：三个probe是按历史表现挑选的诊断，不能求平均冒充Table1；强制横向裁剪改善两例不证明达到论文全量成绩，也不证明裁剪视野与token/PnP几何各自贡献多少。精确作者processed清单、1000次采样与权重组别仍未确认。下一步先预算51类/1000请求的作者描述候选协议，不盲目复跑旧100请求。

In [1]:
from pathlib import Path
import json, hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def file_sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

audit = json.loads(Path("results/co3d_portrait_geometry_audit_20261003.json").read_text())
report_path = Path("results/co3d_landscape_input_diagnostic_v1_20261003.json")
report = json.loads(report_path.read_text())
proof = json.loads(Path("results/co3d_landscape_input_diagnostic_verified_20261003.json").read_text())
baseline_path = Path("results/co3d_pose_100_seed42_adaptation_v3.json")
baseline = json.loads(baseline_path.read_text())
assert file_sha(report_path) == proof["report_sha256"]
assert file_sha(baseline_path) == proof["baseline_report_sha256"] == audit["baseline_report_sha256"]
assert not report["formal_Table1_result"] and not proof["formal_Table1_result"]
assert report["model_forward_count"] == proof["distinct_new_forward_count"] == 5
assert proof["all_270_new_branch_pairs_recomputed"] and proof["landscape_control_matches_archived_baseline"]
assert [r["request"] for r in report["requests"]] == [0, 2, 3]
for path, digest in report["protocol"]["source_sha256"].items():
    assert file_sha(path) == digest
rows = []
for row in report["requests"]:
    original = baseline["requests"][row["request"]]
    for name, value in [("archived original", original), *row["branches"].items()]:
        rows.append({"request": row["request"], "scene": row["scene"], "condition": name,
            "mAA30 (%)": 100*value["metrics"]["mAA_30"],
            "RRA15 (%)": 100*value["metrics"]["RRA_at_15"],
            "RTA15 (%)": 100*value["metrics"]["RTA_at_15"],
            "focal (px)": value["estimated_focal"], "PnP failed views": len(value["pnp_failed_view_indices"])})
table = pd.DataFrame(rows)
display(table.round(4))
print("Candidate orientation counts:", audit["candidate_request_orientation_counts"], "; portrait views:", audit["portrait_view_count"])
print("Actual HF patch class:", audit["public_HF_encoder_patch_class"], "; synthetic token-label mismatch fraction:", audit["token_probe"]["token_label_mismatch_fraction"])
display(pd.DataFrame(audit["geometry_probe"]["PnP"]).round(6))
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, i in zip(axes, [0, 2, 3]):
    data = table[table["request"] == i]
    bars = ax.bar(["Original", "Shape only", "Landscape crop"], data["mAA30 (%)"], color=["#788895", "#e7a54b", "#3796a6"])
    for bar, val in zip(bars, data["mAA30 (%)"]):
        ax.text(bar.get_x()+bar.get_width()/2, val+2, f"{val:.2f}%", ha="center", fontsize=9)
    ax.set_title(f"request {i}: {data.iloc[0]['scene'].split('/')[0]}")
    ax.set_ylim(0, 108); ax.tick_params(axis="x", labelrotation=20)
axes[0].set_ylabel("mAA@30 (%) — higher is better")
fig.suptitle("Fixed-frame input diagnostic — performance-selected probes, NOT Table 1")
fig.tight_layout()
out = Path("results/figures/co3d_landscape_input_diagnostic.png")
out.parent.mkdir(parents=True, exist_ok=True); fig.savefig(out, dpi=160, bbox_inches="tight")
plt.close(fig)
print("5 distinct new forwards; exact landscape control; changed portrait RGB; no diagnostic average.")
print("No formal Table 1 / full-training / full-paper completion claim. Figure:", out)


,request,scene,condition,mAA30 (%),RRA15 (%),RTA15 (%),focal (px),PnP failed views
0,0,bicycle/374_41967_84033,archived original,0.0000,0.0000,11.1111,28.1469,0
1,0,bicycle/374_41967_84033,metadata_landscape,0.0000,4.4444,2.2222,547.1462,0
2,0,bicycle/374_41967_84033,forced_landscape_crop,59.7133,77.7778,77.7778,717.5187,0
3,2,cup/400_51462_101232,archived original,92.2581,100.0000,97.7778,515.7850,0
4,2,cup/400_51462_101232,metadata_landscape,92.2581,100.0000,97.7778,515.7850,0
5,2,cup/400_51462_101232,forced_landscape_crop,92.2581,100.0000,97.7778,515.7850,0
6,3,stopsign/249_26596_53531,archived original,0.0000,24.4444,0.0000,0.0000,10
7,3,stopsign/249_26596_53531,metadata_landscape,0.0000,13.3333,6.6667,585.7639,0
8,3,stopsign/249_26596_53531,forced_landscape_crop,12.4014,28.8889,20.0000,728.6761,0


Candidate orientation counts: {'all_portrait': 63, 'all_landscape': 29, 'mixed': 8} ; portrait views: 661
Actual HF patch class: PatchEmbedDust3R ; synthetic token-label mismatch fraction: 0.9973958134651184


,branch,rotation_error_deg,median_reprojection_error_px,max_reprojection_error_px,positive_depth_fraction,PnP_returned_success
0,native_portrait,0.000000,0.000002,0.000007,1.0,True
1,transposed_pixels_standard_K,179.841571,5.062494,30.787584,1.0,True


5 distinct new forwards; exact landscape control; changed portrait RGB; no diagnostic average.
No formal Table 1 / full-training / full-paper completion claim. Figure: results/figures/co3d_landscape_input_diagnostic.png


## §4.2 / Table 1：历史工程计划与实际预算的边界

旧 seed42 计划强制1000条不同轨迹、每条随机10帧，只是工程提案，不是发布的 `1000 @ Co3d_Multiview(seed=777)`。旧 JSON 和磁盘快照保留，但后续评测不使用它；raw2GiB/processed512MiB 是历史缓存上限，不能当作1000请求能装下的预算证明。下面只读取旧报告，不运行模型或重新写结果。


In [1]:
import json
from pathlib import Path
legacy = json.loads(Path("results/co3d_51_1000_protocol_budget_20261003.json").read_text())
assert legacy["official_51_manifest"] == {"category_count": 51, "sequence_count": 2511, "candidate_frame_count": 498757}
assert legacy["fixed_plan"]["unique_frame_slots"] == 10000
assert not legacy["formal_Table1_result"] and not legacy["author_protocol_equivalence_verified"]
print("Historical engineering proposal: NOT released sampling, NOT a storage fit proof, NOT used for evaluation.")
print("Candidate metadata:", legacy["official_51_manifest"])
print("Legacy proposal distinct sequences / unique frames:", legacy["fixed_plan"]["distinct_sequence_count"], legacy["fixed_plan"]["unique_frame_slots"])
print("Historical disk snapshot bytes:", legacy["storage_envelope"]["current_filesystem_free_bytes"])
print("Historical proposal digest:", legacy["fixed_plan"]["request_plan_sha256"])


Historical engineering proposal: NOT released sampling, NOT a storage fit proof, NOT used for evaluation.
Candidate metadata: {'candidate_frame_count': 498757, 'category_count': 51, 'sequence_count': 2511}
Legacy proposal distinct sequences / unique frames: 1000 10000
Historical disk snapshot bytes: 33527140352
Historical proposal digest: 34847924f082fd4536fe710a688cb82fda684f2692597a785289d3f77c9a7933


## §4.2 / Table 1：51类候选的发布采样器1000@名义追踪

本单元复用发布的 `_get_views/_fetch_views_for_pool` 和 `ResizedDataset`，不是以无重复随机抽帧代替源代码。dataset seed777有发布/作者依据；Python combination seed42、epoch0是明确声明的审计选择，原作者processed JSON/顺序/RNG/HF训练组别尚未恢复。

1000请求在“每个初始尝试都有效”的stub下涉及836轨迹、51类、8835不同帧；581请求有重复视图，最少6不同帧。原始有序候选帧池不改动。实际深度/背景mask、补采和scene重试仍可能改变输入，这不是已准备好的1000请求或Table1分数。进一步计算±4 jitter和最多5个scene尝试的保守可达集合：2205轨迹/103599帧；该集合用于预算上界，不能冒充实际样本。

目录预算服务 `fast3r-co3d51-source-storage.service` 复用38类旧本地完整索引，只读取尚无本地索引的13类目录；原235包完整范围扫描不重做。目录名字/压缩长度不等于RGB/depth/mask CRC、GT/crop或解码证明；预处理容量仍需另算。断点按archive和category原子保存，禁止覆盖旧报告/清理缓存。


In [2]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path("scripts").resolve()))
from audit_co3d_51_released_sampler import build_source_plan, digest, sha, SOURCE_FILES
report = json.loads(Path("results/co3d_51_released_sampler_20261003.json").read_text())
plan = json.loads(Path(report["plan_file"]).read_text())
assert sha(Path(report["plan_file"])) == report["plan_file_sha256"]
assert digest(plan) == report["plan_content_sha256"]
selected = json.loads(Path("data/co3d_test_metadata/selected_seqs_test_reconstructed.json").read_text())
replay = build_source_plan(selected)
replay["candidate_manifest_sha256"] = sha(Path("data/co3d_test_metadata/selected_seqs_test_reconstructed.json"))
replay["source_sha256"] = {p: sha(Path(p)) for p in SOURCE_FILES}
assert replay == plan
assert report["nominal_counts"] == {"nominal_sample_count": 1000, "nominal_distinct_trajectory_count": 836, "nominal_distinct_category_count": 51, "nominal_distinct_frame_count": 8835, "samples_with_duplicate_views": 581, "min_distinct_frames_in_a_sample": 6}
assert report["potential_retry_jitter_frame_count"] == 103599
for key in ("author_protocol_equivalence_verified", "actual_depth_validity_verified", "formal_Table1_result", "directory_storage_budget_complete", "full_paper_completed"):
    assert not report[key]
assert report["model_forward_count"] == report["network_bytes_transferred"] == 0
print("§4.2/Table1: released source ALL-VALID nominal trace, not GT-realized views or pose scores.")
print(json.dumps(report["nominal_counts"], indent=2))
print("Retry/jitter reachability sequences / frames:", report["potential_retry_jitter_sequence_count"], report["potential_retry_jitter_frame_count"])
print("Source choices: combination seed42; dataset seed777; resized epoch0; author RNG unresolved.")
print("Plan exact source replay verified; SHA256:", report["plan_file_sha256"])
print("Legacy 1000-distinct/10000-frame proposal rejected as evaluation protocol; no Table1 score.")


§4.2/Table1: released source ALL-VALID nominal trace, not GT-realized views or pose scores.
{
  "nominal_sample_count": 1000,
  "nominal_distinct_trajectory_count": 836,
  "nominal_distinct_category_count": 51,
  "nominal_distinct_frame_count": 8835,
  "samples_with_duplicate_views": 581,
  "min_distinct_frames_in_a_sample": 6
}
Retry/jitter reachability sequences / frames: 2205 103599
Source choices: combination seed42; dataset seed777; resized epoch0; author RNG unresolved.
Plan exact source replay verified; SHA256: cd7a0b1ef5ea4633a88360f91823a55e5a4e09d028af4c5967df0068ff75ff32
Legacy 1000-distinct/10000-frame proposal rejected as evaluation protocol; no Table1 score.


### §4.2 / Table 1 前提：51类源采样目录预算完成，真实数据准备已接线

目录预算服务已正常结束0；独立核验复算51类的路径集合、数量、大小、源索引与收据SHA。复用38类冻结索引，只补13类缺失目录，不重跑旧235包扫描。

名义8835帧RGB/depth/mask原始成员合计 **6,539,401,033bytes（6.54GB）**；最坏可达补采集合103599帧合计 **76,458,893,204bytes（76.46GB）**。这只是目录声明大小，不是CRC/解码/GT有效性证明。不能以旧raw2GiB缓存限制宣称1000请求能装下，也不预下载整个补采集合。

新版本独立root明确限制raw8GiB、processed3GiB、请求事务4GiB，每次至少预留1GiB；旧raw2GiB/processed512MiB及100请求结果冻结。这个资源包络是fail-closed上限，**不是保证所有补采都能完成的证明**。真实invalidity/oversampling可改变实际请求和空间，达到边界安全停下，不清理旧数据或跳过失败。

首个真实source51/1000@请求10views/7unique已独立只读重放：raw SHA/CRC、processed SHA、RGB tensor、GT、crop K、RNG、load trace和共享after-state完全相同。只验证这个不可变1请求快照，不冒充当前实时数量或1000已就绪；并非全部新帧预处理参考的再次独立重算。

后台`fast3r-co3d51-source-prepare-v1.service`从此边界接续，日志`results/co3d51_source_prepare_v1.log`。单worker完成请求边界保存全部invalidity/RNG/输入哈希，进程可独立于Codex额度继续；恢复先校验代码/候选/协议身份，变更拒绝复用。断电造成未提交processed文件损坏仍需诊断，不能称通用故障恢复。全1000准备后自动进行只读重放，不自动运行GPU。

固定横向512×384 crop取消portrait/square分辨率反转，保留其余发布采样和参考预处理；它是依据作者线索声明的新候选协议，不宣称与作者原processed JSON/RNG或HF论文训练组别等价。此阶段模型前向0，暂无新位姿成绩，正式Table1与整篇仍未完成。


In [1]:
import json, hashlib
from pathlib import Path
import pandas as pd

def paper51_storage_sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

budget_path = Path('results/co3d_51_source_storage_v1_20261003.json')
proof = json.loads(Path('results/co3d_51_source_storage_verified_20261003.json').read_text())
budget = json.loads(budget_path.read_text())
assert proof['report_sha256'] == paper51_storage_sha(budget_path)
assert proof['verifier_sha256'] == paper51_storage_sha('scripts/verify_co3d_51_source_storage.py')
assert budget['category_count'] == proof['category_count'] == 51
assert budget['request_count'] == proof['request_count'] == 1000
assert proof['reused_index_category_count'] == 38 and proof['new_directory_category_count'] == 13
for key, count, raw_total in [('nominal', 8835, 6539401033), ('retry_jitter_closure', 103599, 76458893204)]:
    assert budget[key] == proof[key]
    assert set(proof[key]['member_count_by_kind'].values()) == {count}
    assert sum(proof[key]['advertised_raw_bytes_by_kind'].values()) == raw_total
    assert proof[key]['advertised_raw_bytes'] == raw_total
assert not proof['actual_gt_validity_verified'] and not proof['formal_Table1_result']
assert not proof['processed_storage_budget_measured']
assert not proof['raw_rgb_depth_mask_downloaded']  # Directory audit only; later preparation is separate.
print('独立目录预算：51类；复用38索引/补13类；非GT有效或位姿成绩。')
display(pd.DataFrame([
    {'集合': label, '每种成员数': proof[key]['member_count_by_kind']['images'],
     'raw RGB/depth/mask bytes': proof[key]['advertised_raw_bytes'],
     '十进制GB': proof[key]['advertised_raw_bytes']/1e9}
    for key, label in [('nominal', 'all-valid名义'), ('retry_jitter_closure', '保守补采可达集合')]
]))

prefix_path = Path('results/co3d51_source_inputs_prefix1_verified_20261003.json')
prefix = json.loads(prefix_path.read_text())
root = Path('results/co3d51_source_prepare_v1_progress')
assert prefix['verifier_sha256'] == paper51_storage_sha('scripts/verify_co3d51_source_inputs_v1.py')
assert prefix['verified_request_count'] == 1 and not prefix['all_1000_requests_prepared_and_replayed']
assert prefix['initial_sha256'] == paper51_storage_sha(root/'initial.json')
for name, value in prefix['request_sha256'].items():
    assert paper51_storage_sha(root/name) == value
assert prefix['raw_SHA_CRC_processed_SHA_and_receipts_checked']
assert prefix['input_tensor_GT_rng_trace_and_shared_state_exact_equal']
assert prefix['network_bytes'] == prefix['model_forward_count'] == 0
assert not prefix['formal_Table1_result'] and not prefix['full_paper_completed']
request = json.loads((root/'request_000.json').read_text())['result']
views = request['returned_views']
assert len(views) == 10 and len({v['instance'] for v in views}) == 7
assert all(v['img_shape'] == [3,384,512] and v['true_shape_hw'] == [384,512] for v in views)
print('真实首请求不可变核验快照：10视图/7不同帧；RGB/GT/K/RNG/共享状态精确重放；非1000完成。')
limits = json.loads((root/'initial.json').read_text())['identity']['bindings']
assert [limits[k]//1024**3 for k in ('raw_cache_limit_bytes','processed_limit_bytes','journal_limit_bytes','reserve_bytes')] == [8,3,4,1]
assert not limits['author_rng_recovered'] and not limits['formal_Table1_result']
print('新版本上限 raw8GiB + processed3GiB + journal4GiB；磁盘留1GiB。旧缓存不扩大；到界停下。')
print('后台只准备/只读验证，暂无新GPU位姿分数；作者清单/RNG/HF组别仍待确认。')


独立目录预算：51类；复用38索引/补13类；非GT有效或位姿成绩。


,集合,每种成员数,raw RGB/depth/mask bytes,十进制GB
0,all-valid名义,8835,6539401033,6.539401
1,保守补采可达集合,103599,76458893204,76.458893


真实首请求不可变核验快照：10视图/7不同帧；RGB/GT/K/RNG/共享状态精确重放；非1000完成。
新版本上限 raw8GiB + processed3GiB + journal4GiB；磁盘留1GiB。旧缓存不扩大；到界停下。
后台只准备/只读验证，暂无新GPU位姿分数；作者清单/RNG/HF组别仍待确认。


### §4.2 / Table 1 接续：完整输入门禁与公开 HF 位姿评测队列

新阶段是 **source51 / 1000@ / 固定横向512×384 的候选适配**，不是原作者精确 processed 清单、随机状态或训练权重组别已经复现。旧100请求报告与103个历史单元保持不变。

现在按以下顺序接续：独立CPU准备 → 全1000请求的原始SHA/CRC、processed SHA、实际RGB tensor/GT/K/RNG/trace/共享状态只读重放 → CPU服务正常退出 → fresh offline replay → GPU空闲至少10240MiB两次检查 → 新公开HF位姿评测 → 独立保存位姿/指标核验。GPU空闲检查不是资源预约；失败必须诊断，不能清理其他任务或删失败场景。

网络只收到`img`和`true_shape`。它预测共同坐标系的global点图和置信度，焦距由首视角预测估计，随后沿发布PnP求解像素与三维点的投影关系 $u \sim K(RX+t)$，再转成camera-to-world位姿。GT位姿、GT内参、depth和mask只用于数据准备/有效性及最终误差核验，不进入网络或焦距/PnP。固定seed42+request、16-mixed、DPT chunk2；不使用GT焦距补分。

每个请求10视图，保留全部 $\binom{10}{2}=45$ 对，包括源采样产生的重复帧。PnP失败保留identity并计数，零焦距也不静默过滤。完整1000请求才汇总45000对：分别记录1000个request级指标的算术均值与全部pair pooled指标，二者不混用。独立核验只重算保存的位姿、GT、误差和aggregate，**不是第二次独立网络推理**。

新位姿事务目录`results/co3d51_pose_seed42_progress_v1`上限512MiB，每次至少留1GiB；不可变初始身份绑定输入证明、采样state、权重/config及代码SHA。恢复仅复用已验证完整request，任意断电恢复仍未实测。准备服务与位姿队列均独立于Codex推理额度，但机器关闭/挂起或资源/数据失败会中断；不保证额度刷新时精确唤醒。

下面读取的是12:17采集的 **18/1000文件计数快照**，不是实时数量或18请求独立输入证明。它实证完整摘要/证明缺失时不会加载模型；新增离线损坏注入和边界检查通过。当前没有新位姿分数、正式Table1或整篇完成声明。运行/恢复入口见`CONTINUATION.md`。


In [1]:
import hashlib, json
from pathlib import Path

paper51_gate_path = Path("results/co3d51_pose_input_gate_20261003.json")
paper51_gate = json.loads(paper51_gate_path.read_text())
assert paper51_gate["status"] == "waiting_for_complete1000_input_preparation_and_replay"
assert paper51_gate["expected_request_count"] == 1000
assert paper51_gate["observed_committed_request_file_count"] == 18
assert paper51_gate["observed_prefix_count_is_not_independent_input_replay"] is True
for key in ("complete_preparation_summary_present", "complete_independent_input_proof_present",
            "model_loaded", "formal_Table1_result", "full_paper_completed"):
    assert paper51_gate[key] is False, key
assert paper51_gate["network_bytes"] == paper51_gate["model_forward_count"] == 0
assert hashlib.sha256(Path("scripts/fast3r_hf_co3d51_pose_eval_v1.py").read_bytes()).hexdigest() == paper51_gate["runner_sha256"]
print("真实门禁快照: 18/1000 committed filenames；不是实时数量或已核验输入覆盖")
print("完整摘要/独立证明均未生成；模型未加载，network=0 bytes，forward=0")
print("候选协议: source51 / 1000@ / fixed landscape512x384 / seed42+request / 16-mixed / chunk2")
print("网络输入仅img,true_shape；无GT焦距/PnP；45 pair/request含重复帧与失败identity")
print("独立后台: input prepare -> full1000 offline replay -> wait free GPU >=10240MiB -> pose -> saved-pose audit")
print("新位姿目录上限512MiB；每次预留1GiB；已提交request逐项核验后复用，失败不跳过")
print("正式Table1=False；作者清单/RNG/权重映射未确认；全1000准备与位姿成绩仍待完成")
print("gate snapshot SHA256:", hashlib.sha256(paper51_gate_path.read_bytes()).hexdigest())


真实门禁快照: 18/1000 committed filenames；不是实时数量或已核验输入覆盖
完整摘要/独立证明均未生成；模型未加载，network=0 bytes，forward=0
候选协议: source51 / 1000@ / fixed landscape512x384 / seed42+request / 16-mixed / chunk2
网络输入仅img,true_shape；无GT焦距/PnP；45 pair/request含重复帧与失败identity
独立后台: input prepare -> full1000 offline replay -> wait free GPU >=10240MiB -> pose -> saved-pose audit
新位姿目录上限512MiB；每次预留1GiB；已提交request逐项核验后复用，失败不跳过
正式Table1=False；作者清单/RNG/权重映射未确认；全1000准备与位姿成绩仍待完成
gate snapshot SHA256: c3031319a3ff5444c13d053f25d2c9c67a5ff759549105aa6abd78b643bd6fca


### 用户授权范围更新：暂缓 CO3D、清理本地数据（2026-10-03）

用户明确要求先不做当前CO3D位姿评测，并删除为它下载/生成的本地数据。两个后台服务已停止，准备止于26/1000请求、新位姿前向0。已永久删除16个精确核实的CO3D raw/processed/metadata/目录索引/Range/probe与深度诊断输入目录，实测可用空间增加 **4,562,313,216bytes（约4.25GiB）**。删除不能从回收站恢复，未来重做须先取得新授权并重新准备数据。

代码、公共权重、DTU/NRGBD/7-Scenes/RealEstate10K数据、历史Notebook输出、results JSON/图/事务均保留。59份已提交CO3D产物SHA逐项相同；但依赖被删raw/processed/metadata的历史完整本地输入重放已不可用。之前105单元保留为当时真实证据，不重新执行它们，也不隐藏输入现已删除这一边界。

清理凭据`results/co3d_evaluation_deferred_cleanup_20261003.json`也是两个queue的暂停标记，实际调用均在任何网络/数据/模型操作前退出，不自动恢复。定时任务提示更新两次被工具权限拒绝，**尚未生效**；没有绕过权限编辑自动化配置。以后依最新用户指令、`CONTINUATION.md`和代码标记，不将用户暂停当失败去修复。要改定时卡片文字需允许该工具或在应用中手动移出CO3D。

下一步只用保留的数据整理Table3/4/5差距、设计固定输入的置信度/对齐敏感性诊断，并审计附录BA/深度入口与资源条件。未运行实验不打勾、RE10K缺76不自动大下载、无独立消融权重/128A100训练资源不以短训练替代。整体定位为有明确暂缓项的公开权重部分复现。

清理后的测试套件245项：241执行通过，4项依赖已删除真实CO3D清单的集成检查明确skip并注明用户清理原因，不冒充这4项通过。两个queue保护与bash语法检查通过；此Notebook107单元、原105逐项保留。


In [1]:
import hashlib, json, subprocess
from pathlib import Path

paper_cleanup = json.loads(Path("results/co3d_evaluation_deferred_cleanup_20261003.json").read_text())
assert paper_cleanup["status"] == "user_deferred_CO3D_evaluation_local_data_permanently_deleted"
assert paper_cleanup["all_16_targets_absent"] is True
assert len(paper_cleanup["deleted_targets"]) == 16
for item in paper_cleanup["deleted_targets"]:
    target = Path(item["path"])
    assert target.parent == Path("/home/yyz/fast3r/data") and "co3d" in target.name.lower()
    assert not target.exists(), str(target)
for name, expected in paper_cleanup["preserved_tracked_CO3D_artifact_sha256"].items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == expected, name
assert paper_cleanup["raw_inputs_required_for_local_replay_present"] is False
assert paper_cleanup["automation_prompt_update_applied"] is False
assert paper_cleanup["resume_requires_explicit_user_authorization"] is True
for queue in ("queue_co3d51_source_prepare_v1.sh", "queue_co3d51_pose_eval_v1.sh"):
    stopped = subprocess.run(["/bin/bash", str(Path("scripts") / queue)], capture_output=True, text=True, timeout=3)
    assert stopped.returncode == 0 and "deferred by user" in stopped.stdout
print("用户暂缓CO3D：停止于26/1000输入准备；新位姿前向未开始，不是失败待自动恢复")
print("16个CO3D数据目录全部不存在；永久删除，不能从回收站恢复")
print("已提交CO3D报告/图SHA逐项不变:", len(paper_cleanup["preserved_tracked_CO3D_artifact_sha256"]))
print("清理当次可用空间增加bytes:", paper_cleanup["observed_available_bytes_increase"])
print("清理当次可用空间GiB:", round(paper_cleanup["available_bytes_after_deletion"] / 1024**3, 2))
print("两个queue均在网络/数据/模型操作前退出；未重启/重下载")
print("自动化提示更新受权限限制尚未生效；按最新用户指令/恢复文档/代码保护执行")
print("历史原始输入重放不可用；旧报告不覆盖、不升格正式Table1或整篇完成")
print("下一步：现有重建数据差距整理、固定输入置信度/对齐诊断、附录BA/深度可行性审计")


用户暂缓CO3D：停止于26/1000输入准备；新位姿前向未开始，不是失败待自动恢复
16个CO3D数据目录全部不存在；永久删除，不能从回收站恢复
已提交CO3D报告/图SHA逐项不变: 59
清理当次可用空间增加bytes: 4562313216
清理当次可用空间GiB: 35.18
两个queue均在网络/数据/模型操作前退出；未重启/重下载
自动化提示更新受权限限制尚未生效；按最新用户指令/恢复文档/代码保护执行
历史原始输入重放不可用；旧报告不覆盖、不升格正式Table1或整篇完成
下一步：现有重建数据差距整理、固定输入置信度/对齐诊断、附录BA/深度可行性审计


## Retained reconstruction results: Table 3–5 gap audit

This CPU-only analysis re-aggregates the saved per-scene reports; it does not run Fast3R or change any metric settings. Tables 3 and 4 compare the mean of per-scene median distances. Table 5 compares per-scene mean distances from paired local/global predictions. Scene-dataset distances are multiplied by 100 as in the paper. Table 5's local/global direction is reported separately from its absolute metric gap. The public checkpoint's mapping to the paper's experiment-specific weights remains unverified.

Execution provenance: this code cell was executed directly with the project Python environment as a CPU-only batch; no notebook kernel or GPU was involved.

In [1]:
import json
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
def load_report(relative_path):
    path = project_root / relative_path
    return json.loads(path.read_text())

def mean(values):
    return sum(values) / len(values)

def close_enough(a, b, tolerance=1e-10):
    return abs(a - b) <= tolerance * max(1.0, abs(a), abs(b))

# Table 3/4: reproduce each saved aggregate from the per-scene medians.
table34_specs = [
    ('Table 3', 'NRGBD', 'results/nrgbd_seed42_stride40.json', 100.0, 3.40, 1.01),
    ('Table 3', '7-Scenes', 'results/7scenes_paired_seed42_stride20.json', 100.0, 1.58, 0.93),
    ('Table 4', 'DTU', 'results/dtu_seed42_stride5.json', 1.0, 1.706, 0.857),
]
table34_rows = []
for paper_table, dataset, relative_path, multiplier, paper_acc, paper_comp in table34_specs:
    report = load_report(relative_path)
    assert len(report['scenes']) == report['scene_count']
    for metric in ('accuracy_median', 'completion_median'):
        value = mean([scene['metrics'][metric] for scene in report['scenes']])
        assert close_enough(value, report['aggregate_mean'][metric]), (relative_path, metric)
    acc = report['aggregate_mean']['accuracy_median'] * multiplier
    comp = report['aggregate_mean']['completion_median'] * multiplier
    table34_rows.append({
        'Table': paper_table, 'Dataset': dataset, 'Scenes': report['scene_count'],
        'Acc ours': acc, 'Acc paper': paper_acc,
        'Acc gap %': (acc / paper_acc - 1) * 100,
        'Comp ours': comp, 'Comp paper': paper_comp,
        'Comp gap %': (comp / paper_comp - 1) * 100,
    })

# Table 5: re-average paired per-scene predictions. For DTU, read 10-view rows.
paper_table5 = {
    'NRGBD': {'local': (4.39, 1.28), 'global': (4.85, 1.32)},
    '7-Scenes': {'local': (2.84, 1.37), 'global': (4.81, 1.64)},
    'DTU': {'local': (3.91, 1.41), 'global': (3.88, 1.41)},
}
table5_rows = []
for dataset, relative_path, multiplier in [
    ('NRGBD', 'results/nrgbd_paired_seed42_stride40.json', 100.0),
    ('7-Scenes', 'results/7scenes_paired_seed42_stride20.json', 100.0),
]:
    report = load_report(relative_path)
    assert all(scene['paired_same_forward'] for scene in report['scenes'])
    for head in ('local', 'global'):
        values = [scene['metrics_by_head'][head] for scene in report['scenes']]
        for metric in ('accuracy', 'completion'):
            actual = mean([item[metric] for item in values])
            saved = report['aggregate_by_head'][head][metric]
            assert close_enough(actual, saved), (relative_path, head, metric)
        acc = report['aggregate_by_head'][head]['accuracy'] * multiplier
        comp = report['aggregate_by_head'][head]['completion'] * multiplier
        pacc, pcomp = paper_table5[dataset][head]
        table5_rows.append({
            'Dataset': dataset, 'Head': head, 'Acc ours': acc,
            'Acc paper': pacc, 'Acc gap %': (acc / pacc - 1) * 100,
            'Comp ours': comp, 'Comp paper': pcomp,
            'Comp gap %': (comp / pcomp - 1) * 100,
        })

dtu = load_report('results/dtu_paper_experiments.json')
for head in ('local', 'global'):
    rows = [row for row in dtu['rows'] if row['view_count'] == 10]
    assert len(rows) == 22 and all(row['status'] == 'completed' for row in rows)
    for metric in ('accuracy', 'completion'):
        actual = mean([row['metrics'][head][metric] for row in rows])
        saved = dtu['aggregate'][f'10_views_{head}']['aggregate_mean'][metric]
        assert close_enough(actual, saved), ('DTU', head, metric)
    acc = dtu['aggregate'][f'10_views_{head}']['aggregate_mean']['accuracy']
    comp = dtu['aggregate'][f'10_views_{head}']['aggregate_mean']['completion']
    pacc, pcomp = paper_table5['DTU'][head]
    table5_rows.append({
        'Dataset': 'DTU', 'Head': head, 'Acc ours': acc,
        'Acc paper': pacc, 'Acc gap %': (acc / pacc - 1) * 100,
        'Comp ours': comp, 'Comp paper': pcomp,
        'Comp gap %': (comp / pcomp - 1) * 100,
    })

table34 = pd.DataFrame(table34_rows).round(4)
table5 = pd.DataFrame(table5_rows).round(4)
print('Tables 3/4: mean of per-scene median distances')
print(table34.to_string(index=False))
print('Table 5: paired mean distances; gaps compare each head to its paper row')
print(table5.to_string(index=False))
print('Table 5 effect (global minus aligned local; positive means global has higher error):')
for dataset in ('7-Scenes', 'NRGBD', 'DTU'):
    local = next(row for row in table5_rows if row['Dataset'] == dataset and row['Head'] == 'local')
    global_ = next(row for row in table5_rows if row['Dataset'] == dataset and row['Head'] == 'global')
    p_local = paper_table5[dataset]['local']
    p_global = paper_table5[dataset]['global']
    print(f"{dataset}: ours ΔAcc={global_['Acc ours']-local['Acc ours']:+.4f}, "
          f"paper ΔAcc={p_global[0]-p_local[0]:+.2f}; "
          f"ours ΔComp={global_['Comp ours']-local['Comp ours']:+.4f}, "
          f"paper ΔComp={p_global[1]-p_local[1]:+.2f}")

Tables 3/4: mean of per-scene median distances
  Table  Dataset  Scenes  Acc ours  Acc paper  Acc gap %  Comp ours  Comp paper  Comp gap %
Table 3    NRGBD       9    4.0165      3.400    18.1331     1.2001       1.010     18.8220
Table 3 7-Scenes      18    3.3035      1.580   109.0794     3.1058       0.930    233.9550
Table 4      DTU      22    2.0827      1.706    22.0792     1.0311       0.857     20.3142
Table 5: paired mean distances; gaps compare each head to its paper row
 Dataset   Head  Acc ours  Acc paper  Acc gap %  Comp ours  Comp paper  Comp gap %
   NRGBD  local    9.7108       4.39   121.2037     3.1292        1.28    144.4672
   NRGBD global    9.2594       4.85    90.9159     3.1789        1.32    140.8284
7-Scenes  local    6.3613       2.84   123.9890     7.0516        1.37    414.7146
7-Scenes global    6.9567       4.81    44.6307     6.3510        1.64    287.2587
     DTU  local    4.1451       3.91     6.0130     2.3889        1.41     69.4277
     DTU global

### Appendix C/D/E feasibility review (2026-10-03)

A read-only workspace and environment audit found no InstantSplat, gaussian_renderer, simple_knn, or rmvd module in the project Python environment. Appendix C uses a seven-image CO3D example; those local inputs were removed under the current user scope. Appendix D reports GS-BA on the Tanks & Temples Family scene, which is not present in `data/`. Appendix E/Table 7 requires ScanNet, ETH3D, DTU, and Tanks & Temples under RobustMVD; the existing `scripts/robustmvd_eval.py` imports `rmvd`, hardcodes the ScanNet robustmvd split, and has an external absolute output path. The local DTU reconstruction split has not been verified as a RobustMVD split. No dependency or dataset was downloaded, no package was installed, and no GPU job was started. The existing PLY visualization does not count as Gaussian Splatting or bundle adjustment.

### DTU scan1固定预测阈值诊断（2026-10-03，Table4/5差距诊断，非全量复现）
真实结果：results/diagnostics/dtu_scan1_threshold_sensitivity_seed42_v1.json。宿主GPU单次前向已完成，7组local评测，原始预测前后SHA相同，所有指标有限。使用stride1完整序列的rounded linspace 10视角、seed1052，与历史10视角输入协议一致。首次stride5/均匀采样标签混用在模型加载前被门禁拒绝，随后修复并重试。
baseline mean Accuracy/Completion=3.9710/1.5382；alignment0=3.8170/2.4422；alignment50=3.6107/1.5375；alignment95=4.5749/1.8825。metric25=2.5960/1.8804；metric50=1.7040/4.4900；metric75=1.0783/14.2454。DTU距离不乘100。
过滤预测点可降低Accuracy距离，却损害Completion，不能只挑Accuracy宣称模型变好。单场景不能解释所有数据集差距，也不是按GT选择最佳阈值或新的独立训练消融。结果已做独立CPU结构/SHA/有限值核验；此新增说明不是Jupyter分析单元执行输出，完整分析归档待续。


In [1]:
import json, math
from pathlib import Path
p = json.loads(Path('results/diagnostics/dtu_scan1_threshold_sensitivity_seed42_v1.json').read_text())
assert p['forward_count'] == 1 and p['same_forward_for_all_cases']
assert p['raw_prediction_sha256_before_evaluation'] == p['raw_prediction_sha256_after_evaluation']
assert p['view_count'] == 10 and p['scene_seed'] == 1052 and p['kf_every'] == 1
assert len(p['threshold_cases']) == 7
for c in p['threshold_cases']:
    assert all(math.isfinite(v) for v in c['metrics'].values())
    print(c['name'], 'Acc=', round(c['metrics']['accuracy'], 4), 'Comp=', round(c['metrics']['completion'], 4))
print('Verified: 1 forward, 10 fixed views, 7 finite cases, raw prediction SHA unchanged.')
print('Single-scene diagnostic only; filtering changes coverage, not a full Table4/5 reproduction or GT-selected recommendation.')


baseline Acc= 3.971 Comp= 1.5382
alignment_0 Acc= 3.817 Comp= 2.4422
alignment_50 Acc= 3.6107 Comp= 1.5375
alignment_95 Acc= 4.5749 Comp= 1.8825
metric_25 Acc= 2.596 Comp= 1.8804
metric_50 Acc= 1.704 Comp= 4.49
metric_75 Acc= 1.0783 Comp= 14.2454
Verified: 1 forward, 10 fixed views, 7 finite cases, raw prediction SHA unchanged.
Single-scene diagnostic only; filtering changes coverage, not a full Table4/5 reproduction or GT-selected recommendation.


### 三场景固定前向阈值敏感性（2026-10-03）

在`scan1`基础上，按DTU测试场景的字典序增加`scan10`和`scan11`；每个场景只做一次前向，并用同一原始预测比较7组设置。三场景使用完整stride1序列、rounded linspace索引`[0, 5, 11, 16, 21, 27, 32, 37, 43, 48]`、512分辨率、chunk2和历史scene seed（1052/10052/11052）。输入标签及85/0 baseline的8项local指标逐场景匹配`dtu_paper_experiments.json`历史10-view记录。

协议边界：`alignment percentile`对应仓库参数`min_conf_thr_percentile_for_local_alignment_and_icp`，同时控制local-to-global对应点筛选和GT配准的置信度权重；`metric percentile`筛选进入预测点云/GT配准的高置信度对应点，而Completion仍对全部有效GT计分。因此Accuracy距离下降不能单独解释为重建整体改善。此处三场景均值只作敏感性描述，不代表DTU 22场景Table 4，也不用于选择阈值。



In [1]:
import hashlib, json, math
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
scene_names = ["scan1", "scan10", "scan11"]
report_paths = {
    scene: project_root / f"results/diagnostics/dtu_{scene}_threshold_sensitivity_seed42_v1.json"
    for scene in scene_names
}
reports = {scene: json.loads(path.read_text()) for scene, path in report_paths.items()}
historical = json.loads((project_root / "results/dtu_paper_experiments.json").read_text())
source_checks = []
validation_rows = []
case_maps = {}

for scene, report in reports.items():
    old_row = next(row for row in historical["rows"]
                   if row["scene"] == scene and row["view_count"] == 10)
    case_map = {case["name"]: case for case in report["threshold_cases"]}
    case_maps[scene] = case_map
    baseline = case_map["baseline"]["metrics"]
    historical_local = old_row["metrics"]["local"]
    baseline_exact = all(baseline[key] == historical_local[key] for key in historical_local)
    inputs_exact = report["input_labels"] == old_row["input_labels"]
    prediction_sha_stable = (
        report["raw_prediction_sha256_before_evaluation"]
        == report["raw_prediction_sha256_after_evaluation"]
    )
    finite_cases = len(case_map) == 7 and all(
        math.isfinite(value)
        for case in case_map.values()
        for value in case["metrics"].values()
    )
    source_current = all(
        hashlib.sha256((project_root / rel_path).read_bytes()).hexdigest() == sha
        for rel_path, sha in report["source_sha256"].items()
    )
    source_checks.append(source_current)
    validation_rows.append({
        "scene": scene,
        "views": report["view_count"],
        "seed": report["scene_seed"],
        "inputs_match_history": inputs_exact,
        "baseline_all_8_match": baseline_exact,
        "one_forward": report["forward_count"] == 1 and report["same_forward_for_all_cases"],
        "prediction_sha_stable": prediction_sha_stable,
        "7_cases_finite": finite_cases,
        "source_sha_matches": source_current,
    })

print("Per-scene report and historical-baseline checks:")
display(pd.DataFrame(validation_rows))

case_names = [case["name"] for case in reports["scan1"]["threshold_cases"]]
summary_rows = []
detail_rows = []
baseline_means = {
    metric: float(np.mean([case_maps[scene]["baseline"]["metrics"][metric] for scene in scene_names]))
    for metric in ("accuracy", "completion")
}
for case_name in case_names:
    configs = [case_maps[scene][case_name] for scene in scene_names]
    accuracies = [case["metrics"]["accuracy"] for case in configs]
    completions = [case["metrics"]["completion"] for case in configs]
    mean_accuracy = float(np.mean(accuracies))
    mean_completion = float(np.mean(completions))
    summary_rows.append({
        "case": case_name,
        "alignment/ICP %": configs[0]["alignment_percentile"],
        "metric %": configs[0]["metric_percentile"],
        "3-scene mean Acc": mean_accuracy,
        "3-scene mean Comp": mean_completion,
        "ΔAcc vs 85/0": mean_accuracy - baseline_means["accuracy"],
        "ΔComp vs 85/0": mean_completion - baseline_means["completion"],
    })
    for scene, case in zip(scene_names, configs):
        detail_rows.append({
            "scene": scene,
            "case": case_name,
            "Acc": case["metrics"]["accuracy"],
            "Comp": case["metrics"]["completion"],
        })

print("Descriptive arithmetic mean across the three listed scenes (raw DTU distances; no ×100):")
display(pd.DataFrame(summary_rows).round(4))
print("Per-scene Accuracy/Completion values:")
display(pd.DataFrame(detail_rows).round(4))
print("All three reports use one forward each; same-forward hashes are stable; all 85/0 baselines and 10-view labels match history.")


Per-scene report and historical-baseline checks:


,scene,views,seed,inputs_match_history,baseline_all_8_match,one_forward,prediction_sha_stable,7_cases_finite,source_sha_matches
0,scan1,10,1052,True,True,True,True,True,True
1,scan10,10,10052,True,True,True,True,True,True
2,scan11,10,11052,True,True,True,True,True,True


Descriptive arithmetic mean across the three listed scenes (raw DTU distances; no ×100):


,case,alignment/ICP %,metric %,3-scene mean Acc,3-scene mean Comp,ΔAcc vs 85/0,ΔComp vs 85/0
0,baseline,85.0,0.0,6.9304,2.4108,0.0000,0.0000
1,alignment_0,0.0,0.0,6.0540,2.8598,-0.8764,0.4490
2,alignment_50,50.0,0.0,6.2488,2.3617,-0.6816,-0.0491
3,alignment_95,95.0,0.0,8.7881,3.2870,1.8578,0.8762
4,metric_25,85.0,25.0,6.3263,2.5691,-0.6040,0.1582
5,metric_50,85.0,50.0,4.9653,4.7278,-1.9651,2.3170
6,metric_75,85.0,75.0,3.4452,13.1753,-3.4852,10.7645


Per-scene Accuracy/Completion values:


,scene,case,Acc,Comp
0,scan1,baseline,3.9710,1.5382
1,scan10,baseline,13.8274,4.2092
2,scan11,baseline,2.9927,1.4850
3,scan1,alignment_0,3.8170,2.4422
4,scan10,alignment_0,11.5498,4.7522
5,scan11,alignment_0,2.7953,1.3849
6,scan1,alignment_50,3.6107,1.5375
7,scan10,alignment_50,12.0080,3.9537
8,scan11,alignment_50,3.1277,1.5940
9,scan1,alignment_95,4.5749,1.8825


All three reports use one forward each; same-forward hashes are stable; all 85/0 baselines and 10-view labels match history.


### RE10K缺失源补充审计（2026-10-03）

规定测试集1,832个场景中已验证准备1,756个，缺76个。此前完成的205,763,619,478-byte来源扫描在固定识别规则下找到4,137个场景、恢复0个缺失ID；不重复扫描，也不据此断言这些场景永久不可用。

本轮发现公开候选 [`ghuijo/realestate10k`](https://huggingface.co/datasets/ghuijo/realestate10k/tree/main)，页面标注cc-by-4.0、总计429GB，预览出现test metadata与image路径。页面列出4个约107GB分片，但完整viewer报告崩溃；公开材料没有证明76个规定ID的覆盖，也没有核实图像几何或底层RGB来源/条款。本轮未下载数据，候选不改变当前覆盖或产生新位姿指标。

Google[官方页面](https://google.github.io/realestate10k/download.html)说明其720MB归档是相机轨迹（逐帧时间戳、内参、位姿和视频URL），本身不包含RGB帧。证据与限制记录在[`results/re10k_missing_source_followup_20261003.json`](results/re10k_missing_source_followup_20261003.json)。

In [1]:
from pathlib import Path
import json

verified = json.loads(Path("results/re10k_missing_full_source_verified_20261002.json").read_text())
followup = json.loads(Path("results/re10k_missing_source_followup_20261003.json").read_text())

assert len(verified["missing_scene_ids"]) == verified["missing_scene_count"] == 76
assert verified["prescribed_scene_count"] == 1832
assert verified["previously_verified_prepared_scene_count"] == 1756
assert verified["recognized_source_scene_count"] == 4137
assert verified["recovered_scene_count"] == 0
assert verified["source_stream_sha_verified_by_completed_scanner"] is True
assert verified["archive_independently_rehashed_this_verification"] is False
assert followup["prescribed_split"]["missing_scene_count"] == 76
assert followup["public_candidate"]["exact_missing_id_coverage_verified"] is False
assert followup["public_candidate"]["dataset_payload_bytes_downloaded"] == 0

rows = [
    ("规定划分", f"{verified['previously_verified_prepared_scene_count']}/{verified['prescribed_scene_count']}准备完成；缺{verified['missing_scene_count']}"),
    ("已完成的来源扫描", f"{verified['source_stream_bytes']:,} bytes；识别{verified['recognized_source_scene_count']}个场景；恢复{verified['recovered_scene_count']}个"),
    ("2026-10-02来源报告核验", f"扫描SHA由已完成scanner提供；该核验未再次rehash整包；网络数据负载{verified['network_bytes_this_verification']} bytes"),
    ("新公开候选", f"{followup['public_candidate']['repository']}，页面标注{followup['public_candidate']['page_reported_total_size_gb']}GB；规定ID覆盖未验证，下载数据{followup['public_candidate']['dataset_payload_bytes_downloaded']} bytes"),
    ("可升格正式Table 1", "否；缺失场景未补齐，也没有新的位姿评测"),
]
table = "| 核查项 | 当前证据 |\n| --- | --- |\n" + "\n".join(f"| {name} | {value} |" for name, value in rows)
print(table)

| 核查项 | 当前证据 |
| --- | --- |
| 规定划分 | 1756/1832准备完成；缺76 |
| 已完成的来源扫描 | 205,763,619,478 bytes；识别4137个场景；恢复0个 |
| 2026-10-02来源报告核验 | 扫描SHA由已完成scanner提供；该核验未再次rehash整包；网络数据负载0 bytes |
| 新公开候选 | ghuijo/realestate10k，页面标注429GB；规定ID覆盖未验证，下载数据0 bytes |
| 可升格正式Table 1 | 否；缺失场景未补齐，也没有新的位姿评测 |


## 结项交接：Fast3R公开权重部分复现

完整实验细节见[逐项论文对应表](PAPER_REPRODUCTION.md)，流程、停止边界、学习总结与中英文简历示例见[结项交接文档](REPRODUCTION_HANDOFF.md)。结论是**部分复现，整篇尚未完成**。

### 这次复现走过的流程

1. 将论文表格/图拆成协议清单，标出数据划分、指标统计量、采样、权重与硬件要求。
2. 固定代码、配置和公开checkpoint身份，检查权重/配置哈希；记录seed、环境、图像输入与视角标签。
3. 按规定划分准备数据，核对ID、GT、帧号、RGB解码和相机几何；用manifest、CRC/SHA记录覆盖率，缺失场景不静默跳过。
4. 用公开权重完成DTU 22场景、Neural RGB-D 9场景、7-Scenes 18条测试轨迹评测，保存逐场景指标后按论文口径重新聚合。
5. 用同一次前向比较local/global；固定单份DTU预测对照阈值，独立核对输入、预测SHA、有限指标及历史基线。
6. 对照论文逐项记录完成程度和差距，明确区分代码核对、数据就绪、本地运行、协议等价与论文结果复现。

### 结果与停止边界

- Table 3/4数据划分已跑完，但结果未匹配论文：NRGBD的Accuracy/Completion高18.13%/18.82%，7-Scenes高109.08%/233.95%，DTU高22.08%/20.31%。Table 5已做配对比较，但local优势方向未在所有数据集重现。
- RE10K尚缺76/1,832个规定场景；公开镜像候选的ID覆盖未核实。CO3D按用户要求暂缓，历史候选位姿结果不算正式Table 1。
- 训练型消融和完整训练缺独立权重/研究规模算力；附录C–E还缺规定依赖或数据。论文训练规模是128张A100-80GB、174K steps，当前不尝试以短程训练替代。

### 科研方法与简历表达

这里练习到的是：把论文主张拆成可证伪的实验；固定来源、划分和哈希；控制变量并用同次前向做配对；从逐场景结果重算统计量；保留负结果；把数据覆盖、协议等价、分数接近和完整复现分开陈述；先做资源预算再开跑。

**简历建议：**

> 基于Fast3R官方代码与公开权重开展部分复现，完成DTU 22场景、Neural RGB-D 9场景及7-Scenes 18条测试轨迹评测；搭建数据清单/哈希核验和逐场景指标复核流程，完成local/global同次前向对照及DTU三场景阈值敏感性分析，并量化本地结果与论文指标的差距。

**English:**

> Conducted a partial reproduction of Fast3R using its official codebase and public checkpoint. Evaluated 22 DTU scenes, 9 Neural RGB-D scenes, and all 18 7-Scenes test trajectories; built data provenance and per-scene metric checks, ran same-forward local/global comparisons and a three-scene DTU threshold sensitivity study, and quantified deviations from the paper.

不要写“完整复现”或“达到论文指标”。

## 结项后的本地存储清理（2026-10-07）
用户明确授权永久删除本项目数据集、下载包、模型权重及临时缓存，并关闭定时任务。data/、model.safetensors、temp_preview_frames/、gradio_tmp_dir/已删除；应用确认fast3r自动化删除成功。删除后、编辑本文档前，491份Git跟踪文件SHA逐项不变；模型config.json保留。
实测可用空间增加92,818,866,176bytes（约86.44GiB），核验时可用97,638,150,144bytes（约90.93GiB），项目剩余约248MiB。证据见results/project_storage_cleanup_20261007.json。此单元为清理说明，不是新模型评测输出。
所有先前实验输出保持原样、仍为历史证据。本地模型前向和依赖已删除数据的评测不再可直接执行；未来重跑需要新授权并重新下载，删除不能从回收站恢复。未清理其他项目或共享Conda/HuggingFace缓存。
